# E07 · Arquitectura completa desde los datos crudos (RTX 4090)

**Objetivo:** probar todas las configuraciones sin descartar ninguna por sus resultados, guardar cada paso en su carpeta y terminar con los datos para decidir qué congelar el sábado. Es un único notebook: no depende del repo ni de otros archivos de código.

**Entradas (únicamente):**

| Entrada | Qué es |
|---|---|
| `corpus_manifest.json` | un registro por documento: `doc_id`, `titulo`, `tipo`, `numero`, `anio`, `texto_archivo` |
| textos | archivos `.txt` en la ruta de `texto_archivo`, relativa a `DATOS` |
| `sample_50.jsonl` | las 50 preguntas de muestra (la respuesta solo se usa para puntuar) |
| `scripts/` oficiales | `evaluate.py`, `citations.py`, `common.py` del material del lunes |
| `seed_targets.json` | opcional, para las normas ancla por área |

**Etapas:** E00 entorno, E01 BM25, E02 índices densos, E09 reranker afinado (opcional), E03 búsquedas, E04 rankings, E05 evidencia, E06 generación, E07 rúbrica, E08 juez, E10 producto y, al final, la corrida del sábado.

Si se corta, volvemos a ejecutar todo: cada etapa se salta lo que ya terminó.

## 0. Parámetros
Para rehacer desde un punto en otra carpeta: `SALIDA` nueva y la anterior en `REUTILIZAR`. `DESDE_ETAPA` recalcula esa etapa y las siguientes (lo anterior se archiva).

In [ ]:
from pathlib import Path

DATOS = Path("/home/user/ai-week-2026")                              # carpeta base de las rutas de abajo
ENTRADAS = {
    "manifiesto": "data/releases/corpus_eval_v1/corpus_manifest.json",
    "textos": ".",                                       # base de texto_archivo del manifiesto
    "muestra": "data/oficial/data/sample_50.jsonl",
    "scripts_oficiales": "data/oficial/scripts",
    "esquema": "data/oficial/schema/submission.schema.json",
    "seed_targets": "data/oficial/data/seed_targets.json",
}

SALIDA = Path("/home/user/ai-week-2026/output/e07")                  # ~30 GB libres
REUTILIZAR = []
DESDE_ETAPA = None
ENTRENAR_RERANKER = True
JUEZ_RAGAS = False
JUZGAR_TODO = False
MAX_CORRIDAS_GENERACION = None
CACHE_HF = None
TEST_992 = None


In [ ]:
import os
if CACHE_HF:
    os.environ["HF_HOME"] = CACHE_HF
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
# Dependencias (una vez): torch con CUDA, transformers>=5.3, accelerate, faiss-cpu, pandas, matplotlib,
# psutil, jsonschema, sentencepiece, peft (opcional); juez: scripts/requirements-evaluador.txt

## 1. Código
Cada celda es un módulo. Se ejecutan en orden una vez; no hay que tocarlas.

In [ ]:
import sys, types
from pathlib import Path
from IPython.core.magic import register_cell_magic

def _paquete(nombre):
    if nombre not in sys.modules:
        paquete = types.ModuleType(nombre)
        paquete.__path__ = []
        sys.modules[nombre] = paquete
        if "." in nombre:
            padre, hijo = nombre.rsplit(".", 1)
            setattr(_paquete(padre), hijo, paquete)
    return sys.modules[nombre]

@register_cell_magic
def modulo(nombre, celda):
    """Ejecuta la celda como el módulo `nombre` (así los módulos se importan entre sí)."""
    nombre = nombre.strip()
    padre, hijo = nombre.rsplit(".", 1)
    mod = types.ModuleType(nombre)
    mod.__file__ = str(Path.cwd() / (nombre.replace(".", "_") + ".py"))
    sys.modules[nombre] = mod
    setattr(_paquete(padre), hijo, mod)
    exec(compile(celda, f"<{nombre}>", "exec"), mod.__dict__)

**configuracion**. Entradas y rutas. `preparar_entrada` deja solo los campos de la pregunta.

In [ ]:
%%modulo produccion.configuracion
"""Lectura de config.json y rutas del corpus congelado."""
import json
from pathlib import Path


CAMPOS_ENTRADA = ["id", "formato", "pregunta", "opciones", "area", "sub_tarea", "tema", "complejidad"]
# Nunca entran al sistema: son la respuesta o el fundamento de referencia.
CAMPOS_RESERVADOS = {
    "legal_basis", "respuesta_correcta", "respuesta_esperada", "respuestas_esperadas",
    "texto_respuesta_correcta", "ground_truth", "answer", "correct_answer", "expected_answer",
}


def preparar_entrada(registro):
    """Solo los campos de la pregunta. Descarta respuesta y legal_basis si vienen en el archivo."""
    entrada = {k: registro[k] for k in CAMPOS_ENTRADA if k in registro}
    if "sub_tarea" not in entrada and "subtarea" in registro:
        entrada["sub_tarea"] = registro["subtarea"]
    if not isinstance(entrada.get("id"), int) or entrada.get("formato") not in (
            "multiple_choice", "semi_open", "open_ended"):
        raise ValueError(f"Entrada sin id entero o formato válido: {registro.get('id')}")
    if not str(entrada.get("pregunta") or "").strip():
        raise ValueError(f"Entrada sin pregunta: {entrada['id']}")
    if entrada["formato"] == "multiple_choice" and not entrada.get("opciones"):
        raise ValueError(f"Selección múltiple sin opciones: {entrada['id']}")
    return entrada


def leer_jsonl(ruta):
    with Path(ruta).open(encoding="utf-8-sig") as f:
        return [json.loads(linea) for linea in f if linea.strip()]


def revision(config, modelo):
    """Revisión fijada en el catálogo del repo, si existe; si no, la que diga config."""
    catalogo = config["rutas"].get("catalogo_modelos")
    if catalogo and Path(catalogo).is_file():
        for fila in json.loads(catalogo.read_text(encoding="utf-8")):
            if fila.get("repo_id") == modelo:
                return fila.get("revision")
    return None


def preparar(config):
    """Rutas absolutas a partir de la carpeta de datos crudos."""
    config = json.loads(json.dumps(config))
    base = Path(config["datos"]).expanduser()
    config["rutas"] = {k: (Path(v) if Path(v).is_absolute() else base / v) for k, v in config["entradas"].items() if v}
    config["rutas"].setdefault("textos", base)
    return config


**evidencia**. Identidad de cada norma como la reconoce el evaluador y cabeceras literales (v04).

In [ ]:
%%modulo produccion.evidencia
"""Evidencia y citas para producción.

EvidenciaV04 es copia de legalrag.citations.evidencia. La clase base se reescribe
aquí para no depender de legalrag.evaluation.oficial: solo necesita el manifiesto,
los textos preparados y el extractor oficial de citas (citations.py).
"""
import importlib.util
import json
import re
from collections import OrderedDict
from functools import lru_cache
from pathlib import Path


def cargar_citaciones(ruta_scripts):
    """Importa data/oficial/scripts/citations.py tal cual lo usa el evaluador."""
    ruta = Path(ruta_scripts) / "citations.py"
    spec = importlib.util.spec_from_file_location("citaciones_oficiales", ruta)
    modulo = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(modulo)
    return modulo


def identidad(documento, citas):
    """Cuerpos normativos del documento según sus metadatos (igual que scripts/auxiliares/oficial.py)."""
    tipo = (documento.get("tipo") or "").replace("_", " ")
    numero, anio = documento.get("numero"), documento.get("anio")
    if tipo == "constitucion":
        return {("constitucion", None, None)}
    if tipo in {"ley", "decreto", "decreto ley", "acto legislativo", "resolucion", "circular", "acuerdo"}:
        return citas.bodies(citas.extract(f"{tipo} {numero} de {anio}"))
    if tipo == "decision" and str(numero) == "486":
        return {("decision_andina_486", None, None)}
    if tipo in {"sentencia", "auto"}:
        return citas.bodies(citas.extract(f"Sentencia {numero} de {anio}"))
    return citas.bodies(citas.extract(documento.get("titulo", "")))


class Evidencia:
    """Acceso al corpus congelado: manifiesto, textos y extractor oficial."""

    def __init__(self, manifiesto, textos, citas, cache_textos=512):
        registros = json.loads(Path(manifiesto).read_text(encoding="utf-8"))
        self.documentos = {r["doc_id"]: r for r in registros}
        self.carpeta_textos = Path(textos)
        # texto_archivo del manifiesto es relativo a la carpeta de datos; si falta, <doc_id>.txt
        self.rutas_texto = {r["doc_id"]: self.carpeta_textos / r["texto_archivo"]
                            for r in registros if r.get("texto_archivo")}
        self.citas = citas
        self._leer = lru_cache(maxsize=cache_textos)(self._leer_texto)

    def _leer_texto(self, doc_id):
        return self.rutas_texto.get(doc_id, self.carpeta_textos / f"{doc_id}.txt").read_text(encoding="utf-8")

    def texto(self, doc_id):
        return self._leer(doc_id)

    def verificar(self, pasajes):
        """Cada pasaje con offsets debe ser el tramo literal del documento."""
        for p in pasajes:
            if p["doc_id"] not in self.documentos:
                raise ValueError("doc_id fuera del manifiesto: " + p["doc_id"])
            if "inicio" in p and self.texto(p["doc_id"])[p["inicio"]:p["fin"]] != p["texto"]:
                raise ValueError("Pasaje no literal: " + p["doc_id"])
        return True


NOMBRES = {
    "constitucion": "Constitución Política",
    "codigo_civil": "Código Civil",
    "codigo_penal": "Código Penal",
    "codigo_procedimiento_penal": "Código de Procedimiento Penal",
    "codigo_comercio": "Código de Comercio",
    "codigo_sustantivo_trabajo": "Código Sustantivo del Trabajo",
    "codigo_procesal_trabajo": "Código Procesal del Trabajo",
    "codigo_general_proceso": "Código General del Proceso",
    "cpaca": "Código de Procedimiento Administrativo y de lo Contencioso Administrativo",
    "estatuto_tributario": "Estatuto Tributario",
    "codigo_infancia": "Código de la Infancia y la Adolescencia",
    "codigo_nacional_policia": "Código Nacional de Seguridad y Convivencia Ciudadana",
    "codigo_disciplinario": "Código General Disciplinario",
    "estatuto_consumidor": "Estatuto del Consumidor",
    "decision_andina_486": "Decisión 486 de la Comisión de la Comunidad Andina",
}
TIPOS = {"ley": "Ley", "decreto": "Decreto", "acto_legislativo": "Acto Legislativo",
         "resolucion": "Resolución", "circular": "Circular", "acuerdo": "Acuerdo"}
MAX_CABECERA = 700
# Alcance de la búsqueda del tramo literal cuando las primeras líneas no bastan.
# Corto a propósito: más adentro ya no hay encabezados, solo coincidencias sueltas
# del cuerpo que el extractor lee como una cita.
MAX_BUSQUEDA_CABECERA = 4000
# Pasajes que nombran la norma y no son evidencia recuperada.
CABECERAS = ("cabecera_fuente", "cabecera_normalizada")


def nombre_cuerpo(cuerpo):
    """Texto que el extractor oficial vuelve a leer como el mismo cuerpo."""
    tipo, numero, anio = cuerpo
    if tipo in NOMBRES:
        return NOMBRES[tipo]
    if tipo == "jurisprudencia":
        return f"Sentencia {numero} de {anio}"
    if tipo in TIPOS and numero and anio:
        return f"{TIPOS[tipo]} {numero} de {anio}"
    return None


def _prioridad(cuerpo):
    # Los nombres de código son los que usan las preguntas y el fundamento oficial.
    return 0 if cuerpo[0] in NOMBRES else 1


class EvidenciaV04(Evidencia):
    def __init__(self, manifiesto, textos, citas, cache_textos=512):
        super().__init__(manifiesto, textos, citas, cache_textos)
        self._identidades = {}
        self._cabeceras_v04 = {}
        self._normalizadas = {}

    def identidad(self, doc_id):
        """Cuerpos con que el evaluador puede reconocer el documento."""
        if doc_id not in self._identidades:
            doc = self.documentos[doc_id]
            cuerpos = set(identidad(doc, self.citas))
            # Alias de código solo si el título ES el código ("Código Civil colombiano"),
            # no si lo menciona ("Reforma al Código de Comercio", "desarrolla el artículo 88 de la Constitución").
            titulo = self.citas.norm(doc.get("titulo") or "")
            if titulo.startswith(("codigo", "estatuto", "constitucion")):
                del_titulo = {c for c in self.citas.bodies(self.citas.extract(titulo)) if c[0] in NOMBRES}
                if len(del_titulo) == 1:
                    cuerpos |= del_titulo
            # También si una de las primeras líneas es el nombre del código por sí solo
            # ("CODIGO SUSTANTIVO DEL TRABAJO" en el Decreto 2663 de 1950).
            lineas = [l.strip() for l in self.texto(doc_id)[:3000].splitlines() if l.strip()][:8]
            for linea in lineas:
                normal = self.citas.norm(linea).strip(' ."')
                if len(normal) <= 60 and normal.startswith(("codigo", "estatuto")):
                    de_linea = {c for c in self.citas.bodies(self.citas.extract(normal)) if c[0] in NOMBRES}
                    if len(de_linea) == 1:
                        cuerpos |= de_linea
            self._identidades[doc_id] = cuerpos
        return self._identidades[doc_id]

    def cabecera(self, doc_id):
        """Tramo literal corto que nombra la norma con todos sus alias posibles."""
        if doc_id in self._cabeceras_v04:
            return self._cabeceras_v04[doc_id]
        texto = self.texto(doc_id)
        propia = self.identidad(doc_id)
        lineas = [(m.start(), m.end(), m.group()) for m in re.finditer(r"[^\n]+", texto[:12000])]
        cubiertos = []
        for inicio, fin, linea in lineas:
            if len(linea) > 600:
                continue
            hallados = propia & self.citas.bodies(self.citas.extract(linea))
            if hallados:
                cubiertos.append((inicio, fin, hallados))
        elegida = None
        if cubiertos:
            objetivo = set().union(*(h for _, _, h in cubiertos))
            # Primer tramo contiguo que cubre todos los alias encontrados, sin exceder el máximo.
            for i, (inicio, _, _) in enumerate(cubiertos):
                vistos = set()
                for _, fin, hallados in cubiertos[i:]:
                    if fin - inicio > MAX_CABECERA:
                        break
                    vistos |= hallados
                    if vistos == objetivo:
                        elegida = (inicio, fin)
                        break
                if elegida:
                    break
            if elegida is None:
                # Si no cabe todo, preferir la línea con nombre de código.
                inicio, fin, _ = min(cubiertos, key=lambda c: (min(_prioridad(x) for x in c[2]), c[0]))
                elegida = (inicio, fin)
        if elegida is None and propia:
            elegida = self._tramo_citable(texto[:MAX_BUSQUEDA_CABECERA], propia)
        resultado = None
        if elegida:
            inicio, fin = elegida
            resultado = {"doc_id": doc_id, "inicio": inicio, "fin": fin, "texto": texto[inicio:fin],
                         "tipo_evidencia": "cabecera_fuente"}
        self._cabeceras_v04[doc_id] = resultado
        return resultado

    def _tramo_citable(self, texto, propia):
        """Tramo contiguo de líneas, el más corto, que nombra la norma completa.

        Evalúa el tramo completo y no cada línea por separado, porque la fuente
        parte la cita: "LEY 890" y "DE 2004" quedan en líneas seguidas y ninguna
        de las dos es reconocible por sí sola. Se prefiere el más corto para que
        salga el encabezado y no un párrafo que lo contenga de paso.
        """
        lineas = [(m.start(), m.end()) for m in re.finditer(r"[^\n]+", texto)]
        completo, parcial = None, None
        for i, (inicio, _) in enumerate(lineas):
            for _, fin in lineas[i:]:
                if fin - inicio > MAX_CABECERA:
                    break
                hallados = propia & self.citas.bodies(self.citas.extract(texto[inicio:fin]))
                if not hallados:
                    continue
                if hallados == propia:
                    if completo is None or fin - inicio < completo[1] - completo[0]:
                        completo = (inicio, fin)
                    break
                if parcial is None or fin - inicio < parcial[1] - parcial[0]:
                    parcial = (inicio, fin)
        return completo or parcial

    def cabecera_normalizada(self, doc_id):
        """Cita canónica del documento, tomada de los metadatos del manifiesto.

        Último recurso para las fuentes cuyo texto nunca se nombra a sí mismo en una
        forma que el extractor oficial reconozca: "LEY ESTATUTARIA 1581 DE 2012",
        "LEY 45 DE 5 DE MARZO DE 1936" o "LEY 1032" con la fecha en otra línea. No
        agrega prosa ni resume nada: es la referencia del propio documento, y va sin
        offsets porque no es un tramo literal de la fuente.
        """
        if doc_id in self._normalizadas:
            return self._normalizadas[doc_id]
        propia = self.identidad(doc_id)
        nombres = [nombre_cuerpo(c) for c in sorted(propia, key=lambda c: (_prioridad(c), str(c)))]
        canonico = "; ".join(n for n in nombres if n)
        # El título del manifiesto se lee mejor ("Sentencia C-030 de 2023" y no "C-30"),
        # así que se prefiere cuando el extractor lo reconoce como la misma norma.
        titulo = (self.documentos[doc_id].get("titulo") or "").strip()
        resultado = None
        for texto in (titulo, canonico):
            if texto and propia <= self.citas.bodies(self.citas.extract(texto)):
                resultado = {"doc_id": doc_id, "texto": texto, "tipo_evidencia": "cabecera_normalizada"}
                break
        self._normalizadas[doc_id] = resultado
        return resultado

    def reconstruir(self, pasajes_03, max_unidades=5):
        """Reutiliza las unidades recuperadas en el 03 y rehace sus cabeceras."""
        unidades = [p for p in pasajes_03 if p.get("tipo_evidencia") not in CABECERAS][:max_unidades]
        pasajes, avisos = [], []
        for unidad in unidades:
            if self.texto(unidad["doc_id"])[unidad["inicio"]:unidad["fin"]] != unidad["texto"]:
                raise ValueError("El artículo no conserva el texto literal")
            necesarios = self.identidad(unidad["doc_id"])
            en_texto = self.citas.bodies(self.citas.extract(unidad["texto"]))
            if not necesarios <= en_texto:
                cabecera = self.cabecera(unidad["doc_id"]) or self.cabecera_normalizada(unidad["doc_id"])
                if cabecera:
                    pasajes.append(dict(cabecera, grupo_evidencia=unidad.get("unidad_id"), score=unidad.get("score")))
                    if cabecera["tipo_evidencia"] == "cabecera_normalizada":
                        avisos.append({"doc_id": unidad["doc_id"], "motivo": "cabecera_normalizada_del_manifiesto"})
                else:
                    avisos.append({"doc_id": unidad["doc_id"], "motivo": "sin_cabecera_literal_reconocible"})
            pasajes.append(dict(unidad, tipo_evidencia="unidad"))
        if len(pasajes) > 10:
            raise ValueError("Se superaron los diez pasajes de evidencia")
        self.verificar(pasajes)
        return pasajes, avisos

    def respaldo(self, pasajes):
        """Citas que el evaluador considera respaldadas por estos pasajes."""
        return set().union(*(self.citas.bodies(self.citas.extract(str(p.get("texto") or "")))
                             for p in (pasajes or [])[:10])) if pasajes else set()

    def etiqueta(self, pasaje):
        """Nombre legible de la norma de un pasaje de tipo unidad, verificado contra el extractor."""
        respaldadas = self.identidad(pasaje["doc_id"])
        for cuerpo in sorted(respaldadas, key=lambda c: (_prioridad(c), str(c))):
            nombre = nombre_cuerpo(cuerpo)
            if nombre and cuerpo in self.citas.bodies(self.citas.extract(nombre)):
                return nombre, cuerpo
        doc = self.documentos[pasaje["doc_id"]]
        return doc.get("titulo") or pasaje["doc_id"], None

    def citas_evidencia(self, pasajes, limite=None, solo_cuerpos=None, incluir_primera=False):
        """Frase de fundamento construida solo con las unidades recuperadas.

        Devuelve el texto y los cuerpos que cita. Cada cuerpo se incluye solo si
        el extractor oficial lo encuentra también en los pasajes (sin respaldo = 0).
        """
        respaldo = self.respaldo(pasajes)
        grupos = OrderedDict()
        for pasaje in pasajes:
            if pasaje.get("tipo_evidencia") in CABECERAS:
                continue
            nombre, cuerpo = self.etiqueta(pasaje)
            if cuerpo is None or cuerpo not in respaldo:
                continue
            if solo_cuerpos is not None and cuerpo not in solo_cuerpos and not (incluir_primera and not grupos):
                continue
            articulos = grupos.setdefault((nombre, cuerpo), [])
            articulo = str(pasaje.get("articulo") or "").strip()
            numero = re.match(r"\d+[A-Za-z]?", articulo)
            if numero and numero.group() not in articulos:
                articulos.append(numero.group())
            if limite and len(grupos) >= limite:
                break
        partes = []
        for (nombre, cuerpo), articulos in grupos.items():
            if not articulos:
                partes.append(nombre)
            elif len(articulos) == 1:
                partes.append(f"{nombre}, artículo {articulos[0]}")
            else:
                partes.append(f"{nombre}, artículos {', '.join(articulos[:-1])} y {articulos[-1]}")
        texto = "; ".join(partes)
        cuerpos = self.citas.bodies(self.citas.extract(texto)) if texto else set()
        if cuerpos - respaldo:
            # Nunca debería ocurrir; si ocurre, no se agrega nada.
            return "", set()
        return texto, cuerpos


**politica**. Prompt v04 y postproceso: letra siempre, JSON reparado, citas saneadas, fundamento desde la evidencia.

In [ ]:
%%modulo produccion.politica
"""Política de generación v04 (copia de legalrag.generation.politica).

Se copia para que producción no dependa de cambios del repo. Solo cambia el import.

Generación de la versión 04.

Cambios frente al 03, cada uno ligado a una falla observada:
  * El modelo ya no decide la abstención. El campo `abstencion` sale del esquema
    y lo fija el código (regla general, no por pregunta).
  * En cerradas, `respuesta_correcta` es un enum sin null: siempre hay letra.
  * Cada texto tiene maxLength y la petición usa repeat_penalty, para cortar la
    repetición que agotaba los 1024 tokens y dejaba JSON inválido.
  * Si aun así el JSON queda truncado, se repara en lugar de descartar.
  * Una cita sin respaldo ya no anula la respuesta: se elimina la oración que la
    contiene y se conserva el resto.
  * El fundamento normativo se completa con las normas de los pasajes
    recuperados, en la forma que reconoce el evaluador oficial.
  * El prompt es texto plano con pasajes numerados, no un JSON con metadatos.
"""
import hashlib
import json
import re
import time

from produccion.evidencia import CABECERAS, NOMBRES

CAMPOS = {
    "multiple_choice": ["respuesta_correcta", "justificacion", "descarte_opciones"],
    "semi_open": ["respuesta", "palabras_clave", "referencia_legal"],
    "open_ended": ["marco_normativo", "analisis", "jurisprudencia", "conclusion"],
}

SISTEMA = (
    "Eres un abogado colombiano que responde preguntas de examen. Razonas con cuidado y "
    "respondes en español. Usas como fundamento los pasajes numerados que se te entregan. "
    "Cuando cites una norma, escríbela con su nombre y artículo tal como aparece en el "
    "encabezado del pasaje (por ejemplo: \"artículo 3 de la Ley 472 de 1998\" o "
    "\"artículo 946 del Código Civil\"). No inventes números de leyes, sentencias ni "
    "artículos que no aparezcan en los pasajes."
)

INSTRUCCIONES = {
    "multiple_choice": (
        "Elige exactamente una opción. Debes elegir siempre la opción más probable, incluso si "
        "los pasajes no la respaldan por completo: nunca dejes la pregunta sin responder.\n"
        "En \"justificacion\" explica en 2 a 4 oraciones por qué la opción es correcta, citando "
        "la norma y el artículo de los pasajes que la sustentan.\n"
        "En \"descarte_opciones\" da una razón breve (una oración) para cada opción."
    ),
    "semi_open": (
        "En \"respuesta\" contesta en 3 a 5 oraciones (máximo 150 palabras), de forma directa y "
        "citando la norma y el artículo de los pasajes que la sustentan.\n"
        "En \"palabras_clave\" da de 3 a 6 términos jurídicos centrales.\n"
        "En \"referencia_legal\" escribe la norma y el artículo principal."
    ),
    "open_ended": (
        "En \"marco_normativo\" identifica las normas aplicables de los pasajes (1 a 3 oraciones).\n"
        "En \"analisis\" desarrolla el razonamiento jurídico en 5 a 8 oraciones.\n"
        "En \"jurisprudencia\" menciona solo sentencias que aparezcan en los pasajes; si no hay, "
        "escribe una oración sobre el criterio jurisprudencial general sin inventar números.\n"
        "En \"conclusion\" responde la pregunta en 1 a 3 oraciones."
    ),
}

LONGITUDES = {
    "justificacion": 900, "descarte": 220, "respuesta": 1100, "palabra": 50,
    "referencia_legal": 250, "marco_normativo": 700, "analisis": 1600,
    "jurisprudencia": 600, "conclusion": 600,
}


def esquema(formato, letras, longitudes=True):
    """Esquema JSON para la gramática de llama.cpp. Sin campo de abstención."""
    L = LONGITUDES

    def texto(clave):
        s = {"type": "string", "minLength": 1}
        if longitudes:
            s["maxLength"] = L[clave]
        return s

    if formato == "multiple_choice":
        props = {
            "justificacion": texto("justificacion"),
            "respuesta_correcta": {"type": "string", "enum": list(letras)},
            "descarte_opciones": {"type": "object",
                                  "properties": {l: texto("descarte") for l in letras},
                                  "required": list(letras), "additionalProperties": False},
        }
    elif formato == "semi_open":
        palabras = {"type": "array", "items": texto("palabra"), "minItems": 1}
        if longitudes:
            palabras["maxItems"] = 6
        props = {"respuesta": texto("respuesta"), "palabras_clave": palabras,
                 "referencia_legal": texto("referencia_legal")}
    else:
        props = {k: texto(k) for k in ("marco_normativo", "analisis", "jurisprudencia", "conclusion")}
    return {"type": "object", "properties": props, "required": list(props), "additionalProperties": False}


def bloque_pasajes(pasajes, evidencia, max_caracteres=1800):
    """Pasajes numerados. Solo las unidades; la cabecera se usa para nombrar la norma."""
    lineas, numero = [], 0
    for pasaje in pasajes:
        if pasaje.get("tipo_evidencia") in CABECERAS:
            continue
        numero += 1
        nombre, _ = evidencia.etiqueta(pasaje)
        articulo = pasaje.get("articulo")
        titulo = f"{nombre}, artículo {articulo}" if articulo else nombre
        texto = re.sub(r"\n{3,}", "\n\n", pasaje["texto"].strip())
        if len(texto) > max_caracteres:
            texto = texto[:max_caracteres].rsplit(" ", 1)[0] + " […]"
        lineas.append(f"[{numero}] {titulo}\n{texto}")
    return "\n\n".join(lineas)


def mensajes(entrada, pasajes, evidencia, max_caracteres=1800):
    formato = entrada["formato"]
    partes = ["PASAJES RECUPERADOS", bloque_pasajes(pasajes, evidencia, max_caracteres) or "(ninguno)", ""]
    tipo = {"multiple_choice": "selección múltiple", "semi_open": "respuesta corta",
            "open_ended": "respuesta abierta"}[formato]
    partes += [f"PREGUNTA ({tipo}, {entrada.get('area', '')})", entrada["pregunta"].strip()]
    for letra, opcion in (entrada.get("opciones") or {}).items():
        partes.append(f"{letra}) {opcion}")
    partes += ["", "INSTRUCCIONES", INSTRUCCIONES[formato],
               "Responde solo con el objeto JSON, con las claves: " + ", ".join(
                   k for k in esquema(formato, list((entrada.get("opciones") or {"A": 0}).keys()))["properties"]) + "."]
    return [{"role": "system", "content": SISTEMA}, {"role": "user", "content": "\n".join(partes)}]


def ajustar(cliente, entrada, pasajes, evidencia, contexto, salida, max_caracteres=1800):
    """Quita unidades desde la última hasta que el prompt cabe en el contexto."""
    limite = contexto - salida - 32
    unidades = [p for p in pasajes if p.get("tipo_evidencia") not in CABECERAS]
    omitidas = []
    while True:
        usados_ids = {id(u) for u in unidades}
        usados = [p for p in pasajes if p.get("tipo_evidencia") in CABECERAS or id(p) in usados_ids]
        # Las cabeceras huérfanas se descartan.
        grupos = {u.get("grupo_evidencia", u.get("unidad_id")) for u in unidades}
        usados = [p for p in usados if p.get("tipo_evidencia") not in CABECERAS or p.get("grupo_evidencia") in grupos]
        conversacion = mensajes(entrada, usados, evidencia, max_caracteres)
        tokens = cliente.contar(conversacion)
        if tokens <= limite or not unidades:
            return usados, conversacion, tokens, omitidas
        omitidas.append(unidades.pop().get("unidad_id"))


def reparar_json(texto):
    """Intenta leer un objeto JSON truncado cerrando comillas y llaves."""
    texto = (texto or "").strip()
    try:
        return json.loads(texto), False
    except json.JSONDecodeError:
        pass
    pila, en_cadena, escape = [], False, False
    for c in texto:
        if en_cadena:
            if escape:
                escape = False
            elif c == "\\":
                escape = True
            elif c == '"':
                en_cadena = False
        elif c == '"':
            en_cadena = True
        elif c in "{[":
            pila.append("}" if c == "{" else "]")
        elif c in "}]" and pila:
            pila.pop()
    candidato = texto.rstrip()
    if escape:
        candidato = candidato[:-1]
    if en_cadena:
        candidato += '"'
    candidato = re.sub(r",\s*$", "", candidato)
    for intento in (candidato + "".join(reversed(pila)),
                    re.sub(r',\s*"[^"]*"\s*:?\s*"?$', "", candidato) + "".join(reversed(pila))):
        try:
            return json.loads(intento), True
        except json.JSONDecodeError:
            continue
    return None, True


_ORACION = re.compile(r"(?<=[.;:!?])\s+(?=[A-ZÁÉÍÓÚÑ¿(\"“])")


def sanear(texto, respaldo, citas):
    """Quita las oraciones con citas que el evaluador contaría sin respaldo."""
    if not isinstance(texto, str) or not texto:
        return texto, []
    eliminadas, conservadas = [], []
    for oracion in _ORACION.split(texto.strip()):
        cuerpos = citas.bodies(citas.extract(oracion))
        if cuerpos - respaldo:
            eliminadas.append(oracion)
        else:
            conservadas.append(oracion)
    return " ".join(conservadas).strip(), eliminadas


_PATRONES_CITA = [
    (re.compile(r"(?i)\b(?:la\s+)?sentencia\s+(?:de\s+unificaci[oó]n\s+)?(?:C|T|SU|SL|SC|SP|STC|STL|STP|AC|AL|AP|AU)"
                r"\s*-?\s*\d{1,5}\s*(?:de|del|/|-)\s*\d{2,4}\b"), "la jurisprudencia aplicable"),
    (re.compile(r"(?i)\b(?:C|T|SU|SL|SC|SP|STC|STL)\s*-\s*\d{1,5}\s*(?:de|/|-)\s*\d{2,4}\b"), "la jurisprudencia aplicable"),
    (re.compile(r"(?i)\b(?:la\s+|el\s+)?(?:ley|decreto(?:[\s-]+ley)?|acto\s+legislativo|resoluci[oó]n|acuerdo|circular(?:\s+externa)?)"
                r"\s+(?:n[°ºo]\.?\s*)?\d{1,5}\s*(?:de|del|/|-)\s*\d{4}\b"), "la norma aplicable"),
]
_TILDES = {"a": "[aá]", "e": "[eé]", "i": "[ií]", "o": "[oó]", "u": "[uúü]", "n": "[nñ]"}


def _patron_flexible(variante):
    """Regex que acepta tildes y mayúsculas para un alias del extractor ("codigo civil")."""
    cuerpo = "".join(_TILDES.get(c, re.escape(c)) if c != " " else r"\s+" for c in variante)
    return re.compile(r"(?i)(?<![\w.])(?:el\s+|la\s+)?" + cuerpo + r"(?!\w)")


def quitar_citas(texto, respaldo, citas):
    """Reemplaza solo las menciones de normas sin respaldo, conservando el razonamiento.

    Devuelve None si después del reemplazo aún quedan citas sin respaldo.
    """
    def reemplazo(m, generico):
        cuerpos = citas.bodies(citas.extract(m.group(0)))
        return generico if cuerpos and not cuerpos <= respaldo else m.group(0)

    for patron, generico in _PATRONES_CITA:
        texto = patron.sub(lambda m: reemplazo(m, generico), texto)
    # Códigos por nombre: solo los alias de códigos sin respaldo, del más largo al más corto.
    sin_respaldo = {c[0] for c in citas.bodies(citas.extract(texto)) - respaldo}
    for codigo in sin_respaldo & set(citas.CODES):
        for variante in sorted(citas.CODES[codigo], key=len, reverse=True):
            if len(variante) > 4:  # los alias cortos ("cc", "et") son demasiado ambiguos para reemplazar
                texto = _patron_flexible(variante).sub("la legislación aplicable", texto)
    return None if citas.bodies(citas.extract(texto)) - respaldo else texto


def sanear_cita(texto, respaldo, citas):
    """Como sanear(), pero intenta primero quitar solo la cita y no toda la oración."""
    if not isinstance(texto, str) or not texto:
        return texto, []
    eliminadas, conservadas = [], []
    for oracion in _ORACION.split(texto.strip()):
        if not citas.bodies(citas.extract(oracion)) - respaldo:
            conservadas.append(oracion)
            continue
        limpia = quitar_citas(oracion, respaldo, citas)
        eliminadas.append(oracion)
        if limpia:
            conservadas.append(limpia[0].upper() + limpia[1:])
    return " ".join(conservadas).strip(), eliminadas


def sanear_codigos(texto, respaldo, citas):
    """Como sanear_cita(), pero tolera las citas a códigos por su nombre aunque no estén en la evidencia."""
    if not isinstance(texto, str) or not texto:
        return texto, []
    codigos = {c for c in citas.bodies(citas.extract(texto)) if c[0] in NOMBRES}
    return sanear_cita(texto, respaldo | codigos, citas)


def _anexar(texto, extra, conector):
    texto = (texto or "").strip()
    if not extra:
        return texto
    if texto and not texto.endswith((".", ";", ":")):
        texto += "."
    return (texto + " " if texto else "") + conector + extra + "."


def postprocesar(entrada, crudo, pasajes, evidencia, politica):
    """Convierte la salida del modelo en una respuesta del esquema oficial.

    politica: dict con
        citar_evidencia: "todas" | "ninguna" | int (máximo de normas)
        abstener_libre: "nunca" | "sin_evidencia"
    """
    formato = entrada["formato"]
    citas = evidencia.citas
    respaldo = evidencia.respaldo(pasajes)
    letras = list((entrada.get("opciones") or {}).keys())
    registro = {"oraciones_eliminadas": [], "campos_rellenados": []}
    unidades = [p for p in pasajes if p.get("tipo_evidencia") not in CABECERAS]

    if formato != "multiple_choice" and politica.get("abstener_libre") == "sin_evidencia" and not unidades:
        registro["motivo"] = "sin_evidencia"
        return {"id": entrada["id"], **abstencion(formato, pasajes)}, registro

    salida = {k: v for k, v in (crudo or {}).items() if k in CAMPOS[formato]}

    campos_texto = {"multiple_choice": ["justificacion"], "semi_open": ["respuesta", "referencia_legal"],
                    "open_ended": ["marco_normativo", "analisis", "jurisprudencia", "conclusion"]}[formato]
    # saneo: "oracion" (04: se borra la oración con la cita) | "cita" (05: se reemplaza solo la cita)
    #        | "codigos" (e07: como "cita", pero conserva los códigos nombrados sin artículo en la evidencia)
    #        | "ninguno" (e07: no toca el texto; solo para medir cuánto cuesta no sanear)
    # Una cita correcta sin respaldo vale 0,5 y una incorrecta sin respaldo penaliza; "codigos"
    # apuesta a que los códigos generales (Constitución, Código Civil...) suelen ser correctos.
    funcion_saneo = {"cita": sanear_cita, "codigos": sanear_codigos,
                     "ninguno": lambda texto, respaldo, citas: (texto, [])}.get(politica.get("saneo", "oracion"), sanear)
    for campo in campos_texto:
        salida[campo], quitadas = funcion_saneo(salida.get(campo, ""), respaldo, citas)
        registro["oraciones_eliminadas"] += quitadas

    # citar_evidencia: "todas" | "ninguna" | int | "usadas" (normas que el modelo nombró + la primera recuperada)
    limite = politica.get("citar_evidencia", "todas")
    if limite == "ninguna":
        fundamento = ""
    elif limite == "usadas":
        nombradas = citas.bodies(citas.extract(" ".join(str(salida.get(c) or "") for c in campos_texto)))
        fundamento = evidencia.citas_evidencia(pasajes, solo_cuerpos=nombradas, incluir_primera=True)[0]
    else:
        fundamento = evidencia.citas_evidencia(pasajes, limite=None if limite == "todas" else int(limite))[0]

    if formato == "multiple_choice":
        letra = salida.get("respuesta_correcta")
        if letra not in letras:
            letra = letras[0]
            registro["campos_rellenados"].append("respuesta_correcta")
        salida["respuesta_correcta"] = letra
        descarte = {k: v for k, v in (salida.get("descarte_opciones") or {}).items()
                    if k in letras and k != letra and isinstance(v, str) and v.strip()}
        for k in letras:
            if k != letra and k not in descarte:
                descarte[k] = "No corresponde al supuesto descrito en la pregunta."
                registro["campos_rellenados"].append(f"descarte_{k}")
        salida["descarte_opciones"] = {k: sanear(v, respaldo, citas)[0] or "No corresponde al supuesto descrito."
                                       for k, v in sorted(descarte.items())}
        if not salida.get("justificacion"):
            salida["justificacion"] = f"La opción {letra} es la que mejor se ajusta a los pasajes recuperados."
            registro["campos_rellenados"].append("justificacion")
        salida["justificacion"] = _anexar(salida["justificacion"], fundamento, "Fundamento: ")
    elif formato == "semi_open":
        if not salida.get("respuesta"):
            salida["respuesta"] = "Según los pasajes recuperados, la respuesta depende de las normas citadas."
            registro["campos_rellenados"].append("respuesta")
        palabras = [p for p in (salida.get("palabras_clave") or []) if isinstance(p, str) and p.strip()]
        salida["palabras_clave"] = palabras or [entrada.get("tema") or entrada.get("area") or "derecho colombiano"]
        referencia = salida.get("referencia_legal") or ""
        salida["referencia_legal"] = (fundamento or referencia or "Pasajes recuperados")
    else:
        for campo, defecto in (("marco_normativo", "Normas contenidas en los pasajes recuperados."),
                               ("analisis", "El análisis se basa en los pasajes recuperados."),
                               ("jurisprudencia", "Los pasajes recuperados no incluyen jurisprudencia específica."),
                               ("conclusion", "La conclusión se deriva de las normas citadas.")):
            if not salida.get(campo):
                salida[campo] = defecto
                registro["campos_rellenados"].append(campo)
        salida["marco_normativo"] = _anexar(salida["marco_normativo"], fundamento, "Normas aplicables: ")

    respuesta = {"id": entrada["id"], "formato": formato, **{k: salida[k] for k in CAMPOS[formato]},
                 "abstencion": False, "pasajes_recuperados": pasajes}
    return respuesta, registro


def abstencion(formato, pasajes):
    salida = {k: "" for k in CAMPOS[formato]}
    if formato == "multiple_choice":
        salida.update(respuesta_correcta=None, descarte_opciones={})
    if formato == "semi_open":
        salida["palabras_clave"] = []
    return {"formato": formato, **salida, "abstencion": True, "pasajes_recuperados": pasajes}


def generar(cliente, entrada, pasajes, evidencia, config):
    """Una llamada al servidor local de llama.cpp. Devuelve (salida_cruda, registro)."""
    inicio = time.perf_counter()
    letras = list((entrada.get("opciones") or {}).keys())
    usados, conversacion, tokens, omitidas = ajustar(
        cliente, entrada, pasajes, evidencia, config["contexto"], config["max_tokens"], config["max_caracteres_pasaje"])
    prompt = cliente.aplicar_plantilla(conversacion)
    peticion = {"prompt": prompt, "temperature": 0, "seed": 0, "n_predict": config["max_tokens"],
                "cache_prompt": False, "stream": False,
                "repeat_penalty": config["repeat_penalty"], "repeat_last_n": config["repeat_last_n"],
                "frequency_penalty": 0, "presence_penalty": 0,
                "json_schema": esquema(entrada["formato"], letras, config["longitudes"])}
    registro = {"tokens_prompt": tokens, "omitidas": omitidas, "peticion": {k: v for k, v in peticion.items() if k != "prompt"},
                "sha256_prompt": hashlib.sha256(prompt.encode()).hexdigest()}
    try:
        respuesta, _ = cliente.pedir("/completion", peticion)
    except Exception as error:  # gramática rechazada por el servidor: repetir sin maxLength
        import urllib.error
        if not isinstance(error, urllib.error.HTTPError) or not config["longitudes"]:
            raise
        peticion["json_schema"] = esquema(entrada["formato"], letras, False)
        registro["gramatica_sin_longitudes"] = f"{error.code}: {error.read()[:300]!r}"
        respuesta, _ = cliente.pedir("/completion", peticion)
    contenido = respuesta.get("content") or ""
    crudo, reparado = reparar_json(contenido)
    registro.update(contenido=contenido, reparado=reparado, json_valido=crudo is not None,
                    finish_reason=respuesta.get("stop_type"),
                    tokens_salida=respuesta.get("tokens_predicted"),
                    segundos=time.perf_counter() - inicio)
    return crudo, usados, registro


**modelos**. Encoders (mismos prefijos y pooling del e06), rerankers BGE y Qwen, determinismo.

In [ ]:
%%modulo produccion.modelos
"""Modelos abiertos del sistema: encoders, rerankers y decoders.

Los encoders reproducen exactamente corpus_definitivo.Encoder del e06 (prefijos,
pooling y longitudes), porque un índice construido de una forma y consultado de
otra pierde calidad sin avisar.

Los nombres que empiezan por "simulado/" cargan modelos falsos sin torch
(experimentos/e07/simulado.py). Sirven para probar el pipeline en CPU.

Determinismo: consultas con lote 1 y reranker con lote fijo. En bf16 el puntaje de
un par cambia unas milésimas según con qué otros pares comparte lote, y eso basta
para invertir dos pasajes en la verificación en vivo del sábado.
"""
import gc
import os

INSTRUCCION_QWEN_EMB = "Instruct: Recupera pasajes jurídicos colombianos pertinentes a la pregunta\nQuery: "
INSTRUCCION_QWEN_RERANK = ("Dada una pregunta jurídica colombiana, identifica pasajes que permitan "
                           "responderla con fundamento literal.")


def es_simulado(nombre):
    return str(nombre).startswith("simulado/")


def fijar_determinismo(semilla=0):
    """Mismas salidas en la corrida del sábado y en la regeneración del jurado."""
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    import random
    random.seed(semilla)
    try:
        import numpy as np
        np.random.seed(semilla)
    except ImportError:
        pass
    try:
        import torch
    except ImportError:
        return
    torch.manual_seed(semilla)
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def _dtype():
    import torch
    return torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16


def liberar(*objetos):
    for o in objetos:
        del o
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


class Encoder:
    """Mismo comportamiento que corpus_definitivo.Encoder (e06)."""

    def __init__(self, nombre, revision=None, lote=24, ruta_local=None):
        import torch
        from transformers import AutoModel, AutoTokenizer
        self.torch = torch
        self.nombre = nombre
        self.e5 = "multilingual-e5" in nombre
        self.qwen = "Qwen3-Embedding" in nombre
        origen = ruta_local or nombre
        self.tokenizer = AutoTokenizer.from_pretrained(origen, revision=None if ruta_local else revision)
        self.modelo = AutoModel.from_pretrained(origen, revision=None if ruta_local else revision,
                                                dtype=_dtype()).to("cuda").eval()
        self.max_tokens = 512 if self.e5 else 1024
        self.lote = lote
        self.dimension = self.modelo.config.hidden_size

    def _preparar(self, textos, tarea):
        if self.e5:
            return [("query: " if tarea == "query" else "passage: ") + x for x in textos]
        if self.qwen and tarea == "query":
            return [INSTRUCCION_QWEN_EMB + x for x in textos]
        return list(textos)

    def _lote(self, textos):
        torch = self.torch
        tok = self.tokenizer(textos, padding=True, truncation=True, max_length=self.max_tokens,
                             return_tensors="pt").to("cuda")
        with torch.inference_mode():
            oculto = self.modelo(**tok).last_hidden_state.float()
            if self.e5:
                mascara = tok["attention_mask"].unsqueeze(-1)
                emb = (oculto * mascara).sum(1) / mascara.sum(1)
            elif self.qwen:
                fines = tok["attention_mask"].sum(1) - 1
                if self.tokenizer.padding_side == "left":
                    fines = torch.full_like(fines, oculto.shape[1] - 1)
                emb = oculto[torch.arange(oculto.shape[0], device="cuda"), fines]
            else:
                emb = oculto[:, 0]
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        return emb.cpu().numpy().astype("float32")

    def codificar(self, textos, tarea="passage"):
        """Pasajes en lotes (con reducción ante OOM); consultas siempre de a una."""
        import numpy as np
        torch = self.torch
        textos = self._preparar(textos, tarea)
        if tarea == "query":
            return np.vstack([self._lote([t]) for t in textos])
        salida, i = [], 0
        while i < len(textos):
            try:
                salida.append(self._lote(textos[i:i + self.lote]))
                i += len(salida[-1])
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if self.lote == 1:
                    raise
                self.lote = max(1, self.lote // 2)
                print(f"VRAM llena; lote del encoder reducido a {self.lote}")
        return np.vstack(salida)

    def cerrar(self):
        modelo, self.modelo = self.modelo, None
        liberar(modelo)


class RerankerBGE:
    def __init__(self, nombre, revision=None, max_tokens=1024, lote=16, ruta_local=None):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.torch = torch
        self.nombre = nombre
        origen = ruta_local or nombre
        rev = None if ruta_local else revision
        self.tokenizer = AutoTokenizer.from_pretrained(origen, revision=rev)
        self.modelo = AutoModelForSequenceClassification.from_pretrained(
            origen, revision=rev, dtype=_dtype()).to("cuda").eval()
        self.max_tokens = max_tokens
        self.lote = lote  # fijo: no se reduce ante OOM para no cambiar los puntajes

    def puntuar(self, consulta, textos):
        torch = self.torch
        puntajes = []
        for i in range(0, len(textos), self.lote):
            pares = [(consulta, t) for t in textos[i:i + self.lote]]
            tok = self.tokenizer(pares, padding=True, truncation=True, max_length=self.max_tokens,
                                 return_tensors="pt").to("cuda")
            with torch.inference_mode():
                puntajes.extend(self.modelo(**tok).logits.float().view(-1).cpu().tolist())
        return puntajes

    def cerrar(self):
        modelo, self.modelo = self.modelo, None
        liberar(modelo)


class RerankerQwen:
    """Qwen3-Reranker: probabilidad de "yes" con la plantilla publicada (igual que e06)."""

    def __init__(self, nombre, revision=None, max_tokens=1024, lote=4, ruta_local=None):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.torch = torch
        self.nombre = nombre
        origen = ruta_local or nombre
        rev = None if ruta_local else revision
        self.tokenizer = AutoTokenizer.from_pretrained(origen, revision=rev, padding_side="left")
        self.modelo = AutoModelForCausalLM.from_pretrained(origen, revision=rev, dtype=_dtype()).to("cuda").eval()
        tok = self.tokenizer
        self.si, self.no = tok.convert_tokens_to_ids("yes"), tok.convert_tokens_to_ids("no")
        self.prefijo = tok.encode(
            '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and '
            'the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n'
            '<|im_start|>user\n', add_special_tokens=False)
        self.sufijo = tok.encode("<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n",
                                 add_special_tokens=False)
        self.max_tokens = max_tokens
        self.lote = lote

    def puntuar(self, consulta, textos):
        torch = self.torch
        ids = []
        for t in textos:
            cuerpo = f"<Instruct>: {INSTRUCCION_QWEN_RERANK}\n<Query>: {consulta}\n<Document>: {t}"
            ids.append(self.prefijo + self.tokenizer.encode(cuerpo, add_special_tokens=False)[:self.max_tokens]
                       + self.sufijo)
        puntajes = []
        for i in range(0, len(ids), self.lote):
            entrada = self.tokenizer.pad({"input_ids": ids[i:i + self.lote]}, padding=True,
                                         return_tensors="pt").to("cuda")
            with torch.inference_mode():
                logits = self.modelo(**entrada).logits[:, -1, [self.no, self.si]].float()
                puntajes.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().tolist())
        return puntajes

    def cerrar(self):
        modelo, self.modelo = self.modelo, None
        liberar(modelo)


def cargar_encoder(nombre, revision=None, lote=24, ruta_local=None):
    if es_simulado(nombre):
        from experimentos.e07.simulado import EncoderSimulado
        return EncoderSimulado(nombre)
    return Encoder(nombre, revision, lote, ruta_local)


def cargar_reranker(nombre, revision=None, max_tokens=1024, lote=16, ruta_local=None):
    """nombre puede ser un id de Hugging Face o una carpeta local de fine-tuning."""
    if es_simulado(nombre):
        from experimentos.e07.simulado import RerankerSimulado
        return RerankerSimulado(nombre)
    if "Qwen3-Reranker" in nombre:
        return RerankerQwen(nombre, revision, max_tokens, max(1, lote // 4), ruta_local)
    return RerankerBGE(nombre, revision, max_tokens, lote, ruta_local)


def parametros_modelo(modelo):
    return sum(p.numel() for p in modelo.parameters())


**recuperador**. Consultas, BM25, FAISS, RRF, reranking, router de artículos, normas ancla y selección de pasajes.

In [ ]:
%%modulo produccion.recuperador
"""Recuperación configurable. La misma lógica corre en los experimentos (e07) y en producción.

Configuración de recuperación (R), por ejemplo la R03 del e06:
    {"consulta": "pregunta_opciones", "bm25": true, "densos": ["BAAI/bge-m3"],
     "reranker": "BAAI/bge-reranker-v2-m3", "k_bm25": 100, "k_denso": 100,
     "k_rrf": 100, "rrf_constante": 60, "top_rerank": 50, "expansion_area": 0}

Configuración de pasajes (P):
    {"modo": "v04" | "e06", "max_pasajes": 10, "max_unidades": 8, "max_por_doc": null,
     "max_caracteres_unidad": 2400, "cabecera_normalizada": true,
     "router_articulo": false, "anclas_area": 0}

Las fuentes (BM25, índices densos, rerankers) se inyectan: en producción llaman a
los modelos cargados; en los experimentos leen primero una caché en disco.
"""
import json
import re
import sqlite3
from collections import defaultdict
from pathlib import Path

from produccion.evidencia import CABECERAS

R_BASE = {"consulta": "pregunta_opciones", "bm25": True, "densos": ["BAAI/bge-m3"],
          "reranker": "BAAI/bge-reranker-v2-m3", "k_bm25": 100, "k_denso": 100, "k_rrf": 100,
          "rrf_constante": 60, "top_rerank": 50, "expansion_area": 0}
P_BASE = {"modo": "v04", "max_pasajes": 10, "max_unidades": 8, "max_por_doc": None,
          "max_caracteres_unidad": 2400, "cabecera_normalizada": True, "router_articulo": False,
          "anclas_area": 0}


def completar(base, cambios):
    salida = dict(base)
    salida.update(cambios or {})
    return salida


# ---------------------------------------------------------------- consultas

def texto_opciones(entrada):
    opciones = entrada.get("opciones") or {}
    return entrada["pregunta"] + "\n" + "\n".join(f"{k}. {v}" for k, v in opciones.items())


def consultas(entrada, modo):
    """Textos de consulta. "pregunta_opciones" es exactamente la consulta del e06."""
    if modo == "pregunta_opciones":
        return [texto_opciones(entrada)]
    if modo == "pregunta":
        return [entrada["pregunta"]]
    if modo == "por_opcion":
        opciones = entrada.get("opciones") or {}
        if not opciones:
            return [texto_opciones(entrada)]
        return [texto_opciones(entrada)] + [f"{entrada['pregunta']}\n{k}. {v}" for k, v in opciones.items()]
    raise ValueError("modo de consulta desconocido: " + modo)


def rrf(*rankings, k=100, constante=60):
    fusion = defaultdict(float)
    for ranking in rankings:
        for posicion, (chunk, _) in enumerate(ranking, 1):
            fusion[chunk] += 1 / (constante + posicion)
    return sorted(fusion.items(), key=lambda x: (-x[1], x[0]))[:k]


# ---------------------------------------------------------------- normas ancla por área

def anclas_por_area(ruta_seed_targets):
    """Normas más citadas del banco para cada área, según el material oficial (seed_targets.json).

    Es la distribución pública del banco completo, no respuestas de ítems concretos.
    """
    if not Path(ruta_seed_targets).is_file():
        return {}
    documentos = json.loads(Path(ruta_seed_targets).read_text(encoding="utf-8"))["documentos"]
    por_area = defaultdict(list)
    for d in sorted(documentos, key=lambda x: -x["items_del_banco"]):
        for area in d["areas"]:
            por_area[area].append((tuple(d["canonico"]), d["norma"]))
    return dict(por_area)


# ---------------------------------------------------------------- índice léxico

class Lexico:
    def __init__(self, ruta_sqlite):
        if not Path(ruta_sqlite).is_file():
            raise FileNotFoundError(ruta_sqlite)
        uri = Path(ruta_sqlite).resolve().as_uri() + "?mode=ro"
        self.conn = sqlite3.connect(uri, uri=True, check_same_thread=False, timeout=120)
        self.conn.row_factory = sqlite3.Row

    def total(self):
        return self.conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]

    @staticmethod
    def expresion(texto):
        terminos = re.findall(r"\w+", texto.casefold())
        return " OR ".join('"' + t.replace('"', "") + '"' for t in dict.fromkeys(terminos))

    def buscar(self, texto, k=100):
        expresion = self.expresion(texto)
        if not expresion:
            return []
        return [(int(r[0]), float(-r[1])) for r in self.conn.execute(
            "SELECT rowid, bm25(fts) FROM fts WHERE fts MATCH ? ORDER BY bm25(fts) LIMIT ?",
            (expresion, k)).fetchall()]

    def buscar_en_docs(self, texto, doc_ids, k=5):
        """BM25 restringido a unos documentos (normas ancla)."""
        expresion = self.expresion(texto)
        if not expresion or not doc_ids:
            return []
        marcas = ",".join("?" for _ in doc_ids)
        return [(int(r[0]), float(-r[1])) for r in self.conn.execute(
            f"SELECT fts.rowid, bm25(fts) FROM fts JOIN chunks ON chunks.id = fts.rowid "
            f"WHERE fts MATCH ? AND chunks.doc_id IN ({marcas}) ORDER BY bm25(fts) LIMIT ?",
            (expresion, *doc_ids, k)).fetchall()]

    def filas(self, ids):
        ids = list(ids)
        if not ids:
            return []
        filas = {}
        for i in range(0, len(ids), 900):  # límite de variables de SQLite
            parte = ids[i:i + 900]
            marcas = ",".join("?" for _ in parte)
            filas.update({r["id"]: dict(r) for r in self.conn.execute(
                f"SELECT * FROM chunks WHERE id IN ({marcas})", parte)})
        return [filas[i] for i in ids if i in filas]

    def articulo(self, doc_ids, numero, limite=2):
        """Primer chunk de cada unidad cuyo artículo coincide con el número pedido."""
        if not doc_ids:
            return []
        marcas = ",".join("?" for _ in doc_ids)
        filas = self.conn.execute(
            f"SELECT * FROM chunks WHERE doc_id IN ({marcas}) AND articulo IS NOT NULL ORDER BY id",
            list(doc_ids)).fetchall()
        vistas, salida = set(), []
        for f in filas:
            if normalizar_articulo(f["articulo"]) == normalizar_articulo(numero) and f["unidad_id"] not in vistas:
                vistas.add(f["unidad_id"])
                salida.append(dict(f))
                if len(salida) >= limite:
                    break
        return salida

    def chunks_por_doc(self):
        return {r[0]: r[1] for r in self.conn.execute("SELECT doc_id, COUNT(*) FROM chunks GROUP BY doc_id")}


def normalizar_articulo(valor):
    m = re.search(r"(\d+)\s*([a-zA-Z](?![a-zA-Z]))?", str(valor or ""))
    return (m.group(1).lstrip("0") + (m.group(2) or "").lower()) if m else None


# ---------------------------------------------------------------- índice denso

class IndiceDenso:
    """FAISS FlatIP en CPU (exacto). En producción puede ir a GPU en fp16."""

    def __init__(self, ruta, en_gpu=False):
        import faiss
        self.indice = faiss.read_index(str(ruta))
        self.matriz = None
        if en_gpu:
            import torch
            vectores = self.indice.reconstruct_n(0, self.indice.ntotal)
            self.matriz = torch.from_numpy(vectores).to("cuda", dtype=torch.float16)
            self.indice = None

    @property
    def total(self):
        return self.matriz.shape[0] if self.matriz is not None else self.indice.ntotal

    def buscar(self, vector, k=100):
        if self.matriz is not None:
            import torch
            v = torch.from_numpy(vector.reshape(-1)).to("cuda", dtype=torch.float16)
            valores, indices = torch.topk((self.matriz @ v).float(), k)
            pares = zip(indices.tolist(), valores.tolist())
        else:
            valores, indices = self.indice.search(vector.reshape(1, -1).astype("float32"), k)
            pares = zip(indices[0].tolist(), valores[0].tolist())
        # En corpus_definitivo el id del chunk es la posición FAISS + 1.
        return [(int(i) + 1, float(s)) for i, s in pares if i >= 0]


# ---------------------------------------------------------------- fuentes

class Fuentes:
    """BM25, búsqueda densa y reranking, con caché opcional en memoria/disco.

    encoders: {nombre: objeto con codificar(textos, tarea)}
    indices:  {nombre: IndiceDenso}
    rerankers: {nombre: objeto con puntuar(consulta, textos)}
    cache: dict mutable (lo persiste el llamador) o None.
    """

    def __init__(self, lexico, encoders=None, indices=None, rerankers=None, cache=None):
        self.lexico = lexico
        self.encoders = encoders or {}
        self.indices = indices or {}
        self.rerankers = rerankers or {}
        self.cache = cache

    def _memo(self, clave, calcular):
        if self.cache is None:
            return calcular()
        if clave not in self.cache:
            self.cache[clave] = calcular()
        return self.cache[clave]

    def bm25(self, texto, k):
        return self._memo(json.dumps(["bm25", texto, k], ensure_ascii=False),
                          lambda: self.lexico.buscar(texto, k))

    def denso(self, encoder, texto, k):
        def calcular():
            if encoder not in self.encoders or encoder not in self.indices:
                raise RuntimeError(f"Falta cargar el encoder o el índice {encoder}")
            vector = self.encoders[encoder].codificar([texto], tarea="query")[0]
            return self.indices[encoder].buscar(vector, k)
        return self._memo(json.dumps(["denso", encoder, texto, k], ensure_ascii=False), calcular)

    def rerank(self, reranker, texto, ids):
        def calcular():
            if reranker not in self.rerankers:
                raise RuntimeError(f"Falta cargar el reranker {reranker}")
            filas = self.lexico.filas(ids)
            puntajes = self.rerankers[reranker].puntuar(texto, [f["texto_busqueda"] for f in filas])
            return sorted([(f["id"], float(s)) for f, s in zip(filas, puntajes)], key=lambda x: (-x[1], x[0]))
        return self._memo(json.dumps(["rerank", reranker, texto, list(ids)], ensure_ascii=False), calcular)


def expansion(entrada, anclas, n):
    """Nombres de las n normas ancla del área, solo para la consulta léxica."""
    if not n:
        return ""
    return " " + " ".join(nombre for _, nombre in anclas.get(entrada.get("area") or "", [])[:n])


def recuperar(entrada, R, fuentes, anclas=None):
    """Ranking final [(chunk_id, puntaje)] para una configuración R."""
    finales = []
    for q in consultas(entrada, R["consulta"]):
        listas = []
        if R.get("bm25", True):
            listas.append(fuentes.bm25(q + expansion(entrada, anclas or {}, R.get("expansion_area", 0)), R["k_bm25"]))
        for encoder in R.get("densos") or []:
            listas.append(fuentes.denso(encoder, q, R["k_denso"]))
        if not listas:
            raise ValueError("La configuración no tiene ni BM25 ni encoders")
        fusion = rrf(*listas, k=R["k_rrf"], constante=R["rrf_constante"]) if len(listas) > 1 else listas[0][:R["k_rrf"]]
        if R.get("reranker"):
            fusion = fuentes.rerank(R["reranker"], q, [c for c, _ in fusion[:R["top_rerank"]]])
        finales.append(fusion)
    if len(finales) == 1:
        return finales[0]
    if R.get("reranker"):
        mejor = {}
        for lista in finales:  # mismo reranker: se conserva el mejor puntaje de cada chunk
            for c, s in lista:
                mejor[c] = max(s, mejor.get(c, float("-inf")))
        return sorted(mejor.items(), key=lambda x: (-x[1], x[0]))[:R["top_rerank"]]
    return rrf(*finales, k=R["k_rrf"], constante=R["rrf_constante"])


# ---------------------------------------------------------------- identidades

def indice_identidades(evidencia, ruta_cache=None):
    """cuerpo normativo -> doc_ids. Tarda ~1 min con 14 mil documentos; se guarda en disco."""
    if ruta_cache and Path(ruta_cache).is_file():
        datos = json.loads(Path(ruta_cache).read_text(encoding="utf-8"))
        return {tuple(json.loads(k)): v for k, v in datos.items()}
    salida = defaultdict(list)
    for doc_id in evidencia.documentos:
        try:
            for cuerpo in evidencia.identidad(doc_id):
                salida[tuple(cuerpo)].append(doc_id)
        except FileNotFoundError:
            continue
    if ruta_cache:
        Path(ruta_cache).parent.mkdir(parents=True, exist_ok=True)
        Path(ruta_cache).write_text(json.dumps({json.dumps(list(k)): v for k, v in salida.items()},
                                               ensure_ascii=False), encoding="utf-8")
    return dict(salida)


# ---------------------------------------------------------------- pasajes

class Selector:
    """Convierte un ranking en la evidencia final (máximo 10 pasajes que cuenta el evaluador)."""

    def __init__(self, lexico, evidencia, identidades=None, anclas=None, chunks_por_doc=None):
        self.lexico = lexico
        self.ev = evidencia
        self.identidades = identidades or {}
        self.anclas = anclas or {}
        self._chunks_por_doc = chunks_por_doc

    def _unidad(self, fila, P):
        a, b = fila["inicio"], fila["fin"]
        if fila["unidad_fin"] - fila["unidad_inicio"] <= P["max_caracteres_unidad"]:
            a, b = fila["unidad_inicio"], fila["unidad_fin"]
        return a, b

    def _cabecera(self, doc_id, texto, P):
        citas = self.ev.citas
        propia = self.ev.identidad(doc_id)
        if not propia or propia <= citas.bodies(citas.extract(texto)):
            return None
        cabecera = self.ev.cabecera(doc_id)
        if cabecera is None and P["cabecera_normalizada"]:
            cabecera = self.ev.cabecera_normalizada(doc_id)
        return cabecera

    def router(self, entrada):
        """Artículos nombrados en la pregunta ("artículo 241 del Código Penal"): búsqueda exacta."""
        citas = self.ev.citas
        filas = []
        for cita in sorted(citas.extract(entrada["pregunta"]), key=str):
            if len(cita) < 4 or cita[3] is None:
                continue
            docs = self.identidades.get(tuple(cita[:3]), [])
            filas += self.lexico.articulo(self._principales(docs), cita[3])
        return filas

    def _principales(self, docs, n=2):
        """Entre documentos con la misma identidad, los de más chunks (el código completo)."""
        if self._chunks_por_doc is None:
            self._chunks_por_doc = self.lexico.chunks_por_doc()
        return sorted(docs, key=lambda d: -self._chunks_por_doc.get(d, 0))[:n]

    def ancla(self, entrada, n):
        """Mejor chunk (BM25) de las n normas más citadas del área de la pregunta."""
        filas = []
        for cuerpo, _ in self.anclas.get(entrada.get("area") or "", [])[:n]:
            docs = self._principales(self.identidades.get(tuple(cuerpo), []), 1)
            mejores = self.lexico.buscar_en_docs(texto_opciones(entrada), docs, 1)
            if mejores:
                filas += self.lexico.filas([mejores[0][0]])
        return filas

    def seleccionar(self, entrada, ranking, P):
        if P["modo"] == "e06":
            return self._e06(ranking, P)
        filas_ranking = {f["id"]: f for f in self.lexico.filas([c for c, _ in ranking])}
        candidatos = []
        if P.get("router_articulo"):
            candidatos += [(f, "router", None) for f in self.router(entrada)]
        candidatos += [(filas_ranking[c], "ranking", s) for c, s in ranking if c in filas_ranking]
        anclas = [(f, "ancla", None) for f in self.ancla(entrada, P.get("anclas_area", 0))]
        maximo = P["max_pasajes"]
        pasajes, vistas, con_cabecera = [], set(), set()
        por_doc = defaultdict(int)
        puntaje_base = ranking[0][1] if ranking else 0.0

        def agregar(fila, origen, puntaje):
            if fila["unidad_id"] in vistas:
                return "vista"
            doc_id = fila["doc_id"]
            if P.get("max_por_doc") and por_doc[doc_id] >= P["max_por_doc"] and origen == "ranking":
                return "tope_doc"
            a, b = self._unidad(fila, P)
            texto = self.ev.texto(doc_id)[a:b]
            if not texto.strip():
                return "vacia"
            cabecera = None if doc_id in con_cabecera else self._cabecera(doc_id, texto, P)
            necesarios = 1 + (cabecera is not None)
            if origen == "ancla":
                self._hacer_espacio(pasajes, necesarios, maximo, con_cabecera, por_doc, vistas)
            if len(pasajes) + necesarios > maximo:
                return "sin_cupo"
            vistas.add(fila["unidad_id"])
            puntaje = float(puntaje if puntaje is not None else puntaje_base)
            if cabecera is not None:
                pasajes.append(dict(cabecera, grupo_evidencia=doc_id, score=puntaje, origen=origen))
                con_cabecera.add(doc_id)
            pasajes.append({"doc_id": doc_id, "inicio": a, "fin": b, "texto": texto, "score": puntaje,
                            "articulo": fila["articulo"], "unidad_id": fila["unidad_id"],
                            "grupo_evidencia": doc_id, "tipo_evidencia": "unidad", "origen": origen})
            por_doc[doc_id] += 1
            return "ok"

        unidades = 0
        for fila, origen, puntaje in candidatos:
            if unidades >= P["max_unidades"] or len(pasajes) >= maximo:
                break
            unidades += agregar(fila, origen, puntaje) == "ok"
        # Anclas: solo si su norma no quedó ya respaldada; desplazan las últimas unidades del ranking.
        for fila, origen, _ in anclas:
            if self.ev.identidad(fila["doc_id"]) <= self.ev.respaldo(pasajes):
                continue
            agregar(fila, origen, None)
        self.ev.verificar(pasajes)
        return pasajes[:maximo]

    @staticmethod
    def _hacer_espacio(pasajes, necesarios, maximo, con_cabecera, por_doc, vistas):
        """Quita unidades del ranking desde el final (y su cabecera si queda huérfana)."""
        while len(pasajes) + necesarios > maximo:
            indice = next((i for i in range(len(pasajes) - 1, -1, -1)
                           if pasajes[i].get("origen") == "ranking" and pasajes[i]["tipo_evidencia"] == "unidad"), None)
            if indice is None:
                return
            quitado = pasajes.pop(indice)
            vistas.discard(quitado["unidad_id"])
            por_doc[quitado["doc_id"]] -= 1
            if por_doc[quitado["doc_id"]] == 0:
                for j in range(len(pasajes) - 1, -1, -1):
                    if pasajes[j]["doc_id"] == quitado["doc_id"] and pasajes[j]["tipo_evidencia"] in CABECERAS:
                        pasajes.pop(j)
                        con_cabecera.discard(quitado["doc_id"])

    def _e06(self, ranking, P):
        """Réplica de corpus_definitivo.passages (e06): cabecera de 800 caracteres, máximo 2."""
        citas = self.ev.citas
        seleccion, vistas = [], set()
        filas = {f["id"]: f for f in self.lexico.filas([c for c, _ in ranking])}
        for chunk_id, puntaje in ranking:
            fila = filas.get(chunk_id)
            if fila is None or fila["unidad_id"] in vistas:
                continue
            vistas.add(fila["unidad_id"])
            a, b = self._unidad(fila, P)
            seleccion.append({"doc_id": fila["doc_id"], "inicio": a, "fin": b,
                              "texto": self.ev.texto(fila["doc_id"])[a:b], "score": float(puntaje),
                              "articulo": fila["articulo"], "unidad_id": fila["unidad_id"],
                              "grupo_evidencia": fila["doc_id"], "tipo_evidencia": "unidad", "origen": "ranking"})
            if len(seleccion) >= P["max_pasajes"]:
                break
        extra, docs = [], set()
        for p in seleccion[:4]:
            if p["doc_id"] in docs:
                continue
            docs.add(p["doc_id"])
            inicio = self.ev.texto(p["doc_id"])[:800]
            if not citas.extract(inicio):
                continue
            if citas.bodies(citas.extract(inicio)) <= citas.bodies(citas.extract(p["texto"])):
                continue
            extra.append({"doc_id": p["doc_id"], "inicio": 0, "fin": len(inicio), "texto": inicio,
                          "score": p["score"], "articulo": None, "unidad_id": p["doc_id"] + "__cabecera_literal",
                          "grupo_evidencia": p["doc_id"], "tipo_evidencia": "cabecera_fuente", "origen": "ranking"})
            if len(extra) == 2:
                break
        if extra:
            seleccion = seleccion[:1] + extra + seleccion[1:P["max_pasajes"] - len(extra)]
        return seleccion


def pasajes_salida(pasajes):
    """Solo los campos del esquema oficial."""
    salida = []
    for p in pasajes[:10]:
        fila = {"doc_id": p["doc_id"]}
        if "inicio" in p:
            fila.update(inicio=int(p["inicio"]), fin=int(p["fin"]))
        fila.update(texto=p["texto"], score=float(p.get("score") or 0.0))
        salida.append(fila)
    return salida


def es_cabecera(pasaje):
    return pasaje.get("tipo_evidencia") in CABECERAS


**decoder**. Decoders de transformers (normal, multimodal, con razonamiento) y llama.cpp.

In [ ]:
%%modulo produccion.decoder
"""Decoder local: transformers (por defecto), servidor llama.cpp o simulado.

Todos exponen la interfaz que usan politica.ajustar/politica.generar:
contar(conversacion) y aplicar_plantilla(conversacion). Temperatura 0 siempre.

Variantes que maneja:
  * causal normal (Qwen2.5, Qwen3-Instruct-2507, Salamandra, Llama).
  * multimodal (Qwen3.5-4B): AutoProcessor + AutoModelForMultimodalLM, como en e06.
  * con pensamiento (Qwen3-*-Thinking): se descarta todo lo anterior a </think>.
  * híbridos Qwen3 (Qwen3-8B): enable_thinking=False salvo que se pida pensar.
"""
import json
import re
import time
import urllib.request

from produccion import politica
from produccion.modelos import es_simulado, parametros_modelo

LIMITE_PARAMETROS = 8_000_000_000


def extraer_json(texto):
    """Quita el razonamiento, las cercas ```json y el texto previo al primer '{'."""
    texto = (texto or "").strip()
    if "</think>" in texto:
        texto = texto.rsplit("</think>", 1)[1].strip()
    texto = re.sub(r"^```(?:json)?\s*", "", texto, flags=re.I)
    texto = re.sub(r"\s*```\s*$", "", texto)
    inicio = texto.find("{")
    if inicio > 0:
        texto = texto[inicio:]
    return politica.reparar_json(texto)


class Plantilla:
    def __init__(self, tokenizer, modelo, pensar=False):
        self.tokenizer = tokenizer
        self.hibrido_qwen3 = bool(re.search(r"Qwen3-\d+B$|Qwen3-\d+B-", modelo)) and "2507" not in modelo
        self.pensar = pensar

    def aplicar_plantilla(self, conversacion):
        kwargs = {"tokenize": False, "add_generation_prompt": True}
        if self.hibrido_qwen3 or "Qwen3.5" in getattr(self, "nombre", ""):
            kwargs["enable_thinking"] = self.pensar
        return self.tokenizer.apply_chat_template(conversacion, **kwargs)

    def contar(self, conversacion):
        return len(self.tokenizer(self.aplicar_plantilla(conversacion), add_special_tokens=False)["input_ids"])


class DecoderHF(Plantilla):
    def __init__(self, especificacion, revision=None):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor
        self.torch = torch
        self.esp = especificacion
        self.nombre = especificacion["modelo"]
        origen = especificacion.get("ruta_local") or self.nombre
        rev = None if especificacion.get("ruta_local") else revision
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        self.multimodal = especificacion.get("clase") == "multimodal"
        if self.multimodal:
            from transformers import AutoModelForMultimodalLM, AutoProcessor
            self.procesador = AutoProcessor.from_pretrained(origen, revision=rev)
            tokenizer = self.procesador.tokenizer
            self.modelo = AutoModelForMultimodalLM.from_pretrained(
                origen, revision=rev, dtype=dtype, device_map="cuda").eval()
        else:
            tokenizer = AutoTokenizer.from_pretrained(origen, revision=rev)
            self.modelo = AutoModelForCausalLM.from_pretrained(
                origen, revision=rev, dtype=dtype, device_map="cuda").eval()
        super().__init__(tokenizer, self.nombre, especificacion.get("pensar", False))
        if especificacion.get("adaptador_lora"):
            from peft import PeftModel
            self.modelo = PeftModel.from_pretrained(self.modelo, especificacion["adaptador_lora"]).eval()
        self.parametros = parametros_modelo(self.modelo)
        self._letras = {}

        class PenalizacionGenerados(LogitsProcessor):
            """repeat_penalty al estilo llama.cpp: solo sobre los últimos N tokens GENERADOS.
            La repetition_penalty de transformers castiga también el prompt, y eso penaliza
            copiar los nombres de las normas que vienen en los pasajes."""

            def __init__(self, largo_prompt, penalizacion, ventana):
                self.largo_prompt, self.penalizacion, self.ventana = largo_prompt, penalizacion, ventana

            def __call__(self, input_ids, scores):
                generados = input_ids[:, self.largo_prompt:][:, -self.ventana:]
                if generados.shape[1] == 0 or self.penalizacion == 1.0:
                    return scores
                valores = torch.gather(scores, 1, generados)
                valores = torch.where(valores < 0, valores * self.penalizacion, valores / self.penalizacion)
                return scores.scatter(1, generados, valores)

        self._Penalizacion = PenalizacionGenerados

    def aplicar_plantilla(self, conversacion):
        if self.multimodal:
            conversacion = [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]}
                            for m in conversacion]
            kwargs = {"tokenize": False, "add_generation_prompt": True, "enable_thinking": self.pensar}
            return self.procesador.apply_chat_template(conversacion, **kwargs)
        return super().aplicar_plantilla(conversacion)

    def _ids(self, prompt):
        return self.tokenizer(prompt, add_special_tokens=False, return_tensors="pt")["input_ids"].to("cuda")

    def _eos(self):
        eos = self.modelo.generation_config.eos_token_id
        return set(eos if isinstance(eos, list) else [eos])

    def completar(self, prompt, max_tokens, max_segundos=None, repeat_penalty=1.1, repeat_last_n=256):
        from transformers import LogitsProcessorList
        torch = self.torch
        ids = self._ids(prompt)
        procesadores = LogitsProcessorList([self._Penalizacion(ids.shape[1], repeat_penalty, repeat_last_n)])
        with torch.inference_mode():
            salida = self.modelo.generate(
                input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_tokens,
                do_sample=False, temperature=None, top_p=None, top_k=None, repetition_penalty=1.0,
                logits_processor=procesadores, max_time=max_segundos,
                pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id)
        nuevos = salida[0, ids.shape[1]:]
        texto = self.tokenizer.decode(nuevos, skip_special_tokens=True)
        fin = "eos" if len(nuevos) and int(nuevos[-1]) in self._eos() else "limite"
        return texto, len(nuevos), fin

    def elegir_letra(self, conversacion, letras):
        """Letra por logits del siguiente token, cuando el JSON no trae una válida."""
        torch = self.torch
        prompt = self.aplicar_plantilla(conversacion)
        if self.pensar:
            prompt += "<think>\n\n</think>\n\n"
        ids = self._ids(prompt + "La opción correcta es la ")
        with torch.inference_mode():
            logits = self.modelo(input_ids=ids).logits[0, -1].float()
        puntajes = {}
        for letra in letras:
            if letra not in self._letras:
                self._letras[letra] = sorted({self.tokenizer.encode(v, add_special_tokens=False)[0]
                                              for v in (letra, " " + letra)})
            puntajes[letra] = max(float(logits[i]) for i in self._letras[letra])
        return max(letras, key=lambda l: puntajes[l]), puntajes

    def cerrar(self):
        from produccion.modelos import liberar
        modelo, self.modelo = self.modelo, None
        liberar(modelo)


class DecoderLlama(Plantilla):
    """Servidor llama-server con gramática json_schema (lo que usó el notebook 04)."""

    def __init__(self, especificacion, revision=None, endpoint="http://127.0.0.1:8080"):
        from transformers import AutoTokenizer
        self.esp = especificacion
        self.nombre = especificacion["modelo"]
        self.endpoint = especificacion.get("endpoint", endpoint).rstrip("/")
        super().__init__(AutoTokenizer.from_pretrained(self.nombre, revision=revision), self.nombre,
                         especificacion.get("pensar", False))
        self.parametros = None

    def pedir(self, ruta, cuerpo, timeout=300):
        peticion = urllib.request.Request(self.endpoint + ruta, data=json.dumps(cuerpo).encode("utf-8"),
                                          headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(peticion, timeout=timeout) as r:
            return json.loads(r.read().decode("utf-8")), r.status

    def elegir_letra(self, conversacion, letras):
        prompt = self.aplicar_plantilla(conversacion) + "La opción correcta es la "
        respuesta, _ = self.pedir("/completion", {
            "prompt": prompt, "n_predict": 1, "temperature": 0, "seed": 0, "cache_prompt": False,
            "grammar": "root ::= " + " | ".join(f'"{l}"' for l in letras)})
        letra = (respuesta.get("content") or "").strip()
        return (letra if letra in letras else letras[0]), {}

    def cerrar(self):
        pass


def cargar(especificacion, revision=None):
    """especificacion: dict del decoder (modelo, clase, pensar, backend, max_tokens...)."""
    if es_simulado(especificacion["modelo"]):
        from experimentos.e07.simulado import DecoderSimulado
        return DecoderSimulado(especificacion)
    backend = especificacion.get("backend", "hf")
    if backend == "hf":
        return DecoderHF(especificacion, revision)
    if backend == "llamacpp":
        return DecoderLlama(especificacion, revision)
    raise ValueError("backend debe ser 'hf' o 'llamacpp'")


def generar(decoder, entrada, pasajes, evidencia, esp):
    """Una llamada al decoder. Devuelve (crudo, pasajes_usados, conversacion, registro).

    esp: dict del decoder con contexto, max_tokens por formato, max_segundos,
    repeat_penalty, repeat_last_n, longitudes y max_caracteres_pasaje.
    """
    max_tokens = esp["max_tokens"][entrada["formato"]]
    inicio = time.perf_counter()
    usados, conversacion, tokens, omitidas = politica.ajustar(
        decoder, entrada, pasajes, evidencia, esp["contexto"], max_tokens, esp["max_caracteres_pasaje"])
    registro = {"tokens_prompt": tokens, "omitidas": omitidas}
    if isinstance(decoder, DecoderLlama):
        config_llama = {"contexto": esp["contexto"], "max_tokens": max_tokens,
                        "max_caracteres_pasaje": esp["max_caracteres_pasaje"],
                        "repeat_penalty": esp["repeat_penalty"], "repeat_last_n": esp["repeat_last_n"],
                        "longitudes": esp.get("longitudes", True)}
        crudo, usados, reg = politica.generar(decoder, entrada, pasajes, evidencia, config_llama)
        registro.update({k: v for k, v in reg.items() if k != "peticion"})
    else:
        prompt = decoder.aplicar_plantilla(conversacion)
        texto, n, fin = decoder.completar(prompt, max_tokens, esp.get("max_segundos"),
                                          esp.get("repeat_penalty", 1.1), esp.get("repeat_last_n", 256))
        crudo, reparado = extraer_json(texto)
        registro.update(contenido=texto, reparado=reparado, json_valido=crudo is not None,
                        finish_reason=fin, tokens_salida=n)
    if crudo is not None and not isinstance(crudo, dict):
        crudo = None
        registro["json_valido"] = False
    registro["segundos"] = round(time.perf_counter() - inicio, 3)
    return crudo, usados, conversacion, registro


**etapas**. Checkpoints: una carpeta por nodo con `completo.json` y avance por ítem.

In [ ]:
%%modulo experimentos.e07.etapas
"""Checkpoints por carpeta.

Cada unidad de trabajo es un nodo: etapa + nombre + entradas. La clave del nodo es
el hash de sus entradas, así que cambiar un parámetro crea otra carpeta y nunca
pisa un resultado anterior.

    <SALIDA>/E05_pasajes/R03__P_v04__3f9a1c02de/
        completo.json      marca de fin: entradas, resumen, segundos
        pasajes.jsonl      avance por ítem (se reanuda desde aquí)

Para volver a ejecutar desde un punto en otra carpeta: se crea una SALIDA nueva y
se pasan las anteriores en REUTILIZAR. Los nodos completos que ya existen allí se
leen sin recalcular; los nuevos o cambiados se escriben en la SALIDA nueva.
DESDE_ETAPA obliga a recalcular esa etapa y las siguientes.
"""
import hashlib
import json
import time
import traceback
from datetime import datetime
from pathlib import Path

ETAPAS = ["E00_entorno", "E01_lexico", "E02_denso", "E03_busquedas", "E04_rankings", "E05_pasajes",
          "E06_generacion", "E07_respuestas", "E08_juez", "E09_finetune", "E10_producto"]


def clave(valor, largo=10):
    texto = json.dumps(valor, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()[:largo]


def escribir_json(ruta, valor):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    temporal = ruta.with_name(ruta.name + ".tmp")
    temporal.write_text(json.dumps(valor, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    temporal.replace(ruta)


def leer_json(ruta, defecto=None):
    ruta = Path(ruta)
    return json.loads(ruta.read_text(encoding="utf-8")) if ruta.is_file() else defecto


def leer_jsonl(ruta):
    ruta = Path(ruta)
    if not ruta.is_file():
        return []
    filas = []
    with ruta.open(encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()
            if not linea:
                continue
            try:
                filas.append(json.loads(linea))
            except json.JSONDecodeError:
                break  # última línea cortada por una interrupción: se rehace ese ítem
    return filas


def escribir_jsonl(ruta, filas):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    temporal = ruta.with_name(ruta.name + ".tmp")
    temporal.write_text("".join(json.dumps(f, ensure_ascii=False, default=str) + "\n" for f in filas),
                        encoding="utf-8")
    temporal.replace(ruta)


class Bitacora:
    """JSONL de avance por ítem: se agrega y se hace flush en cada fila."""

    def __init__(self, ruta, campo="id"):
        self.ruta = Path(ruta)
        self.campo = campo
        filas = leer_jsonl(self.ruta)
        self.filas = {f[campo]: f for f in filas}
        if filas and len(filas) != sum(1 for l in self.ruta.open(encoding="utf-8") if l.strip()):
            escribir_jsonl(self.ruta, filas)  # quita la línea corrupta

    def hecho(self, valor):
        return valor in self.filas

    def agregar(self, fila):
        self.ruta.parent.mkdir(parents=True, exist_ok=True)
        with self.ruta.open("a", encoding="utf-8") as f:
            f.write(json.dumps(fila, ensure_ascii=False, default=str) + "\n")
            f.flush()
        self.filas[fila[self.campo]] = fila


class Nodo:
    def __init__(self, espacio, etapa, nombre, entradas):
        self.espacio = espacio
        self.etapa = etapa
        self.nombre = nombre
        self.entradas = entradas
        self.clave = clave(entradas)
        self.carpeta_propia = espacio.salida / etapa / f"{nombre}__{self.clave}"

    @property
    def id(self):
        return f"{self.etapa}/{self.nombre}__{self.clave}"

    def buscar(self):
        """Carpeta completa de este nodo en la SALIDA actual o en las reutilizables."""
        raices = [self.espacio.salida]
        if not self.espacio.recalcular(self.etapa):
            raices += self.espacio.reutilizar
        for raiz in raices:
            carpeta = raiz / self.etapa / f"{self.nombre}__{self.clave}"
            if (carpeta / "completo.json").is_file():
                if raiz == self.espacio.salida and self.espacio.recalcular(self.etapa) \
                        and not self.espacio.ya_recalculado(self):
                    return None
                return carpeta
        return None

    def listo(self):
        return self.buscar() is not None

    @property
    def carpeta(self):
        """Carpeta para leer (si está completa en cualquier raíz) o para escribir."""
        encontrada = self.buscar()
        if encontrada:
            return encontrada
        self.carpeta_propia.mkdir(parents=True, exist_ok=True)
        return self.carpeta_propia

    def resumen(self):
        encontrada = self.buscar()
        return leer_json(encontrada / "completo.json", {}).get("resumen") if encontrada else None

    def terminar(self, resumen, segundos):
        escribir_json(self.carpeta_propia / "completo.json", {
            "nodo": self.id, "entradas": self.entradas, "resumen": resumen,
            "segundos": round(segundos, 1), "fecha": datetime.now().isoformat(timespec="seconds")})
        self.espacio.marcar_recalculado(self)


class Espacio:
    def __init__(self, salida, reutilizar=(), desde_etapa=None, detener_en_error=False):
        self.salida = Path(salida).resolve()
        self.salida.mkdir(parents=True, exist_ok=True)
        self.reutilizar = [Path(r).resolve() for r in reutilizar if Path(r).resolve() != self.salida]
        self.desde = ETAPAS.index(desde_etapa) if desde_etapa else None
        self.detener_en_error = detener_en_error
        self.ruta_registro = self.salida / "registro.jsonl"
        self._recalculados = set()

    def recalcular(self, etapa):
        return self.desde is not None and etapa in ETAPAS and ETAPAS.index(etapa) >= self.desde

    def ya_recalculado(self, nodo):
        return nodo.id in self._recalculados

    def marcar_recalculado(self, nodo):
        self._recalculados.add(nodo.id)

    def nodo(self, etapa, nombre, entradas):
        return Nodo(self, etapa, nombre, entradas)

    def registrar(self, **campos):
        campos = {"fecha": datetime.now().isoformat(timespec="seconds"), **campos}
        with self.ruta_registro.open("a", encoding="utf-8") as f:
            f.write(json.dumps(campos, ensure_ascii=False, default=str) + "\n")

    def ejecutar(self, nodo, funcion, forzar=False):
        """Corre funcion(carpeta) si el nodo no está listo. Un error se registra y no detiene la grilla."""
        if nodo.listo() and not forzar:
            self.registrar(nodo=nodo.id, estado="reutilizado", carpeta=str(nodo.buscar()))
            return nodo.resumen()
        carpeta = nodo.carpeta_propia
        if (carpeta / "completo.json").is_file():
            # Recalcular (DESDE_ETAPA o forzar): lo anterior se archiva, no se borra ni se mezcla.
            archivo = carpeta.with_name(carpeta.name + "__anterior_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
            carpeta.rename(archivo)
        carpeta.mkdir(parents=True, exist_ok=True)
        escribir_json(carpeta / "entradas.json", nodo.entradas)
        self.registrar(nodo=nodo.id, estado="inicio")
        inicio = time.perf_counter()
        try:
            resumen = funcion(carpeta)
        except KeyboardInterrupt:
            self.registrar(nodo=nodo.id, estado="interrumpido", segundos=round(time.perf_counter() - inicio, 1))
            raise
        except Exception as error:
            traza = traceback.format_exc()
            (carpeta / "error.txt").write_text(traza, encoding="utf-8")
            self.registrar(nodo=nodo.id, estado="error", error=f"{type(error).__name__}: {error}",
                           segundos=round(time.perf_counter() - inicio, 1))
            print(f"ERROR en {nodo.id}: {type(error).__name__}: {error}")
            if self.detener_en_error:
                raise
            return None
        segundos = time.perf_counter() - inicio
        nodo.terminar(resumen, segundos)
        self.registrar(nodo=nodo.id, estado="fin", segundos=round(segundos, 1))
        return resumen

    def estado(self):
        """Último estado de cada nodo según registro.jsonl."""
        ultimo = {}
        for fila in leer_jsonl(self.ruta_registro):
            ultimo[fila["nodo"]] = fila
        return list(ultimo.values())


**grilla**. Expansión de la grilla en configuraciones completas y orden de corridas.

In [ ]:
%%modulo experimentos.e07.grilla
"""Expansión de la grilla declarativa (grilla.json) en configuraciones completas.

"Probar todo sin mirar resultados": la lista de corridas se fija antes de ejecutar.
Recuperación y pasajes corren completos (R x P, sin decoder, baratos). La generación
sigue un diseño fijado de antemano: la base, cada factor cambiado de a uno (OFAT)
y un bloque factorial; con "completo": true se agregan al final todas las
combinaciones restantes. Ningún resultado intermedio elimina configuraciones.
"""
import copy
import json
from itertools import product
from pathlib import Path

from produccion.recuperador import P_BASE, R_BASE, completar

META = {"nota", "costo", "requiere_finetune", "requiere_confirmacion", "solo_medicion"}

SIMULACION = {
    "BAAI/bge-m3": "simulado/bge-m3", "intfloat/multilingual-e5-large": "simulado/e5",
    "Qwen/Qwen3-Embedding-0.6B": "simulado/qwen-emb-0.6b", "Qwen/Qwen3-Embedding-4B": "simulado/qwen-emb-4b",
    "BAAI/bge-reranker-v2-m3": "simulado/bge-reranker", "Qwen/Qwen3-Reranker-0.6B": "simulado/qwen-reranker",
}


def _sin_meta(d):
    return {k: v for k, v in d.items() if k not in META}


class Grilla:
    def __init__(self, fuente, simulado=False):
        self.datos = json.loads(json.dumps(fuente)) if isinstance(fuente, dict) else json.loads(Path(fuente).read_text(encoding="utf-8"))
        self.simulado = simulado
        self.base = self.datos["base"]

    # ------------------------------------------------------------ configuraciones completas
    def _modelo(self, nombre):
        if not self.simulado or nombre is None or str(nombre).startswith("ft:"):
            return nombre
        return SIMULACION.get(nombre, "simulado/" + str(nombre).split("/")[-1].lower())

    def R(self, nombre):
        cambios = self.datos["recuperacion"][nombre]
        r = completar(R_BASE, _sin_meta(cambios))
        r["densos"] = [self._modelo(e) for e in r.get("densos") or []]
        r["reranker"] = self._modelo(r.get("reranker"))
        return r

    def P(self, nombre):
        return completar(P_BASE, _sin_meta(self.datos["pasajes"][nombre]))

    def D(self, nombre):
        esp = copy.deepcopy(self.datos["decoder_comun"])
        cambios = self.datos["decoders"][nombre]
        for k, v in cambios.items():
            if k == "max_tokens":
                esp["max_tokens"] = {**esp["max_tokens"], **v}
            elif k != "nota":
                esp[k] = v
        if self.simulado:
            esp["modelo"] = "simulado/" + nombre
        return esp

    def politica(self, nombre):
        return _sin_meta(self.datos["politicas"][nombre])

    def meta(self, tipo, nombre):
        return {k: v for k, v in self.datos[tipo][nombre].items() if k in META}

    # ------------------------------------------------------------ listas
    @property
    def nombres_R(self):
        return list(self.datos["recuperacion"])

    @property
    def nombres_P(self):
        return list(self.datos["pasajes"])

    @property
    def nombres_D(self):
        return list(self.datos["decoders"])

    @property
    def nombres_politica(self):
        return list(self.datos["politicas"])

    def encoders(self):
        vistos = []
        for r in self.nombres_R:
            for e in self.R(r)["densos"]:
                if e not in vistos:
                    vistos.append(e)
        return vistos

    def rerankers(self):
        vistos = []
        for r in self.nombres_R:
            nombre = self.R(r)["reranker"]
            if nombre and nombre not in vistos:
                vistos.append(nombre)
        return vistos

    def corridas_generacion(self):
        """Lista ordenada y sin repetidos de (R, P, D)."""
        b = self.base
        g = self.datos["generacion"]
        orden = []

        def agregar(r, p, d):
            if (r, p, d) not in orden:
                orden.append((r, p, d))

        for bloque in g["diseno"]:
            if bloque == "base":
                agregar(b["R"], b["P"], b["D"])
            elif bloque == "ofat_R":
                for r in self.nombres_R:
                    agregar(r, b["P"], b["D"])
            elif bloque == "ofat_P":
                for p in self.nombres_P:
                    agregar(b["R"], p, b["D"])
            elif bloque == "ofat_D":
                for d in self.nombres_D:
                    agregar(b["R"], b["P"], d)
            elif bloque == "factorial":
                f = g["factorial"]
                for r, p, d in product(f["R"], f["P"], f["D"]):
                    agregar(r, p, d)
        if g.get("completo"):
            for r, p, d in product(self.nombres_R, self.nombres_P, self.nombres_D):
                agregar(r, p, d)
        return orden

    def resumen(self):
        corridas = self.corridas_generacion()
        return {"recuperacion": len(self.nombres_R), "pasajes": len(self.nombres_P),
                "combinaciones_R_x_P": len(self.nombres_R) * len(self.nombres_P),
                "decoders": len(self.nombres_D), "corridas_generacion": len(corridas),
                "politicas": len(self.nombres_politica),
                "respuestas_puntuadas": len(corridas) * len(self.nombres_politica)}


**indices**. Segmentación por artículos desde el texto crudo, BM25 e índices densos reanudables.

In [ ]:
%%modulo experimentos.e07.indices
"""Construcción de índices (E01 léxico, E02 densos) con reanudación y reutilización del e06.

E01 usa la segmentación del repo (legalrag.preprocessing.ingesta vía corpus_definitivo.
build_lexical) para que los chunks sean idénticos a los del e06. E02 reproduce
build_dense: bloques de 1024 chunks en orden de id, checkpoint cada 20 bloques.
"""
import json
import re
import sqlite3
import sys
import time
from contextlib import closing
from pathlib import Path

from experimentos.e07.etapas import escribir_json, leer_json


def resolver(carpeta, archivo):
    """Ruta real de un artefacto: dentro del nodo o apuntada por enlace.json (e06 importado)."""
    carpeta = Path(carpeta)
    enlace = leer_json(carpeta / "enlace.json")
    if enlace and archivo in enlace:
        return Path(enlace[archivo])
    return carpeta / archivo


ARTICULO = re.compile(r"^[ \t]*(?:ART[IÍ]CULO|Art[íi]culo|ARTICULO|Art\.)\s+(\d+[A-Za-z]?(?:\s*bis)?|[ÚU]NICO|PRIMERO)\b[^\n]{0,3}",
                      re.M)


def unidades(texto, minimo_articulos=2):
    """Unidades literales: artículos si el documento tiene al menos 2; si no, el documento completo."""
    marcas = list(ARTICULO.finditer(texto))
    if len(marcas) < minimo_articulos:
        return [(0, len(texto), None)]
    salida = []
    if marcas[0].start() > 0:
        salida.append((0, marcas[0].start(), None))  # encabezado y considerandos
    for i, m in enumerate(marcas):
        fin = marcas[i + 1].start() if i + 1 < len(marcas) else len(texto)
        salida.append((m.start(), fin, re.sub(r"\s+", "", m.group(1))))
    return salida


def construir_lexico(manifiesto, carpeta_textos, carpeta, ventana=1500, solapamiento=200):
    """chunks.sqlite + FTS5 desde el texto crudo, con el mismo esquema del e06.

    Cada chunk es un corte exacto del archivo (offsets reconstruibles). La cabecera
    (título | artículo) solo entra al texto de búsqueda, nunca al pasaje literal.
    """
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    ruta = carpeta / "chunks.sqlite"
    if ruta.exists():
        ruta.unlink()
    conn = sqlite3.connect(ruta)
    conn.executescript("""
    CREATE TABLE chunks (id INTEGER PRIMARY KEY, doc_id TEXT NOT NULL, titulo TEXT NOT NULL, tipo TEXT NOT NULL,
      articulo TEXT, seccion TEXT, unidad_id TEXT NOT NULL, unidad_inicio INTEGER NOT NULL, unidad_fin INTEGER NOT NULL,
      inicio INTEGER NOT NULL, fin INTEGER NOT NULL, texto TEXT NOT NULL, texto_busqueda TEXT NOT NULL, avisos TEXT NOT NULL);
    CREATE VIRTUAL TABLE fts USING fts5(texto_busqueda, content='chunks', content_rowid='id',
      tokenize='unicode61 remove_diacritics 2');""")
    registros = json.loads(Path(manifiesto).read_text(encoding="utf-8"))
    total, faltan, inicio_t = 0, [], time.perf_counter()
    for n, doc in enumerate(registros, 1):
        archivo = Path(carpeta_textos) / doc["texto_archivo"] if doc.get("texto_archivo") else Path(carpeta_textos) / f"{doc['doc_id']}.txt"
        if not archivo.is_file():
            faltan.append(doc["doc_id"])
            continue
        texto = archivo.read_text(encoding="utf-8")
        for k, (a, b, articulo) in enumerate(unidades(texto)):
            paso = ventana - solapamiento
            for inicio in range(a, b, paso):
                fin = min(b, inicio + ventana)
                trozo = texto[inicio:fin]
                if trozo.strip():
                    cabecera = " | ".join(x for x in [doc.get("titulo") or doc["doc_id"],
                                                      f"Artículo {articulo}" if articulo else None] if x)
                    fila = (doc["doc_id"], doc.get("titulo") or doc["doc_id"], doc.get("tipo") or "", articulo, None,
                            f"{doc['doc_id']}__u{k}", a, b, inicio, fin, trozo, cabecera + "\n" + trozo, "[]")
                    cur = conn.execute("INSERT INTO chunks(doc_id,titulo,tipo,articulo,seccion,unidad_id,unidad_inicio,"
                                       "unidad_fin,inicio,fin,texto,texto_busqueda,avisos) VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)", fila)
                    conn.execute("INSERT INTO fts(rowid,texto_busqueda) VALUES(?,?)", (cur.lastrowid, fila[-2]))
                    total += 1
                if fin >= b:
                    break
        if n % 200 == 0:
            conn.commit()
            print(f"Segmentados {n}/{len(registros)} documentos, {total} chunks", end="\r")
    conn.commit()
    conn.execute("CREATE INDEX IF NOT EXISTS idx_doc ON chunks(doc_id)")
    conn.commit()
    conn.close()
    if faltan:
        (carpeta / "documentos_sin_texto.json").write_text(json.dumps(faltan), encoding="utf-8")
    return {"documentos": len(registros) - len(faltan), "sin_texto": len(faltan), "chunks": total,
            "ventana": ventana, "solapamiento": solapamiento, "segundos": round(time.perf_counter() - inicio_t, 1)}


def importar_lexico_e06(runs_e06, carpeta, snapshot_id):
    """Apunta el nodo E01 a un chunks.sqlite del e06 ya construido (no copia 5 GB)."""
    runs_e06 = Path(runs_e06)
    meta = leer_json(runs_e06 / "chunks_complete.json")
    if not meta or meta.get("snapshot_id") != snapshot_id or meta.get("piloto"):
        return None
    escribir_json(Path(carpeta) / "enlace.json", {"chunks.sqlite": str(runs_e06 / "chunks.sqlite")})
    return dict(meta, importado_de=str(runs_e06))


def total_chunks(ruta_sqlite):
    with closing(sqlite3.connect(ruta_sqlite)) as conn:
        return conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]


def importar_denso_e06(runs_e06, encoder, carpeta, total):
    destino = Path(runs_e06) / "indices" / encoder.replace("/", "__")
    meta = leer_json(destino / "complete.json")
    if not meta or meta.get("chunks") != total or not (destino / "index.faiss").is_file():
        return None
    escribir_json(Path(carpeta) / "enlace.json", {"index.faiss": str(destino / "index.faiss")})
    return dict(meta, importado_de=str(destino))


def construir_denso(encoder, ruta_sqlite, carpeta, bloque=1024, cada=20):
    """Índice FlatIP reanudable. Los vectores quedan en el orden de id de chunks.sqlite."""
    import faiss
    import numpy as np
    carpeta = Path(carpeta)
    total = total_chunks(ruta_sqlite)
    indice, ultimo = None, 0
    if (carpeta / "checkpoint.faiss").exists():
        indice = faiss.read_index(str(carpeta / "checkpoint.faiss"))
        ultimo = indice.ntotal
        print(f"Reanudando {encoder.nombre} desde {ultimo}/{total}")
    inicio = time.perf_counter()
    with closing(sqlite3.connect(ruta_sqlite)) as conn:
        conn.row_factory = sqlite3.Row
        bloques = 0
        while ultimo < total:
            filas = conn.execute("SELECT id, texto_busqueda FROM chunks WHERE id > ? ORDER BY id LIMIT ?",
                                 (ultimo, bloque)).fetchall()
            if not filas or filas[0]["id"] != ultimo + 1:
                raise ValueError("Los ids de chunks.sqlite no son consecutivos")
            vectores = encoder.codificar([f["texto_busqueda"] for f in filas], tarea="passage")
            if not np.isfinite(vectores).all():
                raise ValueError("Embeddings no finitos")
            if indice is None:
                indice = faiss.IndexFlatIP(vectores.shape[1])
            indice.add(vectores)
            ultimo = indice.ntotal
            bloques += 1
            if bloques % cada == 0 or ultimo == total:
                temporal = carpeta / "checkpoint.tmp"
                faiss.write_index(indice, str(temporal))
                temporal.replace(carpeta / "checkpoint.faiss")
                velocidad = (ultimo) / max(1e-6, time.perf_counter() - inicio)
                print(f"{encoder.nombre}: {ultimo}/{total} vectores ({velocidad:.0f}/s en esta sesión)")
    faiss.write_index(indice, str(carpeta / "index.faiss"))
    (carpeta / "checkpoint.faiss").unlink(missing_ok=True)
    return {"encoder": encoder.nombre, "chunks": total, "dimension": indice.d,
            "segundos_sesion": round(time.perf_counter() - inicio, 1)}


**rubrica**. Rúbrica oficial por ítem, bootstrap y métricas sin decoder.

In [ ]:
%%modulo experimentos.e07.rubrica
"""Puntaje según la rúbrica oficial, por ítem y agregado, con intervalos bootstrap.

El agregado se calcula con las funciones de data/oficial/scripts/evaluate.py (las
mismas del jurado). La descomposición por ítem reproduce esas fórmulas para poder
remuestrear; `verificar_igualdad` comprueba que ambas den lo mismo.
"""
import importlib.util
import random
import sys
from pathlib import Path

PUNTOS = {"cerradas": 20.0, "ragas": 30.0, "citas": 20.0, "abstencion": 10.0}
W_SIN_RESPALDO = 0.5
W_ABSTENCION = 0.5


def cargar_evaluador(ruta_scripts):
    ruta_scripts = Path(ruta_scripts)
    if str(ruta_scripts) not in sys.path:
        sys.path.insert(0, str(ruta_scripts))
    spec = importlib.util.spec_from_file_location("evaluador_oficial", ruta_scripts / "evaluate.py")
    modulo = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(modulo)
    return modulo


class Rubrica:
    def __init__(self, ruta_scripts, ruta_muestra):
        self.ev = cargar_evaluador(ruta_scripts)
        self.citas = self.ev.citations
        filas = self.ev.read_jsonl(Path(ruta_muestra))
        self.filas = {r["id"]: r for r in filas}
        self.key = {r["id"]: {"respuesta_correcta": r.get("respuesta_correcta"),
                              "respuesta_esperada": r.get("respuesta_esperada"),
                              "legal_basis": r.get("legal_basis"), "pregunta": r["pregunta"],
                              "formato": r["formato"]} for r in filas}
        self.cerradas = [q for q, k in self.key.items()
                         if k["formato"] == "multiple_choice" and q not in self.ev.FLAWED_IDS]
        self.juzgadas = [q for q, k in self.key.items() if k["formato"] != "multiple_choice"]

    # ------------------------------------------------------------ oficial
    def oficial(self, envios):
        subs = {s["id"]: s for s in envios if isinstance(s.get("id"), int)}
        problemas = self.ev.validate(envios, set(self.key))
        return {"validacion": {"errores": len(problemas), "detalle": problemas[:20]},
                "cerradas": self.ev.score_closed(subs, self.key, self.cerradas),
                "citas": self.ev.score_citations(subs, self.key),
                "abstencion": self.ev.score_abstention(subs, self.key, set(self.cerradas))}

    # ------------------------------------------------------------ por ítem
    def por_item(self, envios):
        subs = {s["id"]: s for s in envios if isinstance(s.get("id"), int)}
        cerradas = set(self.cerradas)
        filas = []
        for qid, k in self.key.items():
            s = subs.get(qid)
            abst = bool(s and s.get("abstencion"))
            ref = self.citas.extract(k.get("legal_basis") or "")
            fila = {"id": qid, "formato": k["formato"], "area": self.filas[qid].get("area"),
                    "complejidad": self.filas[qid].get("complejidad"), "abstencion": abst,
                    "respondida": s is not None}
            # cerradas
            fila["cerrada_puntuable"] = qid in cerradas
            fila["cerrada_acierto"] = bool(qid in cerradas and s and not abst
                                           and s.get("respuesta_correcta") == k["respuesta_correcta"])
            # citas
            fila["citas_evaluable"] = bool(ref)
            for campo in ("n_ref", "n_citadas", "aciertos", "aciertos_respaldados", "aciertos_sin_respaldo",
                          "incorrectas", "citas_sin_respaldo"):
                fila[campo] = 0
            if ref:
                if not s or abst:
                    fila["n_ref"] = len(self.citas.bodies(ref))
                else:
                    r = self.citas.score(self.ev.answer_text(s), k.get("legal_basis") or "",
                                         self.ev.citas_respaldadas(s))
                    for campo in ("n_ref", "n_citadas", "aciertos", "aciertos_respaldados",
                                  "aciertos_sin_respaldo", "incorrectas", "citas_sin_respaldo"):
                        fila[campo] = r[campo]
                    fila["detalle_citas"] = r["detalle"]
            # abstención
            fila["abst_evaluable"] = qid in cerradas or bool(ref)
            clase = None
            if fila["abst_evaluable"]:
                if s is None:
                    clase = "fn"
                else:
                    if qid in cerradas:
                        acerto = s.get("respuesta_correcta") == k.get("respuesta_correcta")
                    else:
                        got = self.citas.extract(self.ev.answer_text(s))
                        acerto = bool(self.citas.bodies(ref) & self.citas.bodies(got))
                    clase = ("tp" if not acerto else "fp") if abst else ("tn" if acerto else "fn")
            fila["abst_clase"] = clase
            if s is not None:
                fila["palabras_ragas"] = len(self.ev.ragas_text(s).split()) if k["formato"] != "multiple_choice" else None
            filas.append(fila)
        return filas

    @staticmethod
    def agregar(filas):
        cerr = [f for f in filas if f["cerrada_puntuable"]]
        acc = sum(f["cerrada_acierto"] for f in cerr) / len(cerr) if cerr else 0.0
        n_ref = sum(f["n_ref"] for f in filas if f["citas_evaluable"])
        n_cit = sum(f["n_citadas"] for f in filas if f["citas_evaluable"])
        pond = sum(f["aciertos_respaldados"] + W_SIN_RESPALDO * f["aciertos_sin_respaldo"]
                   for f in filas if f["citas_evaluable"])
        recall = pond / n_ref if n_ref else 0.0
        tasa = sum(f["citas_sin_respaldo"] for f in filas if f["citas_evaluable"]) / n_cit if n_cit else 0.0
        indice = max(0.0, recall - 2 * tasa)
        clases = [f["abst_clase"] for f in filas if f["abst_evaluable"]]
        total = len(clases)
        calib = (clases.count("tn") + W_ABSTENCION * (clases.count("tp") + clases.count("fp"))) / total if total else 0.0
        salida = {"cerradas_acc": acc, "cerradas": PUNTOS["cerradas"] * acc,
                  "citas_recall": recall, "citas_sin_respaldo": tasa, "citas_indice": indice,
                  "citas": PUNTOS["citas"] * indice, "abstencion_calib": calib,
                  "abstencion": PUNTOS["abstencion"] * calib,
                  "abstenciones": sum(f["abstencion"] for f in filas)}
        salida["determinista_50"] = salida["cerradas"] + salida["citas"] + salida["abstencion"]
        return salida

    def verificar_igualdad(self, envios, tolerancia=0.011):
        """El agregado por ítem debe coincidir con el evaluador oficial (redondeo a 2 decimales)."""
        oficial = self.oficial(envios)
        propio = self.agregar(self.por_item(envios))
        for k in ("cerradas", "citas", "abstencion"):
            if abs(oficial[k]["puntos"] - propio[k]) > tolerancia:
                raise AssertionError(f"{k}: oficial {oficial[k]['puntos']} != por ítem {propio[k]:.3f}")
        return oficial, propio


def bootstrap(filas, n=2000, semilla=0, campos=("cerradas", "citas", "abstencion", "determinista_50")):
    """IC 95% remuestreando ítems. Con 50 ítems (15 cerradas) una pregunta mueve ~1,3 puntos."""
    rng = random.Random(semilla)
    muestras = {c: [] for c in campos}
    for _ in range(n):
        remuestra = [filas[rng.randrange(len(filas))] for _ in filas]
        agregado = Rubrica.agregar(remuestra)
        for c in campos:
            muestras[c].append(agregado[c])
    salida = {}
    for c in campos:
        valores = sorted(muestras[c])
        salida[c] = (valores[int(0.025 * n)], valores[int(0.975 * n) - 1])
    return salida


def pareado(filas_a, filas_b, n=2000, semilla=0, campo="determinista_50"):
    """Diferencia B - A con los mismos ítems remuestreados. Devuelve media, IC y P(B > A)."""
    por_id_a = {f["id"]: f for f in filas_a}
    por_id_b = {f["id"]: f for f in filas_b}
    ids = sorted(set(por_id_a) & set(por_id_b))
    rng = random.Random(semilla)
    difs = []
    for _ in range(n):
        muestra = [ids[rng.randrange(len(ids))] for _ in ids]
        a = Rubrica.agregar([por_id_a[i] for i in muestra])[campo]
        b = Rubrica.agregar([por_id_b[i] for i in muestra])[campo]
        difs.append(b - a)
    difs.sort()
    punto = Rubrica.agregar([por_id_b[i] for i in ids])[campo] - Rubrica.agregar([por_id_a[i] for i in ids])[campo]
    return {"diferencia": punto, "ic95": (difs[int(0.025 * n)], difs[int(0.975 * n) - 1]),
            "p_mejor": sum(d > 0 for d in difs) / n}


# ---------------------------------------------------------------- métricas sin decoder

def metricas_ranking(rubrica, rankings, lexico, k=10):
    """Proxy del e06: ¿alguna norma del fundamento aparece en los k primeros chunks? y MRR."""
    citas = rubrica.citas
    evaluables = aciertos = 0
    rr, recall_cuerpos = [], []
    for qid, fila in rubrica.filas.items():
        ref = citas.bodies(citas.extract(fila.get("legal_basis") or ""))
        if not ref or str(qid) not in rankings:
            continue
        evaluables += 1
        chunks = lexico.filas([c for c, _ in rankings[str(qid)][:k]])
        cuerpos = [citas.bodies(citas.extract(c["texto"])) for c in chunks]
        encontrados = [bool(ref & b) for b in cuerpos]
        if any(encontrados):
            aciertos += 1
            rr.append(1 / (encontrados.index(True) + 1))
        recall_cuerpos.append(len(ref & set().union(set(), *cuerpos)) / len(ref))
    return {"items_con_cita": evaluables,
            "respaldo_literal_top10": aciertos / evaluables if evaluables else None,
            "mrr_cita_top10": sum(rr) / evaluables if evaluables else None,
            "recall_cuerpos_top10": sum(recall_cuerpos) / evaluables if evaluables else None}


def techo_pasajes(rubrica, pasajes_por_id):
    """Techo de citas de una evidencia: lo que se obtendría citando todas las normas respaldadas.

    Es el recall ponderado máximo alcanzable (sin decoder), el indicador más directo de
    los 20 puntos de citas, y la fracción de ítems libres con al menos una norma de
    referencia respaldada (lo que cuenta como acierto en abstención).
    """
    citas = rubrica.citas
    n_ref = cubiertos = libres = libres_ok = 0
    distintos, cantidad = [], []
    for qid, fila in rubrica.filas.items():
        pasajes = pasajes_por_id.get(qid) or pasajes_por_id.get(str(qid)) or []
        respaldo = citas.bodies(set().union(set(), *(citas.extract(str(p.get("texto") or "")) for p in pasajes[:10])))
        distintos.append(len(respaldo))
        cantidad.append(len(pasajes))
        ref = citas.bodies(citas.extract(fila.get("legal_basis") or ""))
        if not ref:
            continue
        n_ref += len(ref)
        cubiertos += len(ref & respaldo)
        if fila["formato"] != "multiple_choice":
            libres += 1
            libres_ok += bool(ref & respaldo)
    return {"techo_recall_citas": cubiertos / n_ref if n_ref else None,
            "techo_puntos_citas": PUNTOS["citas"] * cubiertos / n_ref if n_ref else None,
            "libres_con_norma_respaldada": libres_ok / libres if libres else None,
            "normas_distintas_media": sum(distintos) / len(distintos) if distintos else 0,
            "pasajes_media": sum(cantidad) / len(cantidad) if cantidad else 0}


**juez**. Juez RAGAS oficial con caché y similitud local.

In [ ]:
%%modulo experimentos.e07.juez
"""Componente de texto libre (30 puntos): juez RAGAS oficial con caché y un proxy local.

answer_correctness = 0,75 * F1 de afirmaciones (juez LLM) + 0,25 * similitud coseno
(encoder abierto local). El juez es z-ai/glm-5.3-flash por OpenRouter, igual que
evaluate.py. Aquí se juzga ítem por ítem con caché para no pagar dos veces la misma
respuesta. La llave se lee como la lee el evaluador (scripts/.env o variable de
entorno); nunca se escribe en el notebook.

El proxy local solo calcula la parte de similitud (25%). No reemplaza al juez.
"""
import hashlib
import json
import math
from pathlib import Path


def _clave(pregunta, respuesta, referencia, modelo):
    return hashlib.sha256(json.dumps([pregunta, respuesta, referencia, modelo], ensure_ascii=False)
                          .encode("utf-8")).hexdigest()


def disponible(evaluador):
    """(True, "") si hay llave y dependencias del juez; si no, (False, motivo)."""
    import os
    evaluador.cargar_env()
    if not os.environ.get("OPENROUTER_API_KEY", "").strip():
        return False, "Falta OPENROUTER_API_KEY (scripts/.env o variable de entorno)"
    try:
        import datasets  # noqa: F401
        import langchain_huggingface  # noqa: F401
        import langchain_openai  # noqa: F401
        import ragas  # noqa: F401
    except ImportError as exc:
        return False, f"Falta la dependencia del juez: {exc.name} (pip install -r scripts/requirements-evaluador.txt)"
    return True, ""


def juzgar(rubrica, envios, ruta_cache):
    """Correctness por ítem juzgado y el puntaje sobre 30, con las reglas de evaluate.py."""
    ev = rubrica.ev
    subs = {s["id"]: s for s in envios}
    cache_ruta = Path(ruta_cache)
    cache = {}
    if cache_ruta.is_file():
        for linea in cache_ruta.read_text(encoding="utf-8").splitlines():
            if linea.strip():
                fila = json.loads(linea)
                cache[fila["clave"]] = fila["valor"]
    pendientes, claves = [], {}
    for qid in rubrica.juzgadas:
        s = subs.get(qid)
        if not s or s.get("abstencion"):
            continue
        k = rubrica.key[qid]
        c = _clave(k["pregunta"], ev.ragas_text(s), k["respuesta_esperada"], ev.JUEZ_MODELO)
        claves[qid] = c
        if c not in cache:
            pendientes.append(qid)
    if pendientes:
        llave = ev.api_key_juez(ev.JUEZ_BASE_URL)
        from datasets import Dataset
        from langchain_huggingface import HuggingFaceEmbeddings
        from langchain_openai import ChatOpenAI
        from ragas import evaluate as ragas_evaluate
        from ragas.embeddings import LangchainEmbeddingsWrapper
        from ragas.llms import LangchainLLMWrapper
        from ragas.metrics import answer_correctness
        filas = {"question": [], "answer": [], "ground_truth": []}
        for qid in pendientes:
            filas["question"].append(rubrica.key[qid]["pregunta"])
            filas["answer"].append(ev.ragas_text(subs[qid]))
            filas["ground_truth"].append(rubrica.key[qid]["respuesta_esperada"])
        llm = LangchainLLMWrapper(ChatOpenAI(model=ev.JUEZ_MODELO, base_url=ev.JUEZ_BASE_URL, api_key=llave,
                                             temperature=0, max_tokens=ev.JUEZ_MAX_TOKENS,
                                             extra_body={"reasoning": ev.JUEZ_REASONING}))
        emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=ev.JUEZ_ENCODER))
        resultado = ragas_evaluate(Dataset.from_dict(filas), metrics=[answer_correctness], llm=llm, embeddings=emb)
        serie = resultado.to_pandas()["answer_correctness"]
        cache_ruta.parent.mkdir(parents=True, exist_ok=True)
        with cache_ruta.open("a", encoding="utf-8") as f:
            for qid, valor in zip(pendientes, serie):
                valor = None if valor is None or (isinstance(valor, float) and math.isnan(valor)) else float(valor)
                if valor is None:
                    continue  # un fallo del juez no se guarda: se reintenta la próxima vez
                cache[claves[qid]] = valor
                f.write(json.dumps({"clave": claves[qid], "id": qid, "valor": valor}) + "\n")
    por_item = {qid: (cache.get(claves[qid]) if qid in claves else 0.0) for qid in rubrica.juzgadas}
    fallidos = [qid for qid, v in por_item.items() if v is None]
    correctness = sum(v or 0.0 for v in por_item.values()) / len(rubrica.juzgadas) if rubrica.juzgadas else 0.0
    return {"por_item": por_item, "correctness": correctness, "puntos": 30 * correctness,
            "fallidos": fallidos, "nuevos_juzgados": len(pendientes)}


class SimilitudLocal:
    """Coseno con multilingual-e5-large sin prefijos, como HuggingFaceEmbeddings en el juez."""

    def __init__(self, nombre="intfloat/multilingual-e5-large", simulado=False):
        self.simulado = simulado
        if simulado:
            from experimentos.e07.simulado import EncoderSimulado
            self.enc = EncoderSimulado("simulado/e5")
            return
        try:
            from sentence_transformers import SentenceTransformer
            self.st = SentenceTransformer(nombre, device="cuda")
        except ImportError:
            self.st = None
            import torch
            from transformers import AutoModel, AutoTokenizer
            self.torch = torch
            self.tok = AutoTokenizer.from_pretrained(nombre)
            self.modelo = AutoModel.from_pretrained(nombre).to("cuda").eval()

    def vectores(self, textos):
        import numpy as np
        if self.simulado:
            return self.enc.codificar(textos, tarea="passage")
        if self.st is not None:
            return np.asarray(self.st.encode(textos, normalize_embeddings=True))
        torch = self.torch
        salida = []
        for t in textos:
            tok = self.tok([t], truncation=True, max_length=512, return_tensors="pt").to("cuda")
            with torch.inference_mode():
                h = self.modelo(**tok).last_hidden_state
                m = tok["attention_mask"].unsqueeze(-1)
                v = torch.nn.functional.normalize((h * m).sum(1) / m.sum(1), dim=1)
            salida.append(v.float().cpu().numpy()[0])
        return np.vstack(salida)

    def por_item(self, rubrica, envios):
        subs = {s["id"]: s for s in envios}
        ids, respuestas, referencias = [], [], []
        for qid in rubrica.juzgadas:
            s = subs.get(qid)
            if not s or s.get("abstencion"):
                continue
            ids.append(qid)
            respuestas.append(rubrica.ev.ragas_text(s) or " ")
            referencias.append(rubrica.key[qid]["respuesta_esperada"] or " ")
        if not ids:
            return {}
        a, b = self.vectores(respuestas), self.vectores(referencias)
        return {qid: float((a[i] * b[i]).sum()) for i, qid in enumerate(ids)}

    def cerrar(self):
        from produccion.modelos import liberar
        for atributo in ("st", "modelo", "enc"):
            if hasattr(self, atributo):
                liberar(getattr(self, atributo))
                setattr(self, atributo, None)


**pipeline**. Etapas E00 a E08.

In [ ]:
%%modulo experimentos.e07.pipeline
"""Etapas E00-E08 del e07. El notebook solo llama a estas funciones en orden.

Todas las etapas:
  * recorren la grilla completa, sin descartar configuraciones por sus resultados;
  * guardan cada nodo en su carpeta y se saltan los nodos completos;
  * registran errores y siguen con el resto (un modelo que no carga no tumba la grilla);
  * cargan cada modelo una sola vez por etapa y lo liberan al terminar.
"""
import hashlib
import json
import os
import platform
import shutil
import time
from collections import defaultdict
from pathlib import Path

from experimentos.e07 import indices, rubrica as rub
from experimentos.e07.etapas import Bitacora, Espacio, escribir_json, escribir_jsonl, leer_json, leer_jsonl
from experimentos.e07.grilla import Grilla
from produccion import politica
from produccion.configuracion import leer_jsonl as leer_entradas, preparar, preparar_entrada, revision
from produccion.evidencia import EvidenciaV04, cargar_citaciones
from produccion.modelos import cargar_encoder, cargar_reranker, fijar_determinismo, liberar
from produccion.recuperador import (Fuentes, IndiceDenso, Lexico, Selector, anclas_por_area, consultas,
                                    expansion, indice_identidades, pasajes_salida, recuperar)

RAIZ = Path(__file__).resolve().parents[2]


def _hash_lista(valores):
    return hashlib.sha256(json.dumps(sorted(valores), ensure_ascii=False).encode()).hexdigest()[:12]


def _cuda():
    try:
        import torch
        return torch if torch.cuda.is_available() else None
    except ImportError:
        return None


class Contexto:
    def __init__(self, config, grilla, salida, reutilizar=(), desde_etapa=None, detener_en_error=False):
        self.simulado = False
        self.config_original = json.loads(json.dumps(config))
        self.config = preparar(config)
        self.rutas = self.config["rutas"]
        self.espacio = Espacio(salida, reutilizar, desde_etapa, detener_en_error)
        self.grilla = Grilla(grilla)
        self.runs_e06 = None
        self.citas = cargar_citaciones(self.rutas["scripts_oficiales"])
        self.evidencia = EvidenciaV04(self.rutas["manifiesto"], self.rutas["textos"], self.citas)
        self.rubrica = rub.Rubrica(self.rutas["scripts_oficiales"], self.rutas["muestra"])
        self.entradas = [preparar_entrada(x) for x in leer_entradas(self.rutas["muestra"])]
        self.anclas = anclas_por_area(self.rutas.get("seed_targets") or "_no_existe_")
        self.snapshot_id = hashlib.sha256(Path(self.rutas["manifiesto"]).read_bytes()).hexdigest()[:16]
        self._lexico = None
        fijar_determinismo(0)
        escribir_json(self.espacio.salida / "grilla_usada.json", self.grilla.datos)
        escribir_json(self.espacio.salida / "config_usada.json", self.config_original)

    # ------------------------------------------------------------ nodos base
    def nodo_lexico(self):
        return self.espacio.nodo("E01_lexico", "chunks", {"snapshot": self.snapshot_id, "ventana": 1500,
                                                          "solapamiento": 200, "segmentador": "articulos_v1"})

    def ruta_sqlite(self):
        nodo = self.nodo_lexico()
        if not nodo.listo():
            raise RuntimeError("Falta E01 (índice léxico)")
        return indices.resolver(nodo.buscar(), "chunks.sqlite")

    @property
    def lexico(self):
        if self._lexico is None:
            self._lexico = Lexico(self.ruta_sqlite())
        return self._lexico

    def nodo_denso(self, encoder):
        return self.espacio.nodo("E02_denso", encoder.replace("/", "__"),
                                 {"lexico": self.nodo_lexico().clave, "encoder": encoder,
                                  "revision": revision(self.config, encoder)})

    def ruta_faiss(self, encoder):
        nodo = self.nodo_denso(encoder)
        return indices.resolver(nodo.buscar(), "index.faiss") if nodo.listo() else None

    def resolver_modelo(self, nombre):
        """ "ft:ultimo" o "ft:<version>" -> (nombre_registro, carpeta local) o (None, None) si no existe."""
        if not nombre or not str(nombre).startswith("ft:"):
            return nombre, None
        registro = {}
        for raiz in [self.espacio.salida] + self.espacio.reutilizar:
            registro.update(leer_json(raiz / "E09_finetune" / "modelos.json", {}))
        if not registro:
            return None, None
        version = nombre[3:]
        if version == "ultimo":
            version = sorted(registro, key=lambda v: registro[v]["fecha"])[-1]
        if version not in registro:
            return None, None
        return "ft:" + version, registro[version]["carpeta"]

    def R(self, nombre):
        r = self.grilla.R(nombre)
        if r.get("reranker"):
            resuelto, _ = self.resolver_modelo(r["reranker"])
            r["reranker"] = resuelto
            r["_pendiente"] = resuelto is None
        return r


# ---------------------------------------------------------------- E00

def e00_entorno(ctx):
    def correr(carpeta):
        info = {"python": platform.python_version(), "sistema": platform.platform(), "simulado": ctx.simulado,
                "datos": str(ctx.config["datos"]), "salida": str(ctx.espacio.salida),
                "reutilizar": [str(r) for r in ctx.espacio.reutilizar], "snapshot_id": ctx.snapshot_id,
                "documentos": len(ctx.evidencia.documentos), "preguntas": len(ctx.entradas),
                "disco_libre_gb": round(shutil.disk_usage(ctx.espacio.salida).free / 2**30, 1),
                "grilla": ctx.grilla.resumen()}
        for modulo in ("torch", "transformers", "faiss", "numpy", "pandas", "peft", "ragas"):
            try:
                info[modulo] = __import__(modulo).__version__
            except Exception:
                info[modulo] = None
        torch = _cuda()
        if torch:
            p = torch.cuda.get_device_properties(0)
            info["gpu"] = {"nombre": p.name, "vram_gib": round(p.total_memory / 2**30, 1),
                           "bf16": torch.cuda.is_bf16_supported(), "cuda": torch.version.cuda}
        try:
            import psutil
            info["ram_gib"] = round(psutil.virtual_memory().total / 2**30, 1)
        except ImportError:
            pass
        prohibidos = {"legal_basis", "respuesta_correcta", "respuesta_esperada", "texto_respuesta_correcta"}
        assert not any(prohibidos & set(e) for e in ctx.entradas), "La entrada trae campos de respuesta"
        escribir_json(carpeta / "entorno.json", info)
        return info
    nodo = ctx.espacio.nodo("E00_entorno", "entorno", {"fecha": time.strftime("%Y%m%d-%H%M%S")})
    return ctx.espacio.ejecutar(nodo, correr)


# ---------------------------------------------------------------- E01 / E02

def e01_lexico(ctx):
    nodo = ctx.nodo_lexico()

    def correr(carpeta):
        if ctx.runs_e06 and not ctx.simulado:
            meta = indices.importar_lexico_e06(ctx.runs_e06, carpeta, ctx.snapshot_id)
            if meta:
                print("Índice léxico importado del e06:", ctx.runs_e06)
                return meta
        return indices.construir_lexico(ctx.rutas["manifiesto"], ctx.rutas["textos"], carpeta)
    return ctx.espacio.ejecutar(nodo, correr)


def e02_densos(ctx, solo=None):
    """Un índice por encoder de la grilla. Qwen3-Embedding-4B tarda horas: va al final."""
    orden = sorted(ctx.grilla.encoders(), key=lambda e: ("4B" in e, e))
    total = indices.total_chunks(ctx.ruta_sqlite())
    resumenes = {}
    for encoder in orden:
        if solo and encoder not in solo:
            continue
        nodo = ctx.nodo_denso(encoder)

        def correr(carpeta, encoder=encoder):
            if ctx.runs_e06 and not ctx.simulado:
                meta = indices.importar_denso_e06(ctx.runs_e06, encoder, carpeta, total)
                if meta:
                    print(f"{encoder}: índice importado del e06")
                    return meta
            modelo = cargar_encoder(encoder, revision(ctx.config, encoder))
            try:
                return indices.construir_denso(modelo, ctx.ruta_sqlite(), carpeta)
            finally:
                modelo.cerrar()
        resumenes[encoder] = ctx.espacio.ejecutar(nodo, correr)
    return resumenes


# ---------------------------------------------------------------- E03 búsquedas densas

def _textos_consulta(ctx, encoder):
    textos = set()
    for nombre in ctx.grilla.nombres_R:
        r = ctx.grilla.R(nombre)
        if encoder in r["densos"]:
            for e in ctx.entradas:
                textos.update(consultas(e, r["consulta"]))
    return sorted(textos)


def nodo_busquedas(ctx, encoder):
    textos = _textos_consulta(ctx, encoder)
    return ctx.espacio.nodo("E03_busquedas", encoder.replace("/", "__"),
                            {"denso": ctx.nodo_denso(encoder).clave, "textos": _hash_lista(textos), "k": 100}), textos


def e03_busquedas(ctx):
    """Embeddings de consulta y top-100 FAISS por encoder. El índice se carga de a uno (RAM)."""
    salida = {}
    for encoder in ctx.grilla.encoders():
        nodo, textos = nodo_busquedas(ctx, encoder)

        def correr(carpeta, encoder=encoder, textos=textos):
            ruta = ctx.ruta_faiss(encoder)
            if ruta is None:
                raise RuntimeError(f"Falta el índice denso de {encoder} (E02)")
            indice = IndiceDenso(ruta)
            modelo = cargar_encoder(encoder, revision(ctx.config, encoder))
            cache, tiempos = {}, []
            try:
                for texto in textos:
                    t = time.perf_counter()
                    vector = modelo.codificar([texto], tarea="query")[0]
                    cache[json.dumps(["denso", encoder, texto, 100], ensure_ascii=False)] = indice.buscar(vector, 100)
                    tiempos.append((time.perf_counter() - t) * 1000)
            finally:
                modelo.cerrar()
                del indice
            escribir_json(carpeta / "cache.json", cache)
            return {"consultas": len(textos), "ms_media": sum(tiempos) / max(1, len(tiempos))}
        salida[encoder] = ctx.espacio.ejecutar(nodo, correr)
    return salida


def _cache_densa(ctx, encoders):
    cache, ms = {}, {}
    for encoder in encoders:
        nodo, _ = nodo_busquedas(ctx, encoder)
        if not nodo.listo():
            return None, None
        cache.update(leer_json(nodo.buscar() / "cache.json", {}))
        ms[encoder] = (nodo.resumen() or {}).get("ms_media", 0)
    return cache, ms


# ---------------------------------------------------------------- E04 rankings

def nodo_ranking(ctx, nombre):
    r = ctx.R(nombre)
    entradas = {k: v for k, v in r.items() if not k.startswith("_")}
    entradas["densos_claves"] = [ctx.nodo_denso(e).clave for e in r["densos"]]
    entradas["lexico"] = ctx.nodo_lexico().clave
    entradas["anclas"] = r.get("expansion_area", 0) and _hash_lista([json.dumps(v) for v in ctx.anclas.items()])
    return ctx.espacio.nodo("E04_rankings", nombre, entradas), r


def _cache_rerank(ctx, reranker):
    slug = str(reranker).replace("/", "__").replace(":", "_")
    ruta = ctx.espacio.salida / "E04_rankings" / "_cache" / f"{slug}.jsonl"
    cache = {}
    for raiz in [ctx.espacio.salida] + ctx.espacio.reutilizar:
        for fila in leer_jsonl(raiz / "E04_rankings" / "_cache" / f"{slug}.jsonl"):
            cache[fila["k"]] = fila["v"]
    return cache, ruta


def e04_rankings(ctx):
    grupos = defaultdict(list)
    for nombre in ctx.grilla.nombres_R:
        nodo, r = nodo_ranking(ctx, nombre)
        if r.get("_pendiente"):
            ctx.espacio.registrar(nodo=nodo.id, estado="pendiente", motivo="requiere E09 (reranker afinado)")
            continue
        grupos[r.get("reranker")].append((nombre, nodo, r))
    resultados = {}
    for reranker, miembros in grupos.items():
        pendientes = [m for m in miembros if not m[1].listo()]
        modelo = None
        if pendientes and reranker:
            _, local = ctx.resolver_modelo(reranker)
            base = ctx.grilla.datos["finetune"]["reranker_base"] if local else reranker
            if ctx.simulado and local:
                base = "simulado/bge-reranker"
            try:
                modelo = cargar_reranker(base, revision(ctx.config, base), 1024, ctx.config.get("lote_rerank", 16), local)
            except Exception as error:
                for nombre, nodo, _ in pendientes:
                    ctx.espacio.registrar(nodo=nodo.id, estado="error", error=f"no cargó {reranker}: {error}")
                continue
        cache_rr, ruta_cache = _cache_rerank(ctx, reranker) if reranker else ({}, None)
        for nombre, nodo, r in miembros:
            def correr(carpeta, r=r, nombre=nombre):
                cache, ms_denso = _cache_densa(ctx, r["densos"])
                if cache is None:
                    raise RuntimeError("Faltan búsquedas densas (E03) para " + ", ".join(r["densos"]))
                cache.update(cache_rr)
                fuentes = Fuentes(ctx.lexico, rerankers={reranker: modelo} if modelo else {}, cache=cache)
                rankings, tiempos = {}, {}
                for e in ctx.entradas:
                    antes = set(cache)
                    t = time.perf_counter()
                    rankings[str(e["id"])] = recuperar(e, r, fuentes, ctx.anclas)
                    tiempos[str(e["id"])] = (time.perf_counter() - t) * 1000
                    nuevas = [k for k in cache if k not in antes and k.startswith('["rerank"')]
                    if nuevas and ruta_cache:
                        ruta_cache.parent.mkdir(parents=True, exist_ok=True)
                        with ruta_cache.open("a", encoding="utf-8") as f:
                            for k in nuevas:
                                f.write(json.dumps({"k": k, "v": cache[k]}) + "\n")
                                cache_rr[k] = cache[k]
                escribir_json(carpeta / "rankings.json", rankings)
                n_consultas = len(consultas(ctx.entradas[0], r["consulta"]))
                metricas = rub.metricas_ranking(ctx.rubrica, rankings, ctx.lexico)
                metricas.update(ms_recuperacion_medido=sum(tiempos.values()) / len(tiempos),
                                ms_denso_estimado=sum(ms_denso.values()) * n_consultas)
                escribir_json(carpeta / "metricas.json", metricas)
                return metricas
            resultados[nombre] = ctx.espacio.ejecutar(nodo, correr)
        if modelo:
            modelo.cerrar()
    return resultados


# ---------------------------------------------------------------- E05 pasajes

def identidades(ctx):
    nodo = ctx.espacio.nodo("E05_pasajes", "_identidades", {"snapshot": ctx.snapshot_id, "simulado": ctx.simulado})

    def correr(carpeta):
        datos = indice_identidades(ctx.evidencia, carpeta / "identidades.json")
        return {"cuerpos": len(datos)}
    ctx.espacio.ejecutar(nodo, correr)
    return indice_identidades(ctx.evidencia, nodo.buscar() / "identidades.json")


def nodo_pasajes(ctx, nombre_r, nombre_p):
    nodo_r, _ = nodo_ranking(ctx, nombre_r)
    return ctx.espacio.nodo("E05_pasajes", f"{nombre_r}__{nombre_p}",
                            {"ranking": nodo_r.clave, "P": ctx.grilla.P(nombre_p)}), nodo_r


def e05_pasajes(ctx):
    selector = Selector(ctx.lexico, ctx.evidencia, identidades(ctx), ctx.anclas)
    resultados = {}
    for nombre_r in ctx.grilla.nombres_R:
        for nombre_p in ctx.grilla.nombres_P:
            nodo, nodo_r = nodo_pasajes(ctx, nombre_r, nombre_p)
            if not nodo_r.listo():
                continue
            P = ctx.grilla.P(nombre_p)

            def correr(carpeta, nodo_r=nodo_r, P=P):
                rankings = leer_json(nodo_r.buscar() / "rankings.json")
                filas, por_id, tiempos = [], {}, []
                for e in ctx.entradas:
                    t = time.perf_counter()
                    pasajes = selector.seleccionar(e, [tuple(x) for x in rankings[str(e["id"])]], P)
                    tiempos.append((time.perf_counter() - t) * 1000)
                    por_id[e["id"]] = pasajes
                    filas.append({"id": e["id"], "pasajes": pasajes,
                                  "origenes": [p.get("origen") for p in pasajes]})
                escribir_jsonl(carpeta / "pasajes.jsonl", filas)
                metricas = rub.techo_pasajes(ctx.rubrica, por_id)
                metricas["ms_seleccion"] = sum(tiempos) / len(tiempos)
                metricas["pasajes_router"] = sum(o == "router" for f in filas for o in f["origenes"])
                metricas["pasajes_ancla"] = sum(o == "ancla" for f in filas for o in f["origenes"])
                escribir_json(carpeta / "metricas.json", metricas)
                return metricas
            resultados[(nombre_r, nombre_p)] = ctx.espacio.ejecutar(nodo, correr)
    return resultados


# ---------------------------------------------------------------- E06 generación

def nodo_generacion(ctx, nombre_r, nombre_p, nombre_d):
    nodo_p, _ = nodo_pasajes(ctx, nombre_r, nombre_p)
    esp = ctx.grilla.D(nombre_d)
    return ctx.espacio.nodo("E06_generacion", f"{nombre_r}__{nombre_p}__{nombre_d}",
                            {"pasajes": nodo_p.clave, "decoder": esp, "prompt": _huella_prompt()}), nodo_p, esp


def _huella_prompt():
    """Si cambia el prompt de politica.py, cambian las claves y se regenera."""
    texto = politica.SISTEMA + json.dumps(politica.INSTRUCCIONES, ensure_ascii=False) + json.dumps(politica.LONGITUDES)
    return hashlib.sha256(texto.encode()).hexdigest()[:10]


def e06_generacion(ctx, max_corridas=None):
    from produccion.decoder import cargar as cargar_decoder, generar
    corridas = ctx.grilla.corridas_generacion()[:max_corridas]
    por_decoder = defaultdict(list)
    for r, p, d in corridas:
        por_decoder[d].append((r, p))
    resultados = {}
    for nombre_d, pares in por_decoder.items():
        nodos = [(r, p) + nodo_generacion(ctx, r, p, nombre_d) for r, p in pares]
        pendientes = [n for n in nodos if not n[2].listo() and n[3].listo()]
        for r, p, nodo, nodo_p, _ in nodos:
            if not nodo_p.listo():
                ctx.espacio.registrar(nodo=nodo.id, estado="pendiente", motivo="faltan pasajes (E05)")
        if not pendientes:
            for r, p, nodo, _, _ in nodos:
                if nodo.listo():
                    resultados[(r, p, nombre_d)] = nodo.resumen()
            continue
        esp = ctx.grilla.D(nombre_d)
        torch = _cuda()
        try:
            decoder = cargar_decoder(esp, esp.get("revision") or revision(ctx.config, esp["modelo"]))
        except Exception as error:
            for r, p, nodo, _, _ in pendientes:
                ctx.espacio.registrar(nodo=nodo.id, estado="error", error=f"no cargó {esp['modelo']}: {error}")
            print(f"No cargó {esp['modelo']}: {error}")
            continue
        if torch:
            torch.cuda.reset_peak_memory_stats()
        ruta_cache = ctx.espacio.salida / "E06_generacion" / "_cache" / f"{nombre_d}.jsonl"
        cache = Bitacora(ruta_cache, campo="clave")
        for r, p, nodo, nodo_p, _ in nodos:
            def correr(carpeta, nodo_p=nodo_p):
                pasajes = {f["id"]: f["pasajes"] for f in leer_jsonl(nodo_p.buscar() / "pasajes.jsonl")}
                bitacora = Bitacora(carpeta / "crudo.jsonl")
                for n, e in enumerate(ctx.entradas, 1):
                    if bitacora.hecho(e["id"]):
                        continue
                    fila = _generar_item(decoder, generar, e, pasajes[e["id"]], ctx, esp, cache)
                    bitacora.agregar(fila)
                    print(f"{nombre_d} {n}/{len(ctx.entradas)} id={e['id']} {fila['generacion_ms']} ms"
                          f"{' (caché)' if fila['desde_cache'] else ''}", end="\r")
                filas = list(bitacora.filas.values())
                ms = sorted(f["generacion_ms"] for f in filas if not f["desde_cache"]) or [0]
                return {"items": len(filas), "json_invalido": sum(not f["registro"].get("json_valido", False) for f in filas),
                        "letra_por_logits": sum(bool(f.get("letra_por_logits")) for f in filas),
                        "errores": sum(bool(f.get("error")) for f in filas),
                        "ms_media": sum(ms) / len(ms), "ms_p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))],
                        "tokens_salida_media": sum(f["registro"].get("tokens_salida") or 0 for f in filas) / len(filas),
                        "parametros": decoder.parametros,
                        "vram_pico_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2) if torch else None}
            if nodo_p.listo():
                resultados[(r, p, nombre_d)] = ctx.espacio.ejecutar(nodo, correr)
        decoder.cerrar()
        liberar(decoder)
    return resultados


def _generar_item(decoder, generar, entrada, pasajes, ctx, esp, cache):
    letras = list((entrada.get("opciones") or {}).keys())
    inicio = time.perf_counter()
    try:
        usados, conversacion, tokens, omitidas = politica.ajustar(
            decoder, entrada, pasajes, ctx.evidencia, esp["contexto"], esp["max_tokens"][entrada["formato"]],
            esp["max_caracteres_pasaje"])
        clave = hashlib.sha256(json.dumps([esp, conversacion], ensure_ascii=False).encode()).hexdigest()
        if cache.hecho(clave):
            guardado = cache.filas[clave]
            return {"id": entrada["id"], "crudo": guardado["crudo"], "usados": usados,
                    "registro": guardado["registro"], "letra_por_logits": guardado.get("letra_por_logits"),
                    "generacion_ms": guardado["generacion_ms"], "desde_cache": True}
        crudo, usados, conversacion, registro = generar(decoder, entrada, pasajes, ctx.evidencia, esp)
        letra = None
        if entrada["formato"] == "multiple_choice" and (crudo or {}).get("respuesta_correcta") not in letras:
            letra, puntajes = decoder.elegir_letra(conversacion, letras)
            crudo = dict(crudo or {}, respuesta_correcta=letra)
            letra = {"letra": letra, "puntajes": puntajes}
        ms = round((time.perf_counter() - inicio) * 1000)
        cache.agregar({"clave": clave, "crudo": crudo, "registro": registro, "letra_por_logits": letra,
                       "generacion_ms": ms})
        return {"id": entrada["id"], "crudo": crudo, "usados": usados, "registro": registro,
                "letra_por_logits": letra, "generacion_ms": ms, "desde_cache": False}
    except Exception as error:
        return {"id": entrada["id"], "crudo": None, "usados": pasajes, "registro": {"json_valido": False},
                "error": f"{type(error).__name__}: {error}", "letra_por_logits": None,
                "generacion_ms": round((time.perf_counter() - inicio) * 1000), "desde_cache": False}


# ---------------------------------------------------------------- E07 respuestas y rúbrica

def nodo_respuestas(ctx, nombre_r, nombre_p, nombre_d, nombre_pol):
    nodo_g, _, _ = nodo_generacion(ctx, nombre_r, nombre_p, nombre_d)
    return ctx.espacio.nodo("E07_respuestas", f"{nombre_r}__{nombre_p}__{nombre_d}__{nombre_pol}",
                            {"generacion": nodo_g.clave, "politica": ctx.grilla.politica(nombre_pol)}), nodo_g


def e07_respuestas(ctx):
    resultados = {}
    for r, p, d in ctx.grilla.corridas_generacion():
        for pol in ctx.grilla.nombres_politica:
            nodo, nodo_g = nodo_respuestas(ctx, r, p, d, pol)
            if not nodo_g.listo():
                continue
            politica_cfg = ctx.grilla.politica(pol)

            def correr(carpeta, nodo_g=nodo_g, politica_cfg=politica_cfg):
                crudos = {f["id"]: f for f in leer_jsonl(nodo_g.buscar() / "crudo.jsonl")}
                envios, registros = [], []
                for e in ctx.entradas:
                    fila = crudos.get(e["id"])
                    if fila is None:
                        continue
                    respuesta, registro = politica.postprocesar(e, fila["crudo"], fila["usados"], ctx.evidencia, politica_cfg)
                    respuesta["pasajes_recuperados"] = pasajes_salida(respuesta["pasajes_recuperados"])
                    envios.append(respuesta)
                    registros.append({"id": e["id"], **registro})
                escribir_jsonl(carpeta / "submission.jsonl", envios)
                oficial, propio = ctx.rubrica.verificar_igualdad(envios)
                items = ctx.rubrica.por_item(envios)
                escribir_json(carpeta / "score_oficial.json", oficial)
                escribir_jsonl(carpeta / "por_item.jsonl", items)
                escribir_jsonl(carpeta / "postproceso.jsonl", registros)
                ic = rub.bootstrap(items, n=1000)
                return {**propio, "ic95": ic, "errores_formato": oficial["validacion"]["errores"],
                        "oraciones_saneadas": sum(len(x.get("oraciones_eliminadas", [])) for x in registros)}
            resultados[(r, p, d, pol)] = ctx.espacio.ejecutar(nodo, correr)
    return resultados


# ---------------------------------------------------------------- E08 juez

def e08_juez(ctx, ragas=True, juzgar_todo=False):
    """Similitud local para todo; juez RAGAS para la base y las mejores (o todo)."""
    from experimentos.e07 import juez
    nodos = []
    for r, p, d in ctx.grilla.corridas_generacion():
        for pol in ctx.grilla.nombres_politica:
            nodo, _ = nodo_respuestas(ctx, r, p, d, pol)
            if nodo.listo():
                nodos.append(((r, p, d, pol), nodo))
    if not nodos:
        return {}
    # Similitud local (25% de la métrica)
    sim_nodos = [(c, n, ctx.espacio.nodo("E08_juez", "sim__" + "__".join(c), {"respuestas": n.clave}))
                 for c, n in nodos]
    faltan = [x for x in sim_nodos if not x[2].listo()]
    if faltan:
        similitud = juez.SimilitudLocal(simulado=ctx.simulado)
        for combinacion, nodo, nodo_sim in faltan:
            def correr(carpeta, nodo=nodo):
                envios = leer_jsonl(nodo.buscar() / "submission.jsonl")
                por_item = similitud.por_item(ctx.rubrica, envios)
                escribir_json(carpeta / "similitud.json", por_item)
                media = sum(por_item.get(q, 0.0) for q in ctx.rubrica.juzgadas) / len(ctx.rubrica.juzgadas)
                return {"similitud_media": media}
            ctx.espacio.ejecutar(nodo_sim, correr)
        similitud.cerrar()
    if not ragas:
        return {"similitud": len(sim_nodos)}
    ok, motivo = juez.disponible(ctx.rubrica.ev)
    if not ok:
        print("Juez RAGAS no disponible:", motivo)
        return {"similitud": len(sim_nodos), "ragas": motivo}
    base = ctx.grilla.base
    elegibles = [(c, n) for c, n in nodos if not ctx.grilla.datos["politicas"][c[3]].get("solo_medicion")]
    orden = sorted(elegibles, key=lambda x: -(x[1].resumen() or {}).get("determinista_50", 0))
    maximo = len(orden) if juzgar_todo else ctx.grilla.datos["juez"]["max_configuraciones"]
    elegidos = [x for x in nodos if x[0] == (base["R"], base["P"], base["D"], base["politica"])]
    for x in orden:
        if len(elegidos) >= maximo:
            break
        if x not in elegidos:
            elegidos.append(x)
    ruta_cache = ctx.espacio.salida / "E08_juez" / "_cache_ragas.jsonl"
    for raiz in ctx.espacio.reutilizar:  # respuestas ya juzgadas en corridas anteriores
        anterior = raiz / "E08_juez" / "_cache_ragas.jsonl"
        if anterior.is_file() and not ruta_cache.is_file():
            ruta_cache.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(anterior, ruta_cache)
    for combinacion, nodo in elegidos:
        nodo_j = ctx.espacio.nodo("E08_juez", "ragas__" + "__".join(combinacion), {"respuestas": nodo.clave})

        def correr(carpeta, nodo=nodo):
            envios = leer_jsonl(nodo.buscar() / "submission.jsonl")
            resultado = juez.juzgar(ctx.rubrica, envios, ruta_cache)
            escribir_json(carpeta / "ragas.json", resultado)
            if resultado["fallidos"]:
                raise RuntimeError(f"El juez no respondió en {len(resultado['fallidos'])} ítems; se reintenta luego")
            return {k: resultado[k] for k in ("correctness", "puntos", "nuevos_juzgados")}
        ctx.espacio.ejecutar(nodo_j, correr)
    return {"similitud": len(sim_nodos), "ragas": len(elegidos)}


**finetune**. E09: sintéticas, control de fuga, negativos difíciles, reranker afinado, LoRA opcional.

In [ ]:
%%modulo experimentos.e07.finetune
"""E09: datos sintéticos, negativos difíciles y fine-tuning (reranker; decoder LoRA opcional).

Reglas que este módulo respeta (enunciado, secciones 3.2 y 8):
  * Los datos sintéticos salen solo del corpus y de un decoder ABIERTO local.
  * Nunca se usa sample_50 (ni legal_basis ni respuestas) para entrenar: es nuestro
    único conjunto de evaluación. Se descartan las consultas sintéticas que se
    parezcan a alguna pregunta de la muestra (control de fuga).
  * El índice no cambia: afinar el reranker no obliga a reindexar 1,3 M chunks.

Cada entrenamiento crea una versión nueva (ft:<version>) registrada en
E09_finetune/modelos.json con sus datos, hiperparámetros y métricas. La grilla usa
"ft:ultimo" (R13, R14), así que volver a entrenar = correr E09 y luego E04-E10.
"""
import hashlib
import json
import random
import re
import time
from datetime import datetime
from pathlib import Path

from experimentos.e07.etapas import Bitacora, escribir_json, escribir_jsonl, leer_json, leer_jsonl

TIPOS = [
    "una pregunta de selección múltiple (solo el enunciado, sin opciones)",
    "una pregunta de respuesta corta",
    "un caso práctico breve que termine en una pregunta",
]

PROMPT_SINTETICO = (
    "Eres profesor de derecho colombiano y redactas preguntas de examen.\n\n"
    "PASAJE\n{pasaje}\n\n"
    "Redacta {tipo} que se responda con este pasaje. No menciones el número del artículo "
    "ni el nombre o número de la norma. Escribe en español. Responde solo con un objeto JSON "
    "con la clave \"pregunta\"."
)


def _palabras(texto):
    return re.findall(r"\w+", (texto or "").lower())


def _ngramas(texto, n=5):
    p = _palabras(texto)
    return {" ".join(p[i:i + n]) for i in range(len(p) - n + 1)}


# ---------------------------------------------------------------- 1. muestreo de unidades

def muestrear_unidades(lexico, n, semilla=0, proporcion_normas=0.75):
    """Unidades del corpus, 75% artículos de normas y 25% resto (sentencias, conceptos)."""
    rng = random.Random(semilla)
    normas = [r[0] for r in lexico.conn.execute(
        "SELECT MIN(id) FROM chunks WHERE articulo IS NOT NULL AND tipo NOT IN ('sentencia','auto') "
        "AND length(texto) > 300 GROUP BY unidad_id")]
    otros = [r[0] for r in lexico.conn.execute(
        "SELECT MIN(id) FROM chunks WHERE (articulo IS NULL OR tipo IN ('sentencia','auto')) "
        "AND length(texto) > 600 GROUP BY unidad_id")]
    n_normas = min(len(normas), int(n * proporcion_normas))
    elegidos = rng.sample(normas, n_normas) + rng.sample(otros, min(len(otros), n - n_normas))
    rng.shuffle(elegidos)
    return lexico.filas(elegidos)


# ---------------------------------------------------------------- 2. consultas sintéticas

def generar_sinteticas(decoder, filas, carpeta, max_caracteres=1500, semilla=0):
    """Una pregunta por unidad con el decoder abierto (greedy). Reanudable."""
    from produccion.decoder import extraer_json
    bitacora = Bitacora(Path(carpeta) / "sinteticas.jsonl", campo="chunk_id")
    rng = random.Random(semilla)
    for n, fila in enumerate(filas, 1):
        tipo = TIPOS[rng.randrange(len(TIPOS))]
        if bitacora.hecho(fila["id"]):
            continue
        conversacion = [{"role": "user", "content": PROMPT_SINTETICO.format(
            pasaje=fila["texto"][:max_caracteres], tipo=tipo)}]
        texto, _, _ = decoder.completar(decoder.aplicar_plantilla(conversacion), 160)
        crudo, _ = extraer_json(texto)
        pregunta = (crudo or {}).get("pregunta") if isinstance(crudo, dict) else None
        bitacora.agregar({"chunk_id": fila["id"], "unidad_id": fila["unidad_id"], "doc_id": fila["doc_id"],
                          "articulo": fila["articulo"], "tipo": tipo, "pregunta": pregunta,
                          "modelo": getattr(decoder, "nombre", "?")})
        if n % 50 == 0:
            print(f"Sintéticas {n}/{len(filas)}", end="\r")
    return list(bitacora.filas.values())


def control_fuga(sinteticas, muestra, umbral_jaccard=0.5):
    """Descarta consultas sintéticas parecidas a una pregunta de la muestra (5-gramas o Jaccard)."""
    ngr = [(_ngramas(m["pregunta"]), set(_palabras(m["pregunta"]))) for m in muestra]
    limpias, descartadas = [], []
    for s in sinteticas:
        q = s.get("pregunta")
        if not q or len(q) < 20:
            descartadas.append({**s, "motivo": "vacia"})
            continue
        g, p = _ngramas(q), set(_palabras(q))
        peor = max((len(p & mp) / max(1, len(p | mp)) for _, mp in ngr), default=0)
        if any(g & mg for mg, _ in ngr) or peor > umbral_jaccard:
            descartadas.append({**s, "motivo": f"parecida_a_muestra ({peor:.2f})"})
            continue
        limpias.append(s)
    return limpias, descartadas


# ---------------------------------------------------------------- 3. negativos difíciles

def minar(sinteticas, fuentes, R, lexico, negativos=7, saltar=2, semilla=0):
    """Formato FlagEmbedding: {"query", "pos": [..], "neg": [..]}.

    Filtro de ida y vuelta: se conserva la consulta solo si su unidad de origen sale
    en el top 50 del recuperador (si ni el recuperador la encuentra, la consulta
    suele ser mala). Se saltan los 2 primeros candidatos ajenos, que a menudo son
    falsos negativos (otro artículo que también responde).
    """
    from produccion.recuperador import recuperar
    rng = random.Random(semilla)
    ejemplos, sin_ida_vuelta = [], 0
    for s in sinteticas:
        entrada = {"id": s["chunk_id"], "formato": "semi_open", "pregunta": s["pregunta"]}
        ranking = recuperar(entrada, R, fuentes)
        filas = lexico.filas([c for c, _ in ranking[:50]])
        origen = lexico.filas([s["chunk_id"]])[0]
        if origen["unidad_id"] not in {f["unidad_id"] for f in filas}:
            sin_ida_vuelta += 1
            continue
        ajenos = [f for f in filas if f["unidad_id"] != origen["unidad_id"]
                  and not (f["doc_id"] == origen["doc_id"] and f["articulo"] == origen["articulo"])]
        pool = ajenos[saltar:]
        if len(pool) < negativos:
            continue
        elegidos = sorted(rng.sample(range(len(pool)), negativos))
        ejemplos.append({"query": s["pregunta"], "pos": [origen["texto_busqueda"]],
                         "neg": [pool[i]["texto_busqueda"] for i in elegidos],
                         "pos_id": origen["id"], "neg_ids": [pool[i]["id"] for i in elegidos],
                         "doc_id": origen["doc_id"]})
    return ejemplos, {"sin_ida_vuelta": sin_ida_vuelta, "ejemplos": len(ejemplos)}


def dividir(ejemplos, fraccion_val=0.1, semilla=0):
    """Validación separada por documento, para no evaluar con artículos vecinos del entrenamiento."""
    docs = sorted({e["doc_id"] for e in ejemplos})
    random.Random(semilla).shuffle(docs)
    val_docs = set(docs[:max(1, int(len(docs) * fraccion_val))])
    return [e for e in ejemplos if e["doc_id"] not in val_docs], [e for e in ejemplos if e["doc_id"] in val_docs]


# ---------------------------------------------------------------- 4. entrenamiento del reranker

def _mrr_grupos(modelo, tokenizer, ejemplos, max_tokens, torch):
    modelo.eval()
    rr = []
    with torch.inference_mode():
        for e in ejemplos:
            textos = e["pos"][:1] + e["neg"]
            tok = tokenizer([(e["query"], t) for t in textos], padding=True, truncation=True,
                            max_length=max_tokens, return_tensors="pt").to("cuda")
            with torch.autocast("cuda", dtype=torch.bfloat16):
                puntajes = modelo(**tok).logits.float().view(-1)
            rr.append(1.0 / (1 + int((puntajes[1:] > puntajes[0]).sum())))
    return sum(rr) / max(1, len(rr))


def entrenar_reranker(base, entrenamiento, validacion, carpeta, hiper, revision=None):
    """Pérdida listwise (softmax sobre 1 positivo + negativos), como FlagEmbedding."""
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
    torch.manual_seed(hiper["semilla"])
    tokenizer = AutoTokenizer.from_pretrained(base, revision=revision)
    modelo = AutoModelForSequenceClassification.from_pretrained(base, revision=revision).to("cuda")
    modelo.gradient_checkpointing_enable()
    mrr_antes = _mrr_grupos(modelo, tokenizer, validacion, hiper["max_tokens"], torch)
    rng = random.Random(hiper["semilla"])
    datos = list(entrenamiento)
    pasos_totales = hiper["epocas"] * (len(datos) // (hiper["grupos_por_lote"] * hiper["acumulacion"]) or 1)
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=hiper["lr"], weight_decay=0.01)
    programa = get_linear_schedule_with_warmup(optimizador, int(hiper["calentamiento"] * pasos_totales), pasos_totales)
    perdidas, paso = [], 0
    inicio = time.perf_counter()
    for epoca in range(hiper["epocas"]):
        rng.shuffle(datos)
        modelo.train()
        for i in range(0, len(datos), hiper["grupos_por_lote"]):
            lote = datos[i:i + hiper["grupos_por_lote"]]
            pares = [(e["query"], t) for e in lote for t in e["pos"][:1] + e["neg"]]
            tok = tokenizer(pares, padding=True, truncation=True, max_length=hiper["max_tokens"],
                            return_tensors="pt").to("cuda")
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits = modelo(**tok).logits.float().view(len(lote), -1)
            perdida = torch.nn.functional.cross_entropy(logits, torch.zeros(len(lote), dtype=torch.long, device="cuda"))
            (perdida / hiper["acumulacion"]).backward()
            perdidas.append(float(perdida))
            if (i // hiper["grupos_por_lote"] + 1) % hiper["acumulacion"] == 0:
                torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
                optimizador.step()
                programa.step()
                optimizador.zero_grad()
                paso += 1
                if paso % 20 == 0:
                    print(f"época {epoca + 1} paso {paso}/{pasos_totales} pérdida {sum(perdidas[-80:]) / len(perdidas[-80:]):.4f}",
                          end="\r")
    mrr_despues = _mrr_grupos(modelo, tokenizer, validacion, hiper["max_tokens"], torch)
    destino = Path(carpeta) / "modelo"
    modelo.save_pretrained(destino)
    tokenizer.save_pretrained(destino)
    del modelo, optimizador
    torch.cuda.empty_cache()
    return {"mrr_val_antes": mrr_antes, "mrr_val_despues": mrr_despues, "pasos": paso,
            "perdida_final": sum(perdidas[-50:]) / max(1, len(perdidas[-50:])),
            "minutos": round((time.perf_counter() - inicio) / 60, 1), "carpeta_modelo": str(destino)}


def registrar_modelo(salida, version, carpeta_modelo, base, datos_clave, hiper, metricas):
    ruta = Path(salida) / "E09_finetune" / "modelos.json"
    registro = leer_json(ruta, {})
    registro[version] = {"carpeta": str(carpeta_modelo), "base": base, "datos": datos_clave, "hiper": hiper,
                         "metricas": metricas, "fecha": datetime.now().isoformat(timespec="seconds")}
    escribir_json(ruta, registro)
    return registro


# ---------------------------------------------------------------- 5. decoder LoRA (experimental)

def exportar_sft(ctx, decoder, sinteticas, fuentes, R, P, politica_cfg, carpeta, maximo=800):
    """Ejemplos (prompt v04 con pasajes recuperados -> JSON) generados por el propio decoder base.

    Solo se conservan salidas con JSON válido que citan la norma de la unidad de origen:
    es autodestilación filtrada. Enseña formato y estilo de citación, no conocimiento nuevo.
    """
    from produccion import politica
    from produccion.decoder import generar
    from produccion.recuperador import Selector, recuperar
    selector = Selector(ctx.lexico, ctx.evidencia)
    bitacora = Bitacora(Path(carpeta) / "sft.jsonl", campo="chunk_id")
    for s in sinteticas[:maximo]:
        if bitacora.hecho(s["chunk_id"]):
            continue
        entrada = {"id": s["chunk_id"], "formato": "semi_open", "pregunta": s["pregunta"], "area": ""}
        pasajes = selector.seleccionar(entrada, recuperar(entrada, R, fuentes), P)
        crudo, usados, conversacion, registro = generar(decoder, entrada, pasajes, ctx.evidencia,
                                                        ctx.grilla.D(ctx.grilla.base["D"]))
        fila = {"chunk_id": s["chunk_id"], "valido": False}
        if crudo:
            respuesta, _ = politica.postprocesar(entrada, crudo, usados, ctx.evidencia, politica_cfg)
            origen = ctx.evidencia.identidad(s["doc_id"])
            citadas = ctx.citas.bodies(ctx.citas.extract(respuesta["respuesta"] + " " + respuesta["referencia_legal"]))
            if origen & citadas:
                objetivo = {k: respuesta[k] for k in ("respuesta", "palabras_clave", "referencia_legal")}
                fila = {"chunk_id": s["chunk_id"], "valido": True, "mensajes": conversacion,
                        "objetivo": json.dumps(objetivo, ensure_ascii=False)}
        bitacora.agregar(fila)
    return [f for f in bitacora.filas.values() if f["valido"]]


def entrenar_lora(modelo_base, ejemplos, carpeta, lr=1e-4, epocas=1, r=16, alfa=32, acumulacion=8, max_tokens=6144):
    """LoRA sobre las capas de atención. Requiere peft. Pérdida solo sobre la respuesta."""
    import torch
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(modelo_base)
    modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.bfloat16, device_map="cuda")
    modelo.gradient_checkpointing_enable()
    modelo.enable_input_require_grads()
    modelo = get_peft_model(modelo, LoraConfig(r=r, lora_alpha=alfa, lora_dropout=0.05, task_type="CAUSAL_LM",
                                               target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    optimizador = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=lr)
    modelo.train()
    paso = 0
    for _ in range(epocas):
        for i, e in enumerate(ejemplos):
            prompt = tokenizer.apply_chat_template(e["mensajes"], tokenize=False, add_generation_prompt=True)
            ids_prompt = tokenizer(prompt, add_special_tokens=False)["input_ids"]
            ids_obj = tokenizer(e["objetivo"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
            ids = (ids_prompt + ids_obj)[-max_tokens:]
            etiquetas = ([-100] * len(ids_prompt) + ids_obj)[-max_tokens:]
            salida = modelo(input_ids=torch.tensor([ids], device="cuda"), labels=torch.tensor([etiquetas], device="cuda"))
            (salida.loss / acumulacion).backward()
            if (i + 1) % acumulacion == 0:
                optimizador.step()
                optimizador.zero_grad()
                paso += 1
    destino = Path(carpeta) / "adaptador"
    modelo.save_pretrained(destino)
    return {"pasos": paso, "adaptador": str(destino)}


# ---------------------------------------------------------------- orquestación desde el notebook

def e09_reranker(ctx, version=None, forzar=False):
    """Sintéticas -> control de fuga -> negativos -> entrenamiento -> registro de ft:<version>."""
    from produccion.decoder import cargar as cargar_decoder
    from produccion.configuracion import revision
    from produccion.modelos import cargar_encoder
    from produccion.recuperador import Fuentes, IndiceDenso
    from experimentos.e07.pipeline import nodo_busquedas  # noqa: F401  (asegura E03 importable)
    cfg = ctx.grilla.datos["finetune"]
    esp_d = ctx.grilla.D(ctx.grilla.base["D"])

    # 1-2. Sintéticas (dependen del decoder base y de la muestra de unidades)
    nodo_s = ctx.espacio.nodo("E09_finetune", "sinteticas",
                              {"lexico": ctx.nodo_lexico().clave, "n": cfg["unidades_sinteticas"], "decoder": esp_d,
                               "prompt": hashlib.sha256(PROMPT_SINTETICO.encode()).hexdigest()[:8]})

    def correr_sinteticas(carpeta):
        filas = muestrear_unidades(ctx.lexico, cfg["unidades_sinteticas"])
        decoder = cargar_decoder(esp_d, revision(ctx.config, esp_d["modelo"]))
        try:
            sinteticas = generar_sinteticas(decoder, filas, carpeta)
        finally:
            decoder.cerrar()
        muestra = leer_jsonl_muestra(ctx)
        limpias, descartadas = control_fuga(sinteticas, muestra)
        escribir_jsonl(carpeta / "limpias.jsonl", limpias)
        escribir_jsonl(carpeta / "descartadas.jsonl", descartadas)
        return {"generadas": len(sinteticas), "limpias": len(limpias), "descartadas": len(descartadas)}
    ctx.espacio.ejecutar(nodo_s, correr_sinteticas)
    if not nodo_s.listo():
        return None

    # 3. Negativos difíciles con el recuperador de minado
    R = ctx.grilla.R(cfg["recuperacion_minado"])
    nodo_m = ctx.espacio.nodo("E09_finetune", "minado", {"sinteticas": nodo_s.clave, "R": R, "neg": cfg["negativos"]})

    def correr_minado(carpeta):
        sinteticas = leer_jsonl(nodo_s.buscar() / "limpias.jsonl")
        encoders, indices_d = {}, {}
        for e in R["densos"]:
            encoders[e] = cargar_encoder(e, revision(ctx.config, e))
            indices_d[e] = IndiceDenso(ctx.ruta_faiss(e))
        try:
            fuentes = Fuentes(ctx.lexico, encoders, indices_d, cache={})
            ejemplos, info = minar(sinteticas, fuentes, R, ctx.lexico, cfg["negativos"])
        finally:
            for m in encoders.values():
                m.cerrar()
        entrenamiento, validacion = dividir(ejemplos)
        escribir_jsonl(carpeta / "entrenamiento.jsonl", entrenamiento)
        escribir_jsonl(carpeta / "validacion.jsonl", validacion)
        return {**info, "entrenamiento": len(entrenamiento), "validacion": len(validacion)}
    ctx.espacio.ejecutar(nodo_m, correr_minado)
    if not nodo_m.listo():
        return None

    # 4. Entrenamiento y registro
    hiper = cfg["hiper_reranker"]
    nodo_t = ctx.espacio.nodo("E09_finetune", "reranker", {"datos": nodo_m.clave, "base": cfg["reranker_base"],
                                                           "hiper": hiper, "version": version})

    def correr_entrenamiento(carpeta):
        entrenamiento = leer_jsonl(nodo_m.buscar() / "entrenamiento.jsonl")
        validacion = leer_jsonl(nodo_m.buscar() / "validacion.jsonl")
        if ctx.simulado:
            (carpeta / "modelo").mkdir(parents=True, exist_ok=True)
            metricas = {"mrr_val_antes": 0.0, "mrr_val_despues": 0.0, "simulado": True,
                        "carpeta_modelo": str(carpeta / "modelo")}
        else:
            metricas = entrenar_reranker(cfg["reranker_base"], entrenamiento, validacion, carpeta, hiper,
                                         revision(ctx.config, cfg["reranker_base"]))
        nombre = version or "v" + datetime.now().strftime("%m%d_%H%M")
        registrar_modelo(ctx.espacio.salida, nombre, metricas["carpeta_modelo"], cfg["reranker_base"],
                         nodo_m.clave, hiper, metricas)
        return {**metricas, "version": nombre}
    return ctx.espacio.ejecutar(nodo_t, correr_entrenamiento, forzar=forzar)


def leer_jsonl_muestra(ctx):
    return [json.loads(l) for l in Path(ctx.rutas["muestra"]).read_text(encoding="utf-8-sig").splitlines() if l.strip()]


**producto**. E10: tablas, gráficas, decisión y configuración ganadora.

In [ ]:
%%modulo experimentos.e07.producto
"""E10: producto de datos para decidir la arquitectura.

Salidas en <SALIDA>/E10_producto/<fecha>/:
    recuperacion.csv       E04: proxies de recuperación por configuración R
    evidencia.csv          E05: techo de citas por R x P (sin decoder)
    configuraciones.csv    E07/E08: rúbrica, IC95, latencia, VRAM, banderas, por (R, P, D, política)
    por_item.csv           una fila por configuración y pregunta (para análisis fino)
    efectos.csv            efecto de cada nivel frente a la base (OFAT), con IC pareado
    por_area.csv           puntaje determinista por área de las mejores configuraciones
    decision.json          recomendación, alternativas y advertencias
    config_produccion.json configuración lista para correr.py (la ganadora)
    decision.html          reporte autocontenido con tablas y gráficas
"""
import base64
import copy
import io
import json
import time
from pathlib import Path

from experimentos.e07 import rubrica as rub
from experimentos.e07.etapas import escribir_json, leer_json, leer_jsonl
from experimentos.e07.pipeline import (nodo_generacion, nodo_pasajes, nodo_ranking, nodo_respuestas)

AZUL, AZUL_CLARO, ROJO, GRIS, TEXTO, TEXTO_2 = "#2a78d6", "#9ec5f4", "#e34948", "#b9b8b3", "#0b0b0b", "#52514e"


def _estilo():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRIS,
                         "axes.labelcolor": TEXTO_2, "xtick.color": TEXTO_2, "ytick.color": TEXTO_2,
                         "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6, "axes.axisbelow": True,
                         "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "figure.dpi": 110})
    return plt


def _png(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    import matplotlib.pyplot as plt
    plt.close(fig)
    return base64.b64encode(buffer.getvalue()).decode()


def recolectar(ctx):
    import pandas as pd
    g = ctx.grilla
    filas_r = []
    for nombre in g.nombres_R:
        nodo, r = nodo_ranking(ctx, nombre)
        m = nodo.resumen() or {}
        filas_r.append({"R": nombre, "nota": g.datos["recuperacion"][nombre].get("nota", ""),
                        "estado": "ok" if nodo.listo() else ("pendiente_finetune" if r.get("_pendiente") else "falta"),
                        **{k: v for k, v in m.items()}})
    filas_e = []
    for nombre_r in g.nombres_R:
        for nombre_p in g.nombres_P:
            nodo, _ = nodo_pasajes(ctx, nombre_r, nombre_p)
            if nodo.listo():
                filas_e.append({"R": nombre_r, "P": nombre_p, **(nodo.resumen() or {})})
    filas_c, por_item = [], {}
    horas = g.datos["latencia"]
    for r, p, d in g.corridas_generacion():
        nodo_g, _, esp = nodo_generacion(ctx, r, p, d)
        gen = nodo_g.resumen() or {}
        nodo_r, _ = nodo_ranking(ctx, r)
        met_r = nodo_r.resumen() or {}
        nodo_p, _ = nodo_pasajes(ctx, r, p)
        met_p = nodo_p.resumen() or {}
        for pol in g.nombres_politica:
            nodo, _ = nodo_respuestas(ctx, r, p, d, pol)
            if not nodo.listo():
                continue
            res = nodo.resumen() or {}
            clave = f"{r}|{p}|{d}|{pol}"
            por_item[clave] = leer_jsonl(nodo.buscar() / "por_item.jsonl")
            sim = ctx.espacio.nodo("E08_juez", "sim__" + "__".join((r, p, d, pol)), {"respuestas": nodo.clave})
            rag = ctx.espacio.nodo("E08_juez", "ragas__" + "__".join((r, p, d, pol)), {"respuestas": nodo.clave})
            ms_total = (met_r.get("ms_recuperacion_medido", 0) + met_r.get("ms_denso_estimado", 0)
                        + met_p.get("ms_seleccion", 0) + gen.get("ms_media", 0))
            ms_p95 = (met_r.get("ms_recuperacion_medido", 0) + met_r.get("ms_denso_estimado", 0)
                      + met_p.get("ms_seleccion", 0) + gen.get("ms_p95", 0))
            ic = res.get("ic95", {}).get("determinista_50", (None, None))
            fila = {"config": clave, "R": r, "P": p, "D": d, "politica": pol, "modelo": esp["modelo"],
                    "cerradas": res.get("cerradas"), "citas": res.get("citas"), "abstencion": res.get("abstencion"),
                    "determinista_50": res.get("determinista_50"), "ic95_bajo": ic[0], "ic95_alto": ic[1],
                    "cerradas_acc": res.get("cerradas_acc"), "citas_recall": res.get("citas_recall"),
                    "citas_sin_respaldo": res.get("citas_sin_respaldo"), "abstenciones": res.get("abstenciones"),
                    "errores_formato": res.get("errores_formato"), "oraciones_saneadas": res.get("oraciones_saneadas"),
                    "techo_puntos_citas": met_p.get("techo_puntos_citas"),
                    "similitud_local": (sim.resumen() or {}).get("similitud_media"),
                    "ragas_puntos": (rag.resumen() or {}).get("puntos"),
                    "json_invalido": gen.get("json_invalido"), "letra_por_logits": gen.get("letra_por_logits"),
                    "ms_pregunta": ms_total, "ms_p95": ms_p95,
                    "horas_992": ms_total * horas["preguntas_sabado"] / 3.6e6,
                    "vram_decoder_gib": gen.get("vram_pico_gib"), "parametros": gen.get("parametros"),
                    "requiere_confirmacion": bool(g.datos["decoders"][d].get("requiere_confirmacion")),
                    "solo_medicion": bool(g.datos["politicas"][pol].get("solo_medicion")),
                    "usa_finetune": bool(g.datos["recuperacion"][r].get("requiere_finetune"))}
            fila["total_con_ragas"] = (fila["determinista_50"] + fila["ragas_puntos"]
                                       if fila["ragas_puntos"] is not None else None)
            fila["cumple_latencia"] = (ms_p95 <= horas["presupuesto_s_por_pregunta"] * 1000
                                       and fila["horas_992"] <= 0.9 * horas["horas_disponibles"])
            filas_c.append(fila)
    return pd.DataFrame(filas_r), pd.DataFrame(filas_e), pd.DataFrame(filas_c), por_item


def efectos(ctx, configuraciones, por_item):
    """Cambiar un solo factor frente a la base: diferencia en puntos, IC pareado y P(mejor)."""
    import pandas as pd
    b = ctx.grilla.base
    clave_base = f"{b['R']}|{b['P']}|{b['D']}|{b['politica']}"
    if clave_base not in por_item:
        return pd.DataFrame()
    filas = []
    for factor, nombres in (("R", ctx.grilla.nombres_R), ("P", ctx.grilla.nombres_P),
                            ("D", ctx.grilla.nombres_D), ("politica", ctx.grilla.nombres_politica)):
        for nivel in nombres:
            partes = dict(b)
            partes[factor] = nivel
            clave = f"{partes['R']}|{partes['P']}|{partes['D']}|{partes['politica']}"
            if clave not in por_item or clave == clave_base:
                continue
            comparacion = rub.pareado(por_item[clave_base], por_item[clave], n=1000)
            comp_citas = rub.pareado(por_item[clave_base], por_item[clave], n=500, campo="citas")
            comp_cerr = rub.pareado(por_item[clave_base], por_item[clave], n=500, campo="cerradas")
            filas.append({"factor": factor, "nivel": nivel, "config": clave,
                          "delta_determinista": comparacion["diferencia"],
                          "ic95_bajo": comparacion["ic95"][0], "ic95_alto": comparacion["ic95"][1],
                          "p_mejor": comparacion["p_mejor"], "delta_citas": comp_citas["diferencia"],
                          "delta_cerradas": comp_cerr["diferencia"]})
    return pd.DataFrame(filas)


def por_area(configuraciones, por_item, top=6):
    import pandas as pd
    if configuraciones.empty:
        return pd.DataFrame()
    elegidas = list(configuraciones.sort_values("determinista_50", ascending=False)["config"].head(top))
    filas = []
    for clave in elegidas:
        items = por_item[clave]
        for area in sorted({i["area"] for i in items}):
            sub = [i for i in items if i["area"] == area]
            filas.append({"config": clave, "area": area.split(" [")[0], "n": len(sub),
                          "determinista_50": rub.Rubrica.agregar(sub)["determinista_50"]})
    return pd.DataFrame(filas).pivot(index="area", columns="config", values="determinista_50")


def decidir(ctx, configuraciones, efectos_df):
    b = ctx.grilla.base
    if configuraciones.empty:
        return {"recomendada": None, "motivo": "No hay configuraciones puntuadas todavía"}
    df = configuraciones.copy()
    elegibles = df[(~df["requiere_confirmacion"]) & (~df["solo_medicion"]) & (df["cumple_latencia"])]
    con_ragas = elegibles["total_con_ragas"].notna().sum()
    criterio = "total_con_ragas" if con_ragas >= 2 else "determinista_50"
    orden = elegibles.sort_values([criterio, "ms_pregunta"], ascending=[False, True]) if criterio == "determinista_50" \
        else elegibles[elegibles["total_con_ragas"].notna()].sort_values([criterio, "ms_pregunta"], ascending=[False, True])
    if orden.empty:
        return {"recomendada": None, "motivo": "Ninguna configuración cumple latencia y reglas"}
    mejor = orden.iloc[0].to_dict()
    base_clave = f"{b['R']}|{b['P']}|{b['D']}|{b['politica']}"
    base = df[df["config"] == base_clave]
    advertencias = []
    if criterio == "determinista_50":
        advertencias.append("Sin juez RAGAS para comparar: la decisión ignora 30 de los 80 puntos. "
                            "Correr E08 con la llave antes de congelar.")
    if not base.empty and mejor["config"] != base_clave:
        solapan = mejor["ic95_bajo"] is not None and base.iloc[0]["determinista_50"] >= mejor["ic95_bajo"]
        if solapan:
            advertencias.append("El IC95 de la recomendada incluye el puntaje de la base: con 50 preguntas "
                                "la ventaja no es concluyente. Preferir cambios con efecto positivo en citas "
                                "(más estable que cerradas).")
    if "P_" in mejor["P"] and ctx.grilla.P(mejor["P"]).get("cabecera_normalizada"):
        advertencias.append("La evidencia usa cabeceras normalizadas (sin offsets). Verificar con la organización "
                            "que la verificación en vivo las acepte o usar P_literal.")
    if ctx.grilla.datos["politicas"][mejor["politica"]].get("saneo") == "codigos":
        advertencias.append("La política 'codigos' deja citas a códigos sin pasaje: el paso 4 exige que toda norma "
                            "citada provenga de un pasaje. Confirmar antes de usarla.")
    mejores_efectos = []
    if not efectos_df.empty:
        positivos = efectos_df[(efectos_df["delta_determinista"] > 0) & (efectos_df["p_mejor"] >= 0.8)]
        mejores_efectos = positivos.sort_values("delta_determinista", ascending=False).head(8).to_dict("records")
    return {"criterio": criterio, "recomendada": mejor, "base": base.iloc[0].to_dict() if not base.empty else None,
            "alternativas": orden.head(5).to_dict("records"), "efectos_confiables": mejores_efectos,
            "advertencias": advertencias, "configuraciones_elegibles": int(len(elegibles)),
            "configuraciones_puntuadas": int(len(df))}


def config_produccion(ctx, decision):
    if not decision.get("recomendada"):
        return None
    m = decision["recomendada"]
    salida = copy.deepcopy(ctx.config_original)
    r = {k: v for k, v in ctx.R(m["R"]).items() if not k.startswith("_")}
    salida["recuperacion"] = r
    salida["pasajes"] = ctx.grilla.P(m["P"])
    salida["decoder"] = {**salida["decoder"], **ctx.grilla.D(m["D"])}
    salida["politica"] = ctx.grilla.politica(m["politica"])
    if str(r.get("reranker", "")).startswith("ft:"):
        _, carpeta = ctx.resolver_modelo(r["reranker"])
        salida["nota_reranker"] = f"Copiar {carpeta} a indice/modelos/{r['reranker'][3:]} antes de correr"
    salida["origen_e07"] = {"config": m["config"], "determinista_50": m["determinista_50"],
                            "fecha": time.strftime("%Y-%m-%d %H:%M")}
    return salida


def graficas(recuperacion, evidencia, configuraciones, efectos_df, base_clave):
    plt = _estilo()
    imagenes = {}
    if not recuperacion.empty and "respaldo_literal_top10" in recuperacion:
        df = recuperacion.dropna(subset=["respaldo_literal_top10"]).sort_values("respaldo_literal_top10")
        fig, ax = plt.subplots(figsize=(6.5, 0.28 * len(df) + 1))
        ax.barh(df["R"], df["respaldo_literal_top10"], color=AZUL, height=0.6)
        for y, v in enumerate(df["respaldo_literal_top10"]):
            ax.text(v + 0.005, y, f"{v:.3f}", va="center", color=TEXTO_2, fontsize=8)
        ax.set_xlabel("Respaldo literal top 10 (proxy e06)")
        ax.set_title("Recuperación: ¿aparece alguna norma del fundamento en los 10 primeros chunks?")
        ax.set_xlim(0, 1.08)
        imagenes["recuperacion"] = _png(fig)
    if not evidencia.empty and "techo_puntos_citas" in evidencia:
        tabla = evidencia.pivot(index="R", columns="P", values="techo_puntos_citas")
        fig, ax = plt.subplots(figsize=(1.0 * len(tabla.columns) + 2, 0.32 * len(tabla) + 1.2))
        im = ax.imshow(tabla.values, cmap="Blues", aspect="auto", vmin=0, vmax=20)
        ax.set_xticks(range(len(tabla.columns)), tabla.columns, rotation=30, ha="right")
        ax.set_yticks(range(len(tabla.index)), tabla.index)
        ax.grid(False)
        for i in range(tabla.shape[0]):
            for j in range(tabla.shape[1]):
                v = tabla.values[i, j]
                if v == v:
                    ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7,
                            color="white" if v > 12 else TEXTO)
        fig.colorbar(im, ax=ax, label="puntos de citas alcanzables (de 20)")
        ax.set_title("Techo de citas de la evidencia (R x P, sin decoder)")
        imagenes["evidencia"] = _png(fig)
    if not configuraciones.empty:
        df = configuraciones.sort_values("determinista_50", ascending=False).head(25).iloc[::-1]
        fig, ax = plt.subplots(figsize=(7, 0.28 * len(df) + 1))
        colores = [AZUL if c == base_clave else GRIS for c in df["config"]]
        errores = [df["determinista_50"] - df["ic95_bajo"], df["ic95_alto"] - df["determinista_50"]]
        ax.barh(df["config"], df["determinista_50"], color=colores, height=0.6, xerr=errores,
                error_kw={"ecolor": TEXTO_2, "elinewidth": 0.8, "capsize": 2})
        ax.set_xlabel("Puntos deterministas (cerradas + citas + abstención, de 50) con IC95")
        ax.set_title("Mejores configuraciones (la base en azul)")
        ax.tick_params(axis="y", labelsize=7)
        imagenes["configuraciones"] = _png(fig)
        fig, ax = plt.subplots(figsize=(6.5, 4))
        elegibles = configuraciones[~configuraciones["solo_medicion"]]
        ax.scatter(elegibles["horas_992"], elegibles["determinista_50"], s=22, color=AZUL, alpha=0.75,
                   edgecolors="white", linewidths=0.8)
        for _, fila in elegibles.sort_values("determinista_50", ascending=False).head(5).iterrows():
            ax.annotate(fila["config"], (fila["horas_992"], fila["determinista_50"]), fontsize=6.5,
                        color=TEXTO_2, xytext=(4, 2), textcoords="offset points")
        ax.axvline(5.4, color=ROJO, linewidth=1, linestyle="--")
        ax.text(5.45, ax.get_ylim()[0], "límite práctico (90% de 6 h)", color=ROJO, fontsize=7, va="bottom")
        ax.set_xlabel("Horas estimadas para 992 preguntas")
        ax.set_ylabel("Puntos deterministas (de 50)")
        ax.set_title("Puntaje frente a tiempo de ejecución")
        imagenes["latencia"] = _png(fig)
    if not efectos_df.empty:
        df = efectos_df.sort_values("delta_determinista")
        fig, ax = plt.subplots(figsize=(6.5, 0.3 * len(df) + 1))
        colores = [AZUL if v >= 0 else ROJO for v in df["delta_determinista"]]
        ax.barh(df["factor"] + ": " + df["nivel"], df["delta_determinista"], color=colores, height=0.6,
                xerr=[df["delta_determinista"] - df["ic95_bajo"], df["ic95_alto"] - df["delta_determinista"]],
                error_kw={"ecolor": TEXTO_2, "elinewidth": 0.8, "capsize": 2})
        ax.axvline(0, color=TEXTO_2, linewidth=0.8)
        ax.set_xlabel("Diferencia frente a la base (puntos, IC95 pareado)")
        ax.set_title("Efecto de cambiar un solo factor")
        imagenes["efectos"] = _png(fig)
    return imagenes


ALT = {"recuperacion": "Proxy de recuperación por configuración",
       "evidencia": "Techo de citas por recuperación y pasajes", "configuraciones": "Mejores configuraciones",
       "latencia": "Puntaje frente a horas de ejecución", "efectos": "Efecto de cada factor frente a la base"}

HTML = """<!doctype html><html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1"><title>Decisión de arquitectura e07</title>
<style>
:root{{--fondo:#fcfcfb;--texto:#0b0b0b;--texto2:#52514e;--borde:#e6e5e1;--acento:#2a78d6;--alerta:#e34948}}
@media (prefers-color-scheme: dark){{:root{{--fondo:#1a1a19;--texto:#fff;--texto2:#c3c2b7;--borde:#383835;--acento:#3987e5;--alerta:#e66767}}
img{{background:#fcfcfb;border-radius:6px}}}}
body{{background:var(--fondo);color:var(--texto);font:14px/1.5 system-ui,-apple-system,Segoe UI,sans-serif;margin:0;padding:24px 16px}}
main{{max-width:1100px;margin:0 auto}} h1{{font-size:22px}} h2{{font-size:17px;margin-top:32px;border-bottom:1px solid var(--borde);padding-bottom:4px}}
.nota{{color:var(--texto2)}} .caja{{border:1px solid var(--borde);border-radius:8px;padding:12px 16px;margin:12px 0}}
.alerta{{border-color:var(--alerta)}} table{{border-collapse:collapse;font-size:12px;display:block;overflow-x:auto}}
th,td{{border-bottom:1px solid var(--borde);padding:4px 8px;text-align:right;white-space:nowrap}} th{{color:var(--texto2)}}
td:first-child,th:first-child{{text-align:left}} img{{max-width:100%}} code{{font-size:12px}}
</style></head><body><main>
<h1>Decisión de arquitectura (e07)</h1>
<p class="nota">Generado {fecha}. {n_conf} configuraciones puntuadas sobre las 50 preguntas de muestra. Criterio: <b>{criterio}</b>.</p>
<div class="caja"><b>Recomendada:</b> <code>{recomendada}</code><br>{detalle}</div>
{advertencias}
<h2>Recuperación (E04)</h2>{img_recuperacion}{tabla_r}
<h2>Techo de citas de la evidencia (E05)</h2><p class="nota">Puntos de citas que se obtendrían citando todas las normas respaldadas. No depende del decoder.</p>
{img_evidencia}
<h2>Configuraciones completas (E07, E08)</h2>{img_configuraciones}
{img_latencia}{tabla_c}
<h2>Efecto de cada factor</h2>{img_efectos}{tabla_ef}
<h2>Por área</h2>{tabla_area}
</main></body></html>"""


def generar(ctx):
    import pandas as pd
    carpeta = ctx.espacio.salida / "E10_producto" / time.strftime("%Y%m%d_%H%M%S")
    carpeta.mkdir(parents=True, exist_ok=True)
    recuperacion, evidencia, configuraciones, por_item = recolectar(ctx)
    ef = efectos(ctx, configuraciones, por_item)
    area = por_area(configuraciones, por_item)
    decision = decidir(ctx, configuraciones, ef)
    produccion = config_produccion(ctx, decision)
    recuperacion.to_csv(carpeta / "recuperacion.csv", index=False)
    evidencia.to_csv(carpeta / "evidencia.csv", index=False)
    configuraciones.to_csv(carpeta / "configuraciones.csv", index=False)
    ef.to_csv(carpeta / "efectos.csv", index=False)
    area.to_csv(carpeta / "por_area.csv")
    pd.DataFrame([{"config": k, **{c: v for c, v in i.items() if c != "detalle_citas"}}
                  for k, items in por_item.items() for i in items]).to_csv(carpeta / "por_item.csv", index=False)
    escribir_json(carpeta / "decision.json", decision)
    if produccion:
        escribir_json(carpeta / "config_produccion.json", produccion)
    b = ctx.grilla.base
    imagenes = graficas(recuperacion, evidencia, configuraciones, ef, f"{b['R']}|{b['P']}|{b['D']}|{b['politica']}")
    rec = decision.get("recomendada") or {}
    detalle = (f"{rec.get('determinista_50', 0):.2f} / 50 deterministas (IC95 {rec.get('ic95_bajo') or 0:.1f}-"
               f"{rec.get('ic95_alto') or 0:.1f}); RAGAS: {rec.get('ragas_puntos') if rec.get('ragas_puntos') is not None else 'sin juzgar'}; "
               f"{rec.get('ms_pregunta', 0) / 1000:.1f} s por pregunta, {rec.get('horas_992', 0):.2f} h para 992."
               if rec else decision.get("motivo", ""))
    columnas_c = ["config", "cerradas", "citas", "abstencion", "determinista_50", "ic95_bajo", "ic95_alto",
                  "ragas_puntos", "similitud_local", "techo_puntos_citas", "ms_pregunta", "horas_992",
                  "json_invalido", "cumple_latencia", "requiere_confirmacion"]
    fmt = {"float_format": lambda x: f"{x:.3f}", "index": False, "border": 0}
    vacio = ""
    html = HTML.format(
        fecha=time.strftime("%Y-%m-%d %H:%M"), n_conf=len(configuraciones), criterio=decision.get("criterio", "-"),
        recomendada=rec.get("config", "ninguna"), detalle=detalle,
        advertencias="".join(f'<div class="caja alerta">{a}</div>' for a in decision.get("advertencias", [])),
        **{f"img_{k}": (f'<img src="data:image/png;base64,{imagenes[k]}" alt="{ALT[k]}">' if k in imagenes else vacio)
           for k in ALT},
        tabla_r=recuperacion.to_html(**fmt) if not recuperacion.empty else "",
        tabla_c=(configuraciones.sort_values("determinista_50", ascending=False)[
            [c for c in columnas_c if c in configuraciones]].head(40).to_html(**fmt)) if not configuraciones.empty else "",
        tabla_ef=ef.to_html(**fmt) if not ef.empty else "<p class='nota'>Falta la base o no hay variantes.</p>",
        tabla_area=area.to_html(float_format=lambda x: f"{x:.1f}", border=0) if not area.empty else "")
    (carpeta / "decision.html").write_text(html, encoding="utf-8")
    return carpeta, decision, configuraciones, ef, area


**final**. Corrida del sábado con todos los modelos cargados.

In [ ]:
%%modulo experimentos.e07.final
"""Corrida final: responde un archivo de preguntas (test_992) con una configuración de la grilla.

Carga todos los modelos juntos, como el sábado. Guarda cada respuesta al terminarla
(se reanuda) y escribe submission.jsonl en el orden de la entrada.
"""
import json
import time
from pathlib import Path

from experimentos.e07.etapas import Bitacora, escribir_json, escribir_jsonl
from experimentos.e07.pipeline import identidades
from produccion import politica
from produccion.configuracion import leer_jsonl, preparar_entrada, revision
from produccion.decoder import cargar as cargar_decoder, generar
from produccion.modelos import cargar_encoder, cargar_reranker
from produccion.recuperador import Fuentes, IndiceDenso, Selector, pasajes_salida, recuperar


def responder_archivo(ctx, eleccion, ruta_entrada, carpeta, limite=None):
    R = {k: v for k, v in ctx.R(eleccion["R"]).items() if not k.startswith("_")}
    P, D = ctx.grilla.P(eleccion["P"]), ctx.grilla.D(eleccion["D"])
    pol = ctx.grilla.politica(eleccion["politica"])
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    escribir_json(carpeta / "configuracion.json", {"eleccion": eleccion, "R": R, "P": P, "D": D, "politica": pol})
    encoders, indices_d = {}, {}
    for e in R["densos"]:
        encoders[e] = cargar_encoder(e, revision(ctx.config, e))
        indices_d[e] = IndiceDenso(ctx.ruta_faiss(e))
    rerankers = {}
    if R.get("reranker"):
        _, local = ctx.resolver_modelo(R["reranker"])
        base = ctx.grilla.datos["finetune"]["reranker_base"] if local else R["reranker"]
        rerankers[R["reranker"]] = cargar_reranker(base, revision(ctx.config, base), 1024, 16, local)
    fuentes = Fuentes(ctx.lexico, encoders, indices_d, rerankers)
    ident = identidades(ctx) if (P.get("router_articulo") or P.get("anclas_area")) else None
    selector = Selector(ctx.lexico, ctx.evidencia, ident, ctx.anclas)
    decoder = cargar_decoder(D, revision(ctx.config, D["modelo"]))
    entradas = [preparar_entrada(x) for x in leer_jsonl(ruta_entrada)][:limite]
    bitacora = Bitacora(carpeta / "detalles.jsonl")
    inicio_total = time.perf_counter()
    for n, e in enumerate(entradas, 1):
        if bitacora.hecho(e["id"]):
            continue
        t = time.perf_counter()
        letras = list((e.get("opciones") or {}).keys())
        detalle = {"id": e["id"]}
        try:
            ranking = recuperar(e, R, fuentes, ctx.anclas)
            pasajes = selector.seleccionar(e, ranking, P)
            crudo, usados, conversacion, registro = generar(decoder, e, pasajes, ctx.evidencia, D)
            if e["formato"] == "multiple_choice" and (crudo or {}).get("respuesta_correcta") not in letras:
                letra, _ = decoder.elegir_letra(conversacion, letras)
                crudo = dict(crudo or {}, respuesta_correcta=letra)
                detalle["letra_por_logits"] = letra
            respuesta, _ = politica.postprocesar(e, crudo, usados, ctx.evidencia, pol)
            respuesta["pasajes_recuperados"] = pasajes_salida(respuesta["pasajes_recuperados"])
            detalle["json_valido"] = registro.get("json_valido")
        except Exception as error:
            detalle["error"] = f"{type(error).__name__}: {error}"
            respuesta = {"id": e["id"], **politica.abstencion(e["formato"], [])}
        respuesta["latencia_ms"] = round((time.perf_counter() - t) * 1000)
        bitacora.agregar({**detalle, "salida": respuesta})
        media = (time.perf_counter() - inicio_total) / n
        print(f"{n}/{len(entradas)} id={e['id']} {respuesta['latencia_ms']} ms | faltan ~{media * (len(entradas) - n) / 60:.0f} min",
              end="\r")
    salida = [bitacora.filas[e["id"]]["salida"] for e in entradas if bitacora.hecho(e["id"])]
    escribir_jsonl(carpeta / "submission.jsonl", salida)
    return carpeta / "submission.jsonl"


## 2. Grilla
Todas las configuraciones. Se editan aquí. La base es la R03 del e06 con la generación v04.

In [ ]:
GRILLA = {
 "base": {
  "R": "R03",
  "P": "P_v04",
  "D": "qwen25_7b",
  "politica": "v04_cita"
 },
 "recuperacion": {
  "R00": {
   "densos": [],
   "reranker": null,
   "nota": "BM25 solo (e06)"
  },
  "R01": {
   "bm25": false,
   "reranker": null,
   "nota": "BGE-M3 denso solo (e06)"
  },
  "R02": {
   "reranker": null,
   "nota": "RRF BM25 + BGE-M3 (e06)"
  },
  "R03": {
   "nota": "R02 + bge-reranker-v2-m3 (e06, base)"
  },
  "R08": {
   "consulta": "pregunta",
   "nota": "R03 sin opciones en la consulta"
  },
  "R09": {
   "consulta": "por_opcion",
   "nota": "R03 con una consulta por opción en cerradas"
  },
  "R10": {
   "expansion_area": 2,
   "nota": "R03 + nombres de las 2 normas ancla del área en BM25"
  },
  "R12": {
   "top_rerank": 100,
   "nota": "R03 reordenando 100 candidatos"
  },
  "R04h": {
   "densos": [
    "intfloat/multilingual-e5-large"
   ],
   "reranker": null,
   "nota": "RRF BM25 + E5 (e06)"
  },
  "R04": {
   "densos": [
    "intfloat/multilingual-e5-large"
   ],
   "nota": "R04h + bge-reranker (e06)"
  },
  "R05h": {
   "densos": [
    "Qwen/Qwen3-Embedding-0.6B"
   ],
   "reranker": null,
   "nota": "RRF BM25 + Qwen3-Emb-0.6B"
  },
  "R05": {
   "densos": [
    "Qwen/Qwen3-Embedding-0.6B"
   ],
   "nota": "R05h + bge-reranker (e06)"
  },
  "R07": {
   "reranker": "Qwen/Qwen3-Reranker-0.6B",
   "nota": "R02 + Qwen3-Reranker-0.6B (e06)"
  },
  "R11": {
   "densos": [
    "BAAI/bge-m3",
    "intfloat/multilingual-e5-large",
    "Qwen/Qwen3-Embedding-0.6B"
   ],
   "nota": "RRF de BM25 y tres encoders + bge-reranker"
  },
  "R13": {
   "reranker": "ft:ultimo",
   "nota": "R03 con el reranker afinado (E09)",
   "requiere_finetune": true
  },
  "R14": {
   "consulta": "por_opcion",
   "expansion_area": 2,
   "reranker": "ft:ultimo",
   "nota": "R09 + R10 + reranker afinado",
   "requiere_finetune": true
  },
  "R06h": {
   "densos": [
    "Qwen/Qwen3-Embedding-4B"
   ],
   "reranker": null,
   "nota": "RRF BM25 + Qwen3-Emb-4B",
   "costo": "alto"
  },
  "R06": {
   "densos": [
    "Qwen/Qwen3-Embedding-4B"
   ],
   "nota": "R06h + bge-reranker (e06)",
   "costo": "alto"
  }
 },
 "pasajes": {
  "P_e06": {
   "modo": "e06",
   "nota": "Réplica de passages() del e06"
  },
  "P_v04": {
   "nota": "Cabecera para cada norma, hasta 8 unidades (base)"
  },
  "P_router": {
   "router_articulo": true,
   "nota": "Artículo nombrado en la pregunta, búsqueda exacta"
  },
  "P_anclas2": {
   "anclas_area": 2,
   "nota": "Garantiza evidencia de las 2 normas más citadas del área"
  },
  "P_div2": {
   "max_por_doc": 2,
   "nota": "Máximo 2 unidades por documento"
  },
  "P_u5": {
   "max_unidades": 5,
   "nota": "Menos unidades, prompt más corto"
  },
  "P_literal": {
   "cabecera_normalizada": false,
   "nota": "Solo cabeceras literales (más seguro en la verificación en vivo)"
  },
  "P_todo": {
   "router_articulo": true,
   "anclas_area": 2,
   "max_por_doc": 2,
   "nota": "Router + anclas + diversidad"
  }
 },
 "decoders": {
  "qwen25_7b": {
   "modelo": "Qwen/Qwen2.5-7B-Instruct"
  },
  "qwen3_4b": {
   "modelo": "Qwen/Qwen3-4B-Instruct-2507"
  },
  "salamandra_7b": {
   "modelo": "BSC-LT/salamandra-7b-instruct",
   "nota": "Sugerido en el enunciado, entrenado en español"
  },
  "qwen35_4b": {
   "modelo": "Qwen/Qwen3.5-4B",
   "clase": "multimodal",
   "nota": "Necesita transformers >= 5.3 y torchvision"
  },
  "qwen3_4b_thinking": {
   "modelo": "Qwen/Qwen3-4B-Thinking-2507",
   "pensar": true,
   "max_tokens": {
    "multiple_choice": 1600,
    "semi_open": 1600,
    "open_ended": 2000
   },
   "max_segundos": 20,
   "nota": "Razona antes de responder; vigilar la latencia"
  },
  "qwen3_8b": {
   "modelo": "Qwen/Qwen3-8B",
   "requiere_confirmacion": true,
   "nota": "8,2B parámetros: sugerido en el enunciado pero supera 8.000M; confirmar con la organización"
  },
  "llama31_8b": {
   "modelo": "meta-llama/Llama-3.1-8B-Instruct",
   "requiere_confirmacion": true,
   "nota": "8,03B parámetros y acceso restringido en Hugging Face; confirmar con la organización"
  }
 },
 "decoder_comun": {
  "backend": "hf",
  "clase": "causal",
  "pensar": false,
  "contexto": 8192,
  "max_tokens": {
   "multiple_choice": 500,
   "semi_open": 500,
   "open_ended": 800
  },
  "max_segundos": 16,
  "repeat_penalty": 1.1,
  "repeat_last_n": 256,
  "longitudes": true,
  "max_caracteres_pasaje": 1800
 },
 "politicas": {
  "v04_cita": {
   "citar_evidencia": "todas",
   "abstener_libre": "sin_evidencia",
   "saneo": "cita"
  },
  "v04_oracion": {
   "citar_evidencia": "todas",
   "abstener_libre": "sin_evidencia",
   "saneo": "oracion"
  },
  "usadas_cita": {
   "citar_evidencia": "usadas",
   "abstener_libre": "sin_evidencia",
   "saneo": "cita"
  },
  "codigos": {
   "citar_evidencia": "todas",
   "abstener_libre": "sin_evidencia",
   "saneo": "codigos",
   "nota": "Tolera códigos sin respaldo: medir antes de adoptar (ver SPEC, regla del paso 4)"
  },
  "sin_saneo": {
   "citar_evidencia": "todas",
   "abstener_libre": "sin_evidencia",
   "saneo": "ninguno",
   "solo_medicion": true,
   "nota": "Incumple el paso 4: solo mide cuánto aporta el saneo"
  },
  "nunca_abstener": {
   "citar_evidencia": "todas",
   "abstener_libre": "nunca",
   "saneo": "cita"
  }
 },
 "generacion": {
  "diseno": [
   "base",
   "ofat_R",
   "ofat_P",
   "ofat_D",
   "factorial"
  ],
  "factorial": {
   "R": [
    "R03",
    "R09",
    "R11",
    "R14"
   ],
   "P": [
    "P_v04",
    "P_todo"
   ],
   "D": [
    "qwen25_7b",
    "qwen3_4b",
    "salamandra_7b"
   ]
  },
  "completo": false,
  "nota_completo": "true agrega al final todas las combinaciones R x P x D restantes (cientos de horas; solo si sobra tiempo)"
 },
 "juez": {
  "max_configuraciones": 6,
  "siempre": [
   "base"
  ],
  "nota": "El juez RAGAS consume la llave: se juzgan la base y las mejores por puntaje determinista, con caché por respuesta"
 },
 "finetune": {
  "unidades_sinteticas": 3000,
  "negativos": 7,
  "reranker_base": "BAAI/bge-reranker-v2-m3",
  "recuperacion_minado": "R02",
  "hiper_reranker": {
   "lr": 1e-05,
   "epocas": 1,
   "grupos_por_lote": 4,
   "acumulacion": 4,
   "max_tokens": 512,
   "calentamiento": 0.1,
   "semilla": 0
  }
 },
 "latencia": {
  "presupuesto_s_por_pregunta": 20,
  "preguntas_sabado": 992,
  "horas_disponibles": 6
 }
}

In [ ]:
import json
import pandas as pd
from IPython.display import display, HTML
from experimentos.e07 import pipeline as pl, producto, finetune, final

CONFIG = {"datos": str(DATOS), "entradas": ENTRADAS, "lote_rerank": 16}
ctx = pl.Contexto(CONFIG, GRILLA, SALIDA, reutilizar=REUTILIZAR, desde_etapa=DESDE_ETAPA)
pd.DataFrame([ctx.grilla.resumen()])

## E00. Entorno
Si alguna entrada trae `legal_basis` o la respuesta, se detiene aquí.

In [ ]:
entorno = pl.e00_entorno(ctx)
pd.Series({k: v for k, v in entorno.items() if k != "grilla"})

## E01. BM25 desde el texto crudo
Unidades por artículo cuando el documento tiene al menos dos; si no, el documento completo. Ventanas de 1500 caracteres con 200 de solapamiento y offsets exactos.

In [ ]:
pd.Series(pl.e01_lexico(ctx) or {})

## E02. Índices densos
Uno por encoder, reanudable. Qwen3-Embedding-4B va de último (8 a 10 h).

In [ ]:
pd.DataFrame(pl.e02_densos(ctx)).T

## E09. Reranker afinado (opcional)
Sintéticas del corpus con el decoder abierto, nunca con la muestra. Registra `ft:<version>` para R13 y R14. Para reentrenar, cambiar `GRILLA['finetune']` y correr de nuevo desde aquí.

In [ ]:
if ENTRENAR_RERANKER:
    display(pd.Series(finetune.e09_reranker(ctx) or {}))
else:
    print("E09 desactivado: R13 y R14 quedan pendientes")

## E03. Búsquedas densas

In [ ]:
pd.DataFrame(pl.e03_busquedas(ctx)).T

## E04. Rankings
Todas las R. Proxy del e06: ¿aparece una norma del fundamento en los 10 primeros chunks?

In [ ]:
tabla_r = pd.DataFrame({k: v for k, v in pl.e04_rankings(ctx).items() if v}).T
tabla_r.sort_values("respaldo_literal_top10", ascending=False) if not tabla_r.empty else tabla_r

## E05. Evidencia
Todas las R x P. El techo de citas no depende del decoder: es lo que sacaríamos citando todas las normas respaldadas.

In [ ]:
tabla_p = pd.DataFrame([{"R": r, "P": p, **(v or {})} for (r, p), v in pl.e05_pasajes(ctx).items()])
tabla_p.pivot(index="R", columns="P", values="techo_puntos_citas").round(2) if not tabla_p.empty else tabla_p

## E06. Generación
Base, un factor a la vez y bloque factorial, fijados antes de mirar resultados. Cada ítem se guarda al terminar.

In [ ]:
print(len(ctx.grilla.corridas_generacion()[:MAX_CORRIDAS_GENERACION]), "corridas")
gen = pl.e06_generacion(ctx, MAX_CORRIDAS_GENERACION)
pd.DataFrame([{"R": r, "P": p, "D": d, **(v or {})} for (r, p, d), v in gen.items()])

## E07. Rúbrica
Todas las políticas sobre cada generación. El puntaje sale de `evaluate.py`; IC95 por bootstrap.

In [ ]:
res = pl.e07_respuestas(ctx)
tabla_c = pd.DataFrame([{"R": r, "P": p, "D": d, "politica": pol, **{k: v for k, v in (x or {}).items() if k != "ic95"}}
                        for (r, p, d, pol), x in res.items()])
tabla_c.sort_values("determinista_50", ascending=False).head(20).round(2) if not tabla_c.empty else tabla_c

## E08. Texto libre
Similitud local para todas; juez RAGAS para la base y las mejores, con caché.

In [ ]:
pl.e08_juez(ctx, ragas=JUEZ_RAGAS, juzgar_todo=JUZGAR_TODO)

## E10. Producto de datos

In [ ]:
carpeta, decision, configuraciones, efectos, area = producto.generar(ctx)
print("Producto en:", carpeta)
print("Recomendada:", (decision.get("recomendada") or {}).get("config"))
for a in decision.get("advertencias", []):
    print("-", a)
display(efectos.round(2) if not efectos.empty else efectos)
HTML((carpeta / "decision.html").read_text(encoding="utf-8"))

## Sábado. Corrida final
Usa la recomendada (o la base si no hay decisión). Se reanuda si se corta. Antes, probar con `limite=20` sobre la muestra y mirar la latencia.

In [ ]:
rec = decision.get("recomendada") or {}
ELECCION = {k: rec.get(k, ctx.grilla.base[k]) for k in ("R", "P", "D", "politica")}
print(ELECCION)
prueba = final.responder_archivo(ctx, ELECCION, ctx.rutas["muestra"], SALIDA / "final_prueba", limite=20)
lat = pd.read_json(prueba, lines=True)["latencia_ms"]
print(f"media {lat.mean() / 1000:.1f} s, p95 {lat.quantile(0.95) / 1000:.1f} s")
if TEST_992:
    print(final.responder_archivo(ctx, ELECCION, TEST_992, SALIDA / "final_test992"))

## Estado
| Situación | Qué hacer |
|---|---|
| Se cortó | Ejecutar todo de nuevo con los mismos parámetros |
| Cambió un prompt o hiperparámetro | Nada: cambia la clave del nodo y solo se recalcula lo afectado |
| Rehacer sin tocar lo anterior | `SALIDA` nueva, la vieja en `REUTILIZAR` |
| Reentrenar | `ENTRENAR_RERANKER = True`, correr E09 y luego E04 a E10 |

In [ ]:
estado = pd.DataFrame(ctx.espacio.estado())
display(estado["estado"].value_counts() if not estado.empty else estado)
estado[estado["estado"].isin(["error", "pendiente"])].reindex(columns=["nodo", "estado", "error", "motivo"]) if not estado.empty else estado